In [ ]:
# ============================================================
# PHASE 1 — CLEAN RUNTIME SETUP
# ============================================================

import os
import gc
import torch

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/maxmin_research"

for folder in ["cache", "frozen", "results"]:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

SEED = 42
SPLIT_SEED = 42

torch.manual_seed(SEED)

print("=" * 60)
print("PHASE 1 — MSMARCO-XI")
print("=" * 60)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    print("GPU memory     :", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

# Check local Colab disk WITHOUT loading the dataset
disk = os.statvfs("/content")
free_gb = disk.f_bavail * disk.f_frsize / 1024**3
total_gb = disk.f_blocks * disk.f_frsize / 1024**3

print("Colab disk     :", round(free_gb, 2), "GB free /",
      round(total_gb, 2), "GB total")
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PHASE 1 — MSMARCO-XI
PyTorch        : 2.11.0+cu130
CUDA available : True
GPU            : Tesla T4
GPU memory     : 14.56 GB
Colab disk     : 68.72 GB free / 112.64 GB total


In [ ]:
!pip -q install -U sentence-transformers datasets pyarrow
!pip -q install "pyarrow<24"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


In [ ]:
import torch
import sentence_transformers
import datasets
import pyarrow

print("PyTorch           :", torch.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("Datasets          :", datasets.__version__)
print("PyArrow           :", pyarrow.__version__)
print("CUDA available    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU               :", torch.cuda.get_device_name(0))

PyTorch           : 2.11.0+cu130
SentenceTransformers: 6.1.0
Datasets          : 5.0.1
PyArrow           : 23.0.1
CUDA available    : True
GPU               : Tesla T4


In [ ]:
from datasets import load_dataset

hi = load_dataset(
    "ai4bharat/MSMARCO-XI",
    data_files="validation/hinval.parquet",
    split="train",
    streaming=True
)

print(hi)

Repo card metadata block was not found. Setting CardData to empty.


IterableDataset({
    features: ['source_lang', 'target_lang', 'meta', 'Answer', 'query_id', 'query_type', 'passages', 'Eng_Query', 'Eng_Answer', 'query'],
    num_shards: 1
})


In [ ]:
first_hi = next(iter(hi))

print("Keys:")
print(first_hi.keys())

print("\nQuery ID:")
print(first_hi["query_id"])

print("\nQuery:")
print(first_hi["query"])

print("\nPassages type:")
print(type(first_hi["passages"]))

print("\nPassages:")
print(first_hi["passages"])

Keys:
dict_keys(['source_lang', 'target_lang', 'meta', 'Answer', 'query_id', 'query_type', 'passages', 'Eng_Query', 'Eng_Answer', 'query'])

Query ID:
1102432

Query:
कॉर्पोरेशन क्या है?

Passages type:
<class 'dict'>

Passages:
{'English_passages': ['A company is incorporated in a specific nation, often within the bounds of a smaller subset of that nation, such as a state or province. The corporation is then governed by the laws of incorporation in that state. A corporation may issue stock, either private or public, or may be classified as a non-stock corporation. If stock is issued, the corporation will usually be governed by its shareholders, either directly or indirectly.', 'Today, there is a growing community of more than 2,100 Certified B Corps from 50 countries and over 130 industries working together toward 1 unifying goal: to redefine success in business. Join the Movement', 'Corporation definition, an association of individuals, created by law or under authority of law, havin

In [ ]:
import json
import os

records_hi = []

for i, row in enumerate(hi):
    records_hi.append({
        "query_id": row["query_id"],
        "query": row["query"],
        "passages": row["passages"]["Translated_passages"],
        "is_selected": row["passages"]["is_selected"],
    })

print("Hindi records collected:", len(records_hi))

valid_hi = [
    r for r in records_hi
    if any(r["is_selected"])
]

print("Valid Hindi queries:", len(valid_hi))

out_path = os.path.join(
    BASE_DIR, "frozen", "msmarco_xi_hindi_raw.json"
)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(records_hi, f, ensure_ascii=False)

print("Saved:", out_path)
print(
    "File size:",
    round(os.path.getsize(out_path) / 1024**2, 2),
    "MB"
)

Hindi records collected: 97941
Valid Hindi queries: 53895
Saved: /content/drive/MyDrive/maxmin_research/frozen/msmarco_xi_hindi_raw.json
File size: 790.03 MB


In [ ]:
import os
import gc

path = "/content/drive/MyDrive/maxmin_research/frozen/msmarco_xi_hindi_raw.json"

if os.path.exists(path):
    os.remove(path)
    print("Deleted oversized Hindi file.")

# Clear the large in-memory objects
del records_hi
del valid_hi
gc.collect()

print("Cleaned up.")

Deleted oversized Hindi file.
Cleaned up.


In [ ]:
from datasets import load_dataset
import json
import os

hi = load_dataset(
    "ai4bharat/MSMARCO-XI",
    data_files="validation/hinval.parquet",
    split="train",
    streaming=True
)

records_hi = []

for i, row in enumerate(hi):
    if i >= 100:
        break

    records_hi.append({
        "query_id": row["query_id"],
        "query": row["query"],
        "passages": row["passages"]["Translated_passages"],
        "is_selected": row["passages"]["is_selected"],
    })

valid_hi = [
    r for r in records_hi
    if any(r["is_selected"])
]

print("Records:", len(records_hi))
print("Valid queries:", len(valid_hi))
print("First query ID:", records_hi[0]["query_id"])
print("Last query ID:", records_hi[-1]["query_id"])

Repo card metadata block was not found. Setting CardData to empty.


Records: 100
Valid queries: 51
First query ID: 1102432
Last query ID: 191749


In [ ]:
import json
import os

out_path = os.path.join(
    BASE_DIR,
    "frozen",
    "msmarco_xi_hindi_100.json"
)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(records_hi, f, ensure_ascii=False, indent=2)

print("Saved:", out_path)
print("Records:", len(records_hi))
print("Valid queries:", len(valid_hi))
print("Size:", round(os.path.getsize(out_path) / 1024**2, 2), "MB")

Saved: /content/drive/MyDrive/maxmin_research/frozen/msmarco_xi_hindi_100.json
Records: 100
Valid queries: 51
Size: 0.8 MB


In [ ]:
import json
import os

LANGUAGE_FILES = {
    "hindi": "msmarco_xi_hindi_100.json",
    "bengali": "msmarco_xi_ben_100.json",
    "tamil": "msmarco_xi_tam_100.json",
    "telugu": "msmarco_xi_tel_100.json",
    "malayalam": "msmarco_xi_mal_100.json",
    "kannada": "msmarco_xi_kan_100.json",
    "marathi": "msmarco_xi_mar_100.json",
    "gujarati": "msmarco_xi_guj_100.json",
    "punjabi": "msmarco_xi_pan_100.json",
}

corpus = []

for language, filename in LANGUAGE_FILES.items():

    path = os.path.join(BASE_DIR, "frozen", filename)

    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for query_idx, record in enumerate(records):

        for passage_idx, (text, label) in enumerate(
            zip(record["passages"], record["is_selected"])
        ):
            corpus.append({
                "parent_id": (
                    f"{language}_"
                    f"{record['query_id']}_"
                    f"p{passage_idx:02d}"
                ),
                "language": language,
                "source_query_id": record["query_id"],
                "query_idx": query_idx,
                "passage_idx": passage_idx,
                "text": text,
                "label": label,
            })

    print(
        f"{language:12s}: "
        f"{len(records)} queries → "
        f"{sum(len(r['passages']) for r in records)} passages"
    )

print("\n" + "=" * 60)
print("FROZEN NATIVE CORPUS")
print("=" * 60)

print("Total passages:", len(corpus))
print("Unique parent IDs:", len(set(x["parent_id"] for x in corpus)))

assert len(corpus) == 8973
assert len(set(x["parent_id"] for x in corpus)) == 8973

out_path = os.path.join(
    BASE_DIR,
    "frozen",
    "msmarco_xi_native_corpus.json"
)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False)

print("\nSaved:", out_path)
print("✓ Exactly 8,973 native passages")
print("✓ Stable parent IDs")

hindi       : 100 queries → 997 passages
bengali     : 100 queries → 997 passages
tamil       : 100 queries → 997 passages
telugu      : 100 queries → 997 passages
malayalam   : 100 queries → 997 passages
kannada     : 100 queries → 997 passages
marathi     : 100 queries → 997 passages
gujarati    : 100 queries → 997 passages
punjabi     : 100 queries → 997 passages

FROZEN NATIVE CORPUS
Total passages: 8973
Unique parent IDs: 8973

Saved: /content/drive/MyDrive/maxmin_research/frozen/msmarco_xi_native_corpus.json
✓ Exactly 8,973 native passages
✓ Stable parent IDs


In [ ]:
import json
import os

# Load frozen benchmark
benchmark_path = os.path.join(
    BASE_DIR, "frozen", "msmarco_xi_frozen_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

# Load native corpus
corpus_path = os.path.join(
    BASE_DIR, "frozen", "msmarco_xi_native_corpus.json"
)

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

# Build lookup: (language, source_query_id) -> parent IDs
corpus_lookup = {}

for p in corpus:
    key = (p["language"], str(p["source_query_id"]))
    corpus_lookup.setdefault(key, []).append(p["parent_id"])

# Verify every benchmark query
missing = []
relevance_counts = []

for q in benchmark:
    key = (q["language"], str(q["query_id"]))

    if key not in corpus_lookup:
        missing.append(key)
        continue

    # Relevant passages in this query's original candidate set
    relevance_counts.append(
        sum(q["is_selected"])
    )

print("=" * 60)
print("FROZEN BENCHMARK ↔ CORPUS CONSISTENCY CHECK")
print("=" * 60)

print("Benchmark queries :", len(benchmark))
print("Corpus passages   :", len(corpus))
print("Missing queries   :", len(missing))

print(
    "DEV queries       :",
    sum(q["split"] == "dev" for q in benchmark)
)
print(
    "TEST queries      :",
    sum(q["split"] == "test" for q in benchmark)
)

print(
    "Queries with ≥1 relevant passage:",
    sum(x > 0 for x in relevance_counts)
)

assert len(corpus) == 8973
assert len(benchmark) == 459
assert not missing
assert all(x > 0 for x in relevance_counts)

print("\n✓ Every frozen query exists in the frozen corpus")
print("✓ 459 valid queries confirmed")
print("✓ DEV/TEST split preserved")
print("✓ Every query has at least one relevant passage")

FROZEN BENCHMARK ↔ CORPUS CONSISTENCY CHECK
Benchmark queries : 459
Corpus passages   : 8973
Missing queries   : 0
DEV queries       : 225
TEST queries      : 234
Queries with ≥1 relevant passage: 459

✓ Every frozen query exists in the frozen corpus
✓ 459 valid queries confirmed
✓ DEV/TEST split preserved
✓ Every query has at least one relevant passage


In [ ]:
import json
import os
from datetime import datetime

manifest = {
    "experiment_id": "MSMARCO_XI_MAXMIN_001",
    "dataset": "AI4Bharat MSMARCO-XI",
    "languages": [
        "hindi", "bengali", "tamil", "telugu",
        "malayalam", "kannada", "marathi",
        "gujarati", "punjabi"
    ],
    "corpus": {
        "total_passages": len(corpus),
        "passages_per_language": 997,
        "source_queries_per_language": 100
    },
    "evaluation_queries": {
        "total_valid": len(benchmark),
        "dev": sum(q["split"] == "dev" for q in benchmark),
        "test": sum(q["split"] == "test" for q in benchmark),
        "split_seed": 42
    },
    "retrieval": {
        "primary_metric": "nDCG@10",
        "secondary_metrics": [
            "MRR@10",
            "Recall@10"
        ],
        "parent_aggregation": "MAX"
    },
    "status": "FROZEN_DATASET",
    "created_at": datetime.now().isoformat()
}

manifest_path = os.path.join(
    BASE_DIR,
    "frozen",
    "experiment_001_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=" * 60)
print("EXPERIMENT 001 — DATASET MANIFEST")
print("=" * 60)

print(json.dumps(manifest, indent=2))
print("\nSaved:", manifest_path)
print("\n✓ Dataset definition frozen")

EXPERIMENT 001 — DATASET MANIFEST
{
  "experiment_id": "MSMARCO_XI_MAXMIN_001",
  "dataset": "AI4Bharat MSMARCO-XI",
  "languages": [
    "hindi",
    "bengali",
    "tamil",
    "telugu",
    "malayalam",
    "kannada",
    "marathi",
    "gujarati",
    "punjabi"
  ],
  "corpus": {
    "total_passages": 8973,
    "passages_per_language": 997,
    "source_queries_per_language": 100
  },
  "evaluation_queries": {
    "total_valid": 459,
    "dev": 225,
    "test": 234,
    "split_seed": 42
  },
  "retrieval": {
    "primary_metric": "nDCG@10",
    "secondary_metrics": [
      "MRR@10",
      "Recall@10"
    ],
    "parent_aggregation": "MAX"
  },
  "status": "FROZEN_DATASET",
  "created_at": "2026-10-02T16:52:36.732872"
}

Saved: /content/drive/MyDrive/maxmin_research/frozen/experiment_001_manifest.json

✓ Dataset definition frozen


In [ ]:
from collections import Counter
import statistics

print("=" * 60)
print("NATIVE CORPUS STRUCTURE SANITY CHECK")
print("=" * 60)

language_counts = Counter(p["language"] for p in corpus)
passage_lengths = [len(p["text"]) for p in corpus]

print("\nPassages by language:")
for lang, count in sorted(language_counts.items()):
    print(f"  {lang:12s}: {count}")

print("\nText length (characters):")
print(f"  Min    : {min(passage_lengths)}")
print(f"  Median : {statistics.median(passage_lengths):.0f}")
print(f"  Mean   : {statistics.mean(passage_lengths):.1f}")
print(f"  Max    : {max(passage_lengths)}")

print("\nSample passages:")
for p in corpus[:3]:
    print(f"\n[{p['language']}] {p['parent_id']}")
    print(p["text"][:500])

print("\n✓ Corpus loaded without modification")

NATIVE CORPUS STRUCTURE SANITY CHECK

Passages by language:
  bengali     : 997
  gujarati    : 997
  hindi       : 997
  kannada     : 997
  malayalam   : 997
  marathi     : 997
  punjabi     : 997
  tamil       : 997
  telugu      : 997

Text length (characters):
  Min    : 29
  Median : 300
  Mean   : 327.4
  Max    : 7515

Sample passages:

[hindi] hindi_1102432_p00
एक कंपनी एक विशिष्ट देश में निगमित होती है, अक्सर उस देश के एक छोटे उपसमूह, जैसे कि एक राज्य या प्रांत, की सीमाओं के भीतर। निगम तब उस राज्य में निगमन के कानूनों द्वारा शासित होता है। एक निगम या तो निजी या सार्वजनिक स्टॉक जारी कर सकता है, या इसे गैर-स्टॉक निगम के रूप में वर्गीकृत किया जा सकता है। यदि स्टॉक जारी किया जाता है, तो निगम आमतौर पर अपने शेयरधारकों द्वारा सीधे या परोक्ष रूप से शासित होगा।

[hindi] hindi_1102432_p01
आज, 50 देशों से 2,100 से अधिक प्रमाणित बी कोर का एक बढ़ता हुआ समुदाय है और 130 से अधिक उद्योग एक एकीकृत लक्ष्य की दिशा में मिलकर काम कर रहे हैं: व्यवसाय में सफलता को पुनः परिभाषित करना। आंदोलन में शा

In [ ]:
import re

# Sentence-ending punctuation used by the MSMARCO-XI passages.
SENTENCE_PATTERN = re.compile(
    r'(?<=[.!?।॥！？])\s+'
)

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()

    if not text:
        return []

    sentences = SENTENCE_PATTERN.split(text)

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]

# Test on the corpus
sentence_counts = []

for p in corpus:
    sentences = split_sentences(p["text"])
    sentence_counts.append(len(sentences))

print("=" * 60)
print("SENTENCE SEGMENTATION SANITY CHECK")
print("=" * 60)

print("Passages:", len(corpus))
print("Total sentences:", sum(sentence_counts))
print("Mean sentences/passage:", sum(sentence_counts) / len(sentence_counts))
print("Min sentences/passage:", min(sentence_counts))
print("Max sentences/passage:", max(sentence_counts))

print("\nExamples:")

for p in corpus[:3]:
    sentences = split_sentences(p["text"])

    print(f"\n[{p['language']}] {p['parent_id']}")
    for i, s in enumerate(sentences):
        print(f"  S{i}: {s}")

print("\n✓ Deterministic splitter defined")

SENTENCE SEGMENTATION SANITY CHECK
Passages: 8973
Total sentences: 31792
Mean sentences/passage: 3.5430736654407666
Min sentences/passage: 1
Max sentences/passage: 135

Examples:

[hindi] hindi_1102432_p00
  S0: एक कंपनी एक विशिष्ट देश में निगमित होती है, अक्सर उस देश के एक छोटे उपसमूह, जैसे कि एक राज्य या प्रांत, की सीमाओं के भीतर।
  S1: निगम तब उस राज्य में निगमन के कानूनों द्वारा शासित होता है।
  S2: एक निगम या तो निजी या सार्वजनिक स्टॉक जारी कर सकता है, या इसे गैर-स्टॉक निगम के रूप में वर्गीकृत किया जा सकता है।
  S3: यदि स्टॉक जारी किया जाता है, तो निगम आमतौर पर अपने शेयरधारकों द्वारा सीधे या परोक्ष रूप से शासित होगा।

[hindi] hindi_1102432_p01
  S0: आज, 50 देशों से 2,100 से अधिक प्रमाणित बी कोर का एक बढ़ता हुआ समुदाय है और 130 से अधिक उद्योग एक एकीकृत लक्ष्य की दिशा में मिलकर काम कर रहे हैं: व्यवसाय में सफलता को पुनः परिभाषित करना।
  S1: आंदोलन में शामिल हों

[hindi] hindi_1102432_p02
  S0: निगम की परिभाषा, व्यक्तियों का एक समूह, जो कानून द्वारा या कानून के अधिकार के तहत बनाया गया

In [ ]:
from collections import defaultdict
import statistics

stats = defaultdict(list)

for p in corpus:
    n_sent = len(split_sentences(p["text"]))
    stats[p["language"]].append(n_sent)

print("=" * 70)
print("SENTENCE-COUNT DISTRIBUTION BY LANGUAGE")
print("=" * 70)

for lang in sorted(stats):
    values = stats[lang]

    print(
        f"{lang:12s} | "
        f"mean={statistics.mean(values):5.2f} | "
        f"median={statistics.median(values):4.1f} | "
        f"min={min(values):2d} | "
        f"max={max(values):3d}"
    )

# Identify unusually long passages
long_passages = []

for p in corpus:
    sentences = split_sentences(p["text"])

    if len(sentences) >= 20:
        long_passages.append({
            "parent_id": p["parent_id"],
            "language": p["language"],
            "sentences": len(sentences),
            "characters": len(p["text"])
        })

print("\nPassages with >=20 sentences:", len(long_passages))

print("\nLongest passages:")
for x in sorted(long_passages, key=lambda z: z["sentences"], reverse=True)[:10]:
    print(
        f"  {x['language']:12s} "
        f"{x['parent_id']} | "
        f"{x['sentences']} sentences | "
        f"{x['characters']} chars"
    )

print("\n✓ Sentence count remains:", sum(
    len(split_sentences(p["text"])) for p in corpus
))

SENTENCE-COUNT DISTRIBUTION BY LANGUAGE
bengali      | mean= 3.25 | median= 3.0 | min= 1 | max= 17
gujarati     | mean= 3.54 | median= 3.0 | min= 1 | max= 17
hindi        | mean= 3.58 | median= 3.0 | min= 1 | max= 18
kannada      | mean= 3.69 | median= 3.0 | min= 1 | max= 46
malayalam    | mean= 3.55 | median= 3.0 | min= 1 | max= 28
marathi      | mean= 3.75 | median= 3.0 | min= 1 | max=135
punjabi      | mean= 3.55 | median= 3.0 | min= 1 | max= 25
tamil        | mean= 3.50 | median= 3.0 | min= 1 | max= 68
telugu       | mean= 3.48 | median= 3.0 | min= 1 | max= 17

Passages with >=20 sentences: 7

Longest passages:
  marathi      marathi_180855_p03 | 135 sentences | 7385 chars
  tamil        tamil_165349_p00 | 68 sentences | 3016 chars
  kannada      kannada_180855_p03 | 46 sentences | 2421 chars
  tamil        tamil_180855_p03 | 45 sentences | 2994 chars
  malayalam    malayalam_110842_p04 | 28 sentences | 2606 chars
  kannada      kannada_180855_p06 | 28 sentences | 2322 chars
  punj

In [ ]:
sentence_corpus = []

for p in corpus:
    sentences = split_sentences(p["text"])

    for sentence_idx, sentence in enumerate(sentences):
        sentence_corpus.append({
            "sentence_id": (
                f"{p['parent_id']}_s{sentence_idx:03d}"
            ),
            "parent_id": p["parent_id"],
            "language": p["language"],
            "sentence_idx": sentence_idx,
            "text": sentence,
        })

# Save frozen sentence representation
sentence_corpus_path = os.path.join(
    BASE_DIR,
    "frozen",
    "msmarco_xi_sentence_corpus.json"
)

with open(sentence_corpus_path, "w", encoding="utf-8") as f:
    json.dump(
        sentence_corpus,
        f,
        ensure_ascii=False
    )

print("=" * 60)
print("FROZEN SENTENCE CORPUS")
print("=" * 60)

print("Native passages :", len(corpus))
print("Sentences       :", len(sentence_corpus))
print("Expected        :", 31792)

assert len(sentence_corpus) == 31792
assert len({
    s["sentence_id"] for s in sentence_corpus
}) == 31792

print("\n✓ 31,792 deterministic sentence records")
print("✓ Unique sentence IDs")
print("✓ Saved:", sentence_corpus_path)

FROZEN SENTENCE CORPUS
Native passages : 8973
Sentences       : 31792
Expected        : 31792

✓ 31,792 deterministic sentence records
✓ Unique sentence IDs
✓ Saved: /content/drive/MyDrive/maxmin_research/frozen/msmarco_xi_sentence_corpus.json


In [ ]:
import torch

print("=" * 60)
print("GPU READINESS CHECK")
print("=" * 60)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = (
        torch.cuda.mem_get_info()[0] / 1024**3
    )

    print("GPU     :", gpu)
    print(f"VRAM    : {total:.2f} GB")
    print(f"Free    : {free:.2f} GB")
else:
    raise RuntimeError("CUDA GPU is not available.")

print("\n✓ GPU ready")

GPU READINESS CHECK
PyTorch : 2.11.0+cu130
CUDA    : True
GPU     : Tesla T4
VRAM    : 14.56 GB
Free    : 14.46 GB

✓ GPU ready


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda"
)

model.max_seq_length = 512

print("=" * 60)
print("QWEN EMBEDDING MODEL")
print("=" * 60)

print("Model :", MODEL_NAME)
print("Device:", model.device)
print("Dimension:", model.get_sentence_embedding_dimension())
print("Max sequence length:", model.max_seq_length)

assert str(model.device).startswith("cuda")
assert model.get_sentence_embedding_dimension() == 1024

print("\n✓ Qwen3-Embedding-0.6B loaded on GPU")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

QWEN EMBEDDING MODEL
Model : Qwen/Qwen3-Embedding-0.6B
Device: cuda:0
Dimension: 1024
Max sequence length: 512

✓ Qwen3-Embedding-0.6B loaded on GPU


/tmp/ipykernel_19846/4199869150.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimension:", model.get_sentence_embedding_dimension())
/tmp/ipykernel_19846/4199869150.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == 1024


In [ ]:
import os
import json
import numpy as np
import torch
from tqdm.auto import tqdm

CACHE_DIR = os.path.join(BASE_DIR, "cache", "qwen_sentence_embeddings")
os.makedirs(CACHE_DIR, exist_ok=True)

EMBED_DIM = 1024
BATCH_SIZE = 64
CHECKPOINT_SIZE = 1000

texts = [s["text"] for s in sentence_corpus]
ids = [s["sentence_id"] for s in sentence_corpus]

final_path = os.path.join(
    CACHE_DIR,
    "qwen_sentence_embeddings.npy"
)

ids_path = os.path.join(
    CACHE_DIR,
    "sentence_ids.json"
)

# Resume from completed checkpoints
checkpoint_files = sorted(
    f for f in os.listdir(CACHE_DIR)
    if f.startswith("checkpoint_") and f.endswith(".npy")
)

completed = 0
checkpoint_embeddings = []

if checkpoint_files:
    for filename in checkpoint_files:
        arr = np.load(os.path.join(CACHE_DIR, filename))
        checkpoint_embeddings.append(arr)
        completed += len(arr)

    print(f"Resuming from {completed:,} embeddings")

else:
    print("Starting fresh embedding run")

remaining_texts = texts[completed:]

if remaining_texts:
    new_embeddings = []

    for start in tqdm(
        range(0, len(remaining_texts), CHECKPOINT_SIZE),
        desc="Embedding checkpoints"
    ):
        block = remaining_texts[start:start + CHECKPOINT_SIZE]

        emb = model.encode(
            block,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        new_embeddings.append(emb)

        checkpoint_index = completed + start + len(emb)

        np.save(
            os.path.join(
                CACHE_DIR,
                f"checkpoint_{checkpoint_index:06d}.npy"
            ),
            emb
        )

        print(f"Saved checkpoint: {checkpoint_index:,}/{len(texts):,}")

    checkpoint_embeddings.extend(new_embeddings)

embeddings = np.vstack(checkpoint_embeddings)

assert embeddings.shape == (len(sentence_corpus), EMBED_DIM)

# Final consolidated embedding matrix
np.save(final_path, embeddings)

with open(ids_path, "w", encoding="utf-8") as f:
    json.dump(ids, f, ensure_ascii=False)

print("\n" + "=" * 60)
print("QWEN SENTENCE EMBEDDINGS COMPLETE")
print("=" * 60)

print("Embeddings :", embeddings.shape)
print("Dtype      :", embeddings.dtype)
print("Normalized :", np.allclose(
    np.linalg.norm(embeddings[:100], axis=1),
    1.0,
    atol=1e-4
))

print("Saved      :", final_path)
print("IDs saved  :", ids_path)

print("\n✓ 31,792 sentence embeddings cached")

Starting fresh embedding run


Embedding checkpoints:   0%|          | 0/32 [00:00<?, ?it/s]

Saved checkpoint: 1,000/31,792
Saved checkpoint: 2,000/31,792
Saved checkpoint: 3,000/31,792
Saved checkpoint: 4,000/31,792
Saved checkpoint: 5,000/31,792
Saved checkpoint: 6,000/31,792
Saved checkpoint: 7,000/31,792
Saved checkpoint: 8,000/31,792
Saved checkpoint: 9,000/31,792
Saved checkpoint: 10,000/31,792
Saved checkpoint: 11,000/31,792
Saved checkpoint: 12,000/31,792
Saved checkpoint: 13,000/31,792
Saved checkpoint: 14,000/31,792
Saved checkpoint: 15,000/31,792
Saved checkpoint: 16,000/31,792
Saved checkpoint: 17,000/31,792
Saved checkpoint: 18,000/31,792
Saved checkpoint: 19,000/31,792
Saved checkpoint: 20,000/31,792
Saved checkpoint: 21,000/31,792
Saved checkpoint: 22,000/31,792
Saved checkpoint: 23,000/31,792
Saved checkpoint: 24,000/31,792
Saved checkpoint: 25,000/31,792
Saved checkpoint: 26,000/31,792
Saved checkpoint: 27,000/31,792
Saved checkpoint: 28,000/31,792
Saved checkpoint: 29,000/31,792
Saved checkpoint: 30,000/31,792
Saved checkpoint: 31,000/31,792
Saved checkpoint:

In [ ]:
import numpy as np
import os

EMBED_PATH = os.path.join(
    BASE_DIR,
    "cache",
    "qwen_sentence_embeddings",
    "qwen_sentence_embeddings.npy"
)

emb = np.load(EMBED_PATH, mmap_mode="r")

norms = np.linalg.norm(emb[:1000], axis=1)

print("Shape:", emb.shape)
print("Min norm :", norms.min())
print("Mean norm:", norms.mean())
print("Max norm :", norms.max())

print("\nNormalized:", np.allclose(norms, 1.0, atol=1e-4))

Shape: (31792, 1024)
Min norm : 0.99804956
Mean norm: 1.0004425
Max norm : 1.0038674

Normalized: False


In [ ]:
import numpy as np
import os

EMBED_PATH = os.path.join(
    BASE_DIR,
    "cache",
    "qwen_sentence_embeddings",
    "qwen_sentence_embeddings.npy"
)

emb = np.load(EMBED_PATH)

# Explicit L2 normalization
norms = np.linalg.norm(emb, axis=1, keepdims=True)
emb = emb / norms

# Save normalized matrix
np.save(EMBED_PATH, emb.astype(np.float32))

# Verify
check = np.linalg.norm(emb[:1000], axis=1)

print("Shape:", emb.shape)
print("Dtype:", emb.dtype)
print("Min norm :", check.min())
print("Mean norm:", check.mean())
print("Max norm :", check.max())
print("Normalized:", np.allclose(check, 1.0, atol=1e-5))

del emb, norms, check

Shape: (31792, 1024)
Dtype: float32
Min norm : 0.9999999
Mean norm: 1.0
Max norm : 1.0000001
Normalized: True


In [ ]:
import json
import os
import numpy as np

CACHE_DIR = os.path.join(
    BASE_DIR,
    "cache",
    "qwen_sentence_embeddings"
)

with open(
    os.path.join(CACHE_DIR, "sentence_ids.json"),
    "r",
    encoding="utf-8"
) as f:
    embedding_ids = json.load(f)

corpus_ids = [s["sentence_id"] for s in sentence_corpus]

emb = np.load(
    os.path.join(CACHE_DIR, "qwen_sentence_embeddings.npy"),
    mmap_mode="r"
)

print("Embedding rows :", len(emb))
print("Embedding IDs  :", len(embedding_ids))
print("Corpus IDs     :", len(corpus_ids))

assert len(emb) == len(embedding_ids) == len(corpus_ids)
assert embedding_ids == corpus_ids

print("\n✓ Embedding count matches sentence corpus")
print("✓ Sentence ID order matches exactly")
print("✓ Embedding matrix is aligned with frozen sentence corpus")

Embedding rows : 31792
Embedding IDs  : 31792
Corpus IDs     : 31792

✓ Embedding count matches sentence corpus
✓ Sentence ID order matches exactly
✓ Embedding matrix is aligned with frozen sentence corpus


In [ ]:
import json
import os

NATIVE_PATH = os.path.join(
    BASE_DIR,
    "frozen",
    "msmarco_xi_native_corpus.json"
)

with open(NATIVE_PATH, "r", encoding="utf-8") as f:
    native_corpus = json.load(f)

print("Native passages:", len(native_corpus))

# Inspect structure
print("\nFirst record:")
print(native_corpus[0])

# Stable ordered IDs and texts
native_ids = [p["parent_id"] for p in native_corpus]
native_texts = [p["text"] for p in native_corpus]

assert len(native_ids) == 8973
assert len(native_texts) == 8973
assert len(set(native_ids)) == 8973
assert all(native_texts)

print("\n✓ 8,973 native passages loaded")
print("✓ Parent IDs are unique")
print("✓ Every passage has text")
print("✓ Native corpus order is frozen")

Native passages: 8973

First record:
{'parent_id': 'hindi_1102432_p00', 'language': 'hindi', 'source_query_id': 1102432, 'query_idx': 0, 'passage_idx': 0, 'text': 'एक कंपनी एक विशिष्ट देश में निगमित होती है, अक्सर उस देश के एक छोटे उपसमूह, जैसे कि एक राज्य या प्रांत, की सीमाओं के भीतर। निगम तब उस राज्य में निगमन के कानूनों द्वारा शासित होता है। एक निगम या तो निजी या सार्वजनिक स्टॉक जारी कर सकता है, या इसे गैर-स्टॉक निगम के रूप में वर्गीकृत किया जा सकता है। यदि स्टॉक जारी किया जाता है, तो निगम आमतौर पर अपने शेयरधारकों द्वारा सीधे या परोक्ष रूप से शासित होगा।', 'label': 0}

✓ 8,973 native passages loaded
✓ Parent IDs are unique
✓ Every passage has text
✓ Native corpus order is frozen


In [ ]:
NATIVE_CACHE = os.path.join(
    BASE_DIR,
    "cache",
    "qwen_native_embeddings"
)
os.makedirs(NATIVE_CACHE, exist_ok=True)

NATIVE_EMBED_PATH = os.path.join(
    NATIVE_CACHE,
    "qwen_native_embeddings.npy"
)

NATIVE_IDS_PATH = os.path.join(
    NATIVE_CACHE,
    "parent_ids.json"
)

BATCH_SIZE = 64
CHECKPOINT_SIZE = 1000
EMBED_DIM = 1024

# Already completed?
if os.path.exists(NATIVE_EMBED_PATH):
    native_embeddings = np.load(NATIVE_EMBED_PATH, mmap_mode="r")

    print("Existing native embeddings found")
    print("Shape:", native_embeddings.shape)

else:
    checkpoint_files = sorted(
        f for f in os.listdir(NATIVE_CACHE)
        if f.startswith("checkpoint_") and f.endswith(".npy")
    )

    completed = 0
    checkpoint_embeddings = []

    for filename in checkpoint_files:
        arr = np.load(os.path.join(NATIVE_CACHE, filename))
        checkpoint_embeddings.append(arr)
        completed += len(arr)

    if completed:
        print(f"Resuming from {completed:,} passages")
    else:
        print("Starting fresh native embedding run")

    remaining_texts = native_texts[completed:]

    for start in tqdm(
        range(0, len(remaining_texts), CHECKPOINT_SIZE),
        desc="Native embedding checkpoints"
    ):
        block = remaining_texts[start:start + CHECKPOINT_SIZE]

        emb = model.encode(
            block,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        checkpoint_embeddings.append(emb)

        checkpoint_index = completed + start + len(emb)

        np.save(
            os.path.join(
                NATIVE_CACHE,
                f"checkpoint_{checkpoint_index:06d}.npy"
            ),
            emb
        )

        print(
            f"Saved checkpoint: "
            f"{checkpoint_index:,}/{len(native_texts):,}"
        )

    native_embeddings = np.vstack(checkpoint_embeddings)

    # Explicit normalization, same policy as sentence embeddings
    native_embeddings /= np.linalg.norm(
        native_embeddings,
        axis=1,
        keepdims=True
    )

    np.save(NATIVE_EMBED_PATH, native_embeddings)

    with open(NATIVE_IDS_PATH, "w", encoding="utf-8") as f:
        json.dump(native_ids, f, ensure_ascii=False)

    print("\n" + "=" * 60)
    print("NATIVE EMBEDDINGS COMPLETE")
    print("=" * 60)

    print("Shape:", native_embeddings.shape)
    print("Dtype:", native_embeddings.dtype)

Starting fresh native embedding run


Native embedding checkpoints:   0%|          | 0/9 [00:00<?, ?it/s]

Saved checkpoint: 1,000/8,973
Saved checkpoint: 2,000/8,973
Saved checkpoint: 3,000/8,973
Saved checkpoint: 4,000/8,973
Saved checkpoint: 5,000/8,973
Saved checkpoint: 6,000/8,973
Saved checkpoint: 7,000/8,973
Saved checkpoint: 8,000/8,973
Saved checkpoint: 8,973/8,973

NATIVE EMBEDDINGS COMPLETE
Shape: (8973, 1024)
Dtype: float32


In [ ]:
import json
import os
import numpy as np

NATIVE_CACHE = os.path.join(
    BASE_DIR,
    "cache",
    "qwen_native_embeddings"
)

emb = np.load(
    os.path.join(NATIVE_CACHE, "qwen_native_embeddings.npy"),
    mmap_mode="r"
)

with open(
    os.path.join(NATIVE_CACHE, "parent_ids.json"),
    "r",
    encoding="utf-8"
) as f:
    embedding_ids = json.load(f)

norms = np.linalg.norm(emb[:1000], axis=1)

print("Embedding shape :", emb.shape)
print("Embedding IDs   :", len(embedding_ids))
print("Corpus IDs      :", len(native_ids))

print("\nNorm min :", norms.min())
print("Norm mean:", norms.mean())
print("Norm max :", norms.max())

assert emb.shape == (8973, 1024)
assert embedding_ids == native_ids
assert np.allclose(norms, 1.0, atol=1e-5)

print("\n✓ Native embeddings are normalized")
print("✓ Native embedding IDs match corpus exactly")
print("✓ Native baseline representation is ready")

Embedding shape : (8973, 1024)
Embedding IDs   : 8973
Corpus IDs      : 8973

Norm min : 0.9999999
Norm mean: 1.0
Norm max : 1.0000001

✓ Native embeddings are normalized
✓ Native embedding IDs match corpus exactly
✓ Native baseline representation is ready


In [ ]:
import json
import os
import numpy as np

FROZEN_BENCHMARK_PATH = os.path.join(
    BASE_DIR,
    "frozen",
    "msmarco_xi_frozen_benchmark.json"
)

with open(FROZEN_BENCHMARK_PATH, "r", encoding="utf-8") as f:
    frozen_benchmark = json.load(f)

print("Frozen queries:", len(frozen_benchmark))

assert len(frozen_benchmark) == 459

QUERY_CACHE = os.path.join(
    BASE_DIR,
    "cache",
    "qwen_query_embeddings"
)
os.makedirs(QUERY_CACHE, exist_ok=True)

QUERY_EMBED_PATH = os.path.join(
    QUERY_CACHE,
    "qwen_query_embeddings.npy"
)

query_ids = [q["query_id"] for q in frozen_benchmark]
query_texts = [q["query"] for q in frozen_benchmark]

QUERY_PREFIX = (
    "Instruct: Given a web search query, retrieve relevant passages "
    "that answer the query\n"
    "Query:"
)

query_inputs = [
    f"{QUERY_PREFIX}{q}"
    for q in query_texts
]

if os.path.exists(QUERY_EMBED_PATH):
    query_embeddings = np.load(QUERY_EMBED_PATH)

    print("Existing query embeddings found")
    print("Shape:", query_embeddings.shape)

else:
    query_embeddings = model.encode(
        query_inputs,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype(np.float32)

    query_embeddings /= np.linalg.norm(
        query_embeddings,
        axis=1,
        keepdims=True
    )

    np.save(QUERY_EMBED_PATH, query_embeddings)

    print("Query embeddings created")
    print("Shape:", query_embeddings.shape)

assert query_embeddings.shape == (459, 1024)

print("\n✓ 459 frozen queries ready")
print("✓ Query instruction applied")
print("✓ Query embeddings normalized")
print("✓ Saved to Drive")

Frozen queries: 459


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Query embeddings created
Shape: (459, 1024)

✓ 459 frozen queries ready
✓ Query instruction applied
✓ Query embeddings normalized
✓ Saved to Drive


In [ ]:
q = frozen_benchmark[0]

print("Number of passages:", len(q["passages"]))
print("\nFirst passage object:")
print(q["passages"][0])
print("\nType:", type(q["passages"][0]))

Number of passages: 10

First passage object:
सीमावर्ती विकार के लिए नौ डीएसएम-IV मानदंडों में से तीन इस समूह में आते हैं: 1. प्रभावी (भावनात्मक) अस्थिरता, जिसमें तीव्र, प्रकरणिक भावनात्मक पीड़ा, चिड़चिड़ापन और चिंता/घबराहट के दौरे शामिल हैं। 2. अनुचित, तीव्र और नियंत्रित करने में कठिन क्रोध। 3. खालीपन की पुरानी भावनाएं।

Type: <class 'str'>


In [ ]:
q = frozen_benchmark[0]

print(q.keys())

print("\nType of is_selected:")
print(type(q["is_selected"]))

print("\nis_selected:")
print(q["is_selected"])

print("\nNumber of labels:", len(q["is_selected"]))

dict_keys(['language', 'query_id', 'query', 'passages', 'is_selected', 'split'])

Type of is_selected:
<class 'list'>

is_selected:
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]

Number of labels: 10


In [ ]:
# Verify benchmark passage index → native parent alignment

checked = 0
mismatches = []

for q in frozen_benchmark:
    qid = q["query_id"]

    for i, (text, label) in enumerate(
        zip(q["passages"], q["is_selected"])
    ):
        matches = [
            p for p in native_corpus
            if p["source_query_id"] == qid
            and p["passage_idx"] == i
        ]

        if len(matches) != 1:
            mismatches.append((qid, i, len(matches)))
            continue

        if matches[0]["text"] != text:
            mismatches.append((qid, i, "TEXT_MISMATCH"))
            continue

        checked += 1

print("Aligned passages checked:", checked)
print("Mismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatches:")
    print(mismatches[:10])
else:
    print("\n✓ Every benchmark passage maps uniquely to its frozen parent_id")

Aligned passages checked: 0
Mismatches: 4563

First mismatches:
[(1090350, 0, 9), (1090350, 1, 9), (1090350, 2, 9), (1090350, 3, 9), (1090350, 4, 9), (1090350, 5, 9), (1090350, 6, 9), (1090350, 7, 9), (1090350, 8, 9), (1090350, 9, 9)]


In [ ]:
qid = frozen_benchmark[0]["query_id"]
lang = frozen_benchmark[0]["language"]

matches = [
    p for p in native_corpus
    if p["source_query_id"] == qid
]

print("Benchmark language :", lang)
print("Benchmark query ID :", qid)
print("Native matches for query ID:", len(matches))

print("\nLanguages represented:")
print(sorted(set(p["language"] for p in matches)))

print("\nMatches in benchmark language:")
lang_matches = [
    p for p in matches
    if p["language"] == lang
]

print("Count:", len(lang_matches))

for p in lang_matches[:3]:
    print(
        p["language"],
        p["source_query_id"],
        p["passage_idx"],
        p["parent_id"]
    )

Benchmark language : hindi
Benchmark query ID : 1090350
Native matches for query ID: 90

Languages represented:
['bengali', 'gujarati', 'hindi', 'kannada', 'malayalam', 'marathi', 'punjabi', 'tamil', 'telugu']

Matches in benchmark language:
Count: 10
hindi 1090350 0 hindi_1090350_p00
hindi 1090350 1 hindi_1090350_p01
hindi 1090350 2 hindi_1090350_p02


In [ ]:
native_lookup = {
    (
        p["language"],
        p["source_query_id"],
        p["passage_idx"]
    ): p["parent_id"]
    for p in native_corpus
}

checked = 0
mismatches = []

for q in frozen_benchmark:
    lang = q["language"]
    qid = q["query_id"]

    for i, text in enumerate(q["passages"]):

        key = (lang, qid, i)

        if key not in native_lookup:
            mismatches.append((lang, qid, i, "MISSING"))
            continue

        parent_id = native_lookup[key]

        # Verify the actual text too
        native_record = next(
            p for p in native_corpus
            if p["parent_id"] == parent_id
        )

        if native_record["text"] != text:
            mismatches.append((lang, qid, i, "TEXT_MISMATCH"))
            continue

        checked += 1

print("Aligned passages checked:", checked)
print("Mismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatches:")
    print(mismatches[:10])
else:
    print("\n✓ All benchmark passages map exactly")
    print("✓ Language + query_id + passage_idx uniquely identifies parent")
    print("✓ Frozen benchmark ↔ native corpus alignment verified")

Aligned passages checked: 4563
Mismatches: 0

✓ All benchmark passages map exactly
✓ Language + query_id + passage_idx uniquely identifies parent
✓ Frozen benchmark ↔ native corpus alignment verified


In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

# ---------------------------------------------------------
# 1. Native embedding matrix
# ---------------------------------------------------------
native_emb = np.load(
    os.path.join(
        BASE_DIR,
        "cache",
        "qwen_native_embeddings",
        "qwen_native_embeddings.npy"
    ),
    mmap_mode="r"
)

# ---------------------------------------------------------
# 2. Frozen passage → parent lookup
# ---------------------------------------------------------
native_lookup = {
    (
        p["language"],
        p["source_query_id"],
        p["passage_idx"]
    ): p["parent_id"]
    for p in native_corpus
}

# ---------------------------------------------------------
# 3. Query embedding lookup
# ---------------------------------------------------------
query_index = {
    qid: i
    for i, qid in enumerate(query_ids)
}

# ---------------------------------------------------------
# 4. Evaluation
# ---------------------------------------------------------
def evaluate_native(query_records, top_k=10):

    ndcg_scores = []
    mrr_scores = []
    recall_scores = []

    for q in query_records:

        q_emb = query_embeddings[
            query_index[q["query_id"]]
        ]

        # Normalized vectors → dot product = cosine similarity
        scores = native_emb @ q_emb

        ranking = np.argsort(-scores)[:top_k]

        # Convert the query's relevant candidate passages
        # into their frozen native parent IDs.
        relevant = set()

        for i, label in enumerate(q["is_selected"]):

            if label == 1:

                key = (
                    q["language"],
                    q["query_id"],
                    i
                )

                relevant.add(native_lookup[key])

        # Relevance of retrieved passages
        retrieved_relevance = np.array(
            [
                1.0 if native_ids[idx] in relevant else 0.0
                for idx in ranking
            ],
            dtype=np.float32
        )

        # -----------------------------
        # nDCG@10
        # -----------------------------
        ideal = np.sort(
            retrieved_relevance
        )[::-1]

        dcg = np.sum(
            retrieved_relevance /
            np.log2(np.arange(2, top_k + 2))
        )

        idcg = np.sum(
            ideal /
            np.log2(np.arange(2, top_k + 2))
        )

        ndcg = dcg / idcg if idcg > 0 else 0.0

        ndcg_scores.append(ndcg)

        # -----------------------------
        # MRR@10
        # -----------------------------
        rr = 0.0

        for rank, rel in enumerate(
            retrieved_relevance,
            start=1
        ):
            if rel > 0:
                rr = 1.0 / rank
                break

        mrr_scores.append(rr)

        # -----------------------------
        # Recall@10
        # -----------------------------
        recall_scores.append(
            retrieved_relevance.sum() /
            len(relevant)
        )

    return {
        "nDCG@10": float(np.mean(ndcg_scores)),
        "MRR@10": float(np.mean(mrr_scores)),
        "Recall@10": float(np.mean(recall_scores)),
        "queries": len(query_records)
    }


# ---------------------------------------------------------
# 5. Frozen DEV / TEST split
# ---------------------------------------------------------
dev_queries = [
    q for q in frozen_benchmark
    if q["split"] == "dev"
]

test_queries = [
    q for q in frozen_benchmark
    if q["split"] == "test"
]

# ---------------------------------------------------------
# 6. Run
# ---------------------------------------------------------
dev_result = evaluate_native(dev_queries)
test_result = evaluate_native(test_queries)

print("=" * 60)
print("NATIVE BASELINE — QWEN")
print("=" * 60)

print("\nDEV")
for k, v in dev_result.items():
    if isinstance(v, float):
        print(f"{k:12}: {v:.6f}")
    else:
        print(f"{k:12}: {v}")

print("\nTEST")
for k, v in test_result.items():
    if isinstance(v, float):
        print(f"{k:12}: {v:.6f}")
    else:
        print(f"{k:12}: {v}")

NATIVE BASELINE — QWEN

DEV
nDCG@10     : 0.119839
MRR@10      : 0.082829
Recall@10   : 0.237778
queries     : 225

TEST
nDCG@10     : 0.106391
MRR@10      : 0.072885
Recall@10   : 0.211538
queries     : 234


In [ ]:
import numpy as np

# ---------------------------------------------------------
# Native corpus metadata lookup by embedding row
# ---------------------------------------------------------
native_langs = np.array([
    p["language"]
    for p in native_corpus
])

# Sanity alignment
assert len(native_langs) == native_emb.shape[0] == len(native_ids)

# ---------------------------------------------------------
# Fixed random sample: 20 queries
# ---------------------------------------------------------
rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(frozen_benchmark),
    size=20,
    replace=False
)

cross_language_count = 0
total_retrieved = 0

print("=" * 80)
print("DIAGNOSTIC 1A — CROSS-LANGUAGE RETRIEVAL")
print("=" * 80)

for sample_no, qi in enumerate(sample_indices, start=1):

    q = frozen_benchmark[qi]
    q_emb = query_embeddings[query_index[q["query_id"]]]

    # Current all-language retrieval — unchanged
    scores = native_emb @ q_emb
    ranking = np.argsort(-scores)[:10]

    # Relevant parent IDs for this query
    relevant = {
        native_lookup[
            (q["language"], q["query_id"], i)
        ]
        for i, label in enumerate(q["is_selected"])
        if label == 1
    }

    print(
        f"\nQuery {sample_no}/20 | "
        f"Language: {q['language']} | "
        f"ID: {q['query_id']}"
    )

    print(
        f"{'Rank':<6}"
        f"{'Language':<12}"
        f"{'Similarity':<13}"
        f"{'Relevant':<11}"
        f"{'Parent ID'}"
    )

    print("-" * 65)

    for rank, idx in enumerate(ranking, start=1):

        retrieved_lang = native_langs[idx]
        parent_id = native_ids[idx]

        is_relevant = parent_id in relevant

        if retrieved_lang != q["language"]:
            cross_language_count += 1

        total_retrieved += 1

        print(
            f"{rank:<6}"
            f"{retrieved_lang:<12}"
            f"{scores[idx]:<13.6f}"
            f"{str(is_relevant):<11}"
            f"{parent_id}"
        )

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    f"Different-language top-10 results: "
    f"{cross_language_count}/{total_retrieved}"
)

print(
    f"Cross-language share: "
    f"{cross_language_count / total_retrieved:.4%}"
)

DIAGNOSTIC 1A — CROSS-LANGUAGE RETRIEVAL

Query 1/20 | Language: marathi | ID: 267380
Rank  Language    Similarity   Relevant   Parent ID
-----------------------------------------------------------------
1     bengali     0.661260     False      bengali_267380_p02
2     punjabi     0.642945     False      punjabi_267380_p02
3     malayalam   0.641301     False      malayalam_267380_p08
4     punjabi     0.633121     False      punjabi_267380_p04
5     tamil       0.632449     False      tamil_267380_p02
6     gujarati    0.623866     False      gujarati_267380_p01
7     malayalam   0.618080     False      malayalam_267380_p01
8     gujarati    0.617212     False      gujarati_267380_p02
9     marathi     0.614276     True       marathi_267380_p02
10    hindi       0.611921     False      hindi_267380_p02

Query 2/20 | Language: marathi | ID: 229215
Rank  Language    Similarity   Relevant   Parent ID
-----------------------------------------------------------------
1     tamil       0.5

In [ ]:
# ============================================================
# DIAGNOSTIC 1b — Native retrieval restricted to query language
# ============================================================

import numpy as np
from collections import defaultdict

TOP_K = 10

# ------------------------------------------------------------
# 1. Build native-corpus indices by language
# ------------------------------------------------------------
lang_to_indices = defaultdict(list)

for i, p in enumerate(native_corpus):
    lang_to_indices[p["language"]].append(i)

print("Native passages by language:")
for lang in sorted(lang_to_indices):
    print(f"  {lang}: {len(lang_to_indices[lang])}")

# Frozen corpus should contain exactly 997 passages per language
assert all(len(v) == 997 for v in lang_to_indices.values()), \
    "Unexpected native corpus size for at least one language."

# ------------------------------------------------------------
# 2. Metric helpers
# ------------------------------------------------------------
def dcg_at_k(rel):
    return sum(
        r / np.log2(i + 2)
        for i, r in enumerate(rel[:TOP_K])
    )

def ndcg_at_k(rel):
    actual = dcg_at_k(rel)

    ideal_rel = sorted(rel, reverse=True)
    ideal = dcg_at_k(ideal_rel)

    return actual / ideal if ideal > 0 else 0.0

def mrr_at_k(rel):
    for i, r in enumerate(rel[:TOP_K]):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.0

def recall_at_k(rel, num_relevant):
    return sum(rel[:TOP_K]) / num_relevant if num_relevant > 0 else 0.0


# ------------------------------------------------------------
# 3. Evaluate Native retrieval within query language only
# ------------------------------------------------------------
def evaluate_native_own_language(records):
    ndcgs = []
    mrrs = []
    recalls = []

    for q_idx in records:
        q = frozen_benchmark[q_idx]

        lang = q["language"]
        qid = q["query_id"]

        # Query embedding
        q_emb = query_embeddings[q_idx]

        # ONLY passages from the query's language
        candidate_indices = np.asarray(
            lang_to_indices[lang],
            dtype=np.int32
        )

        candidate_emb = native_emb[candidate_indices]

        # Dense cosine/dot-product retrieval
        scores = candidate_emb @ q_emb

        order = np.argsort(-scores)[:TOP_K]
        ranked_indices = candidate_indices[order]

        # Relevant parent IDs for this query
        relevant_parent_ids = {
            native_lookup[(lang, qid, passage_idx)]
            for passage_idx, label in enumerate(q["is_selected"])
            if label == 1
        }

        # Parent-level relevance
        rel = np.array([
            1.0 if native_ids[idx] in relevant_parent_ids else 0.0
            for idx in ranked_indices
        ])

        ndcgs.append(ndcg_at_k(rel))
        mrrs.append(mrr_at_k(rel))
        recalls.append(
            recall_at_k(rel, len(relevant_parent_ids))
        )

    return {
        "n_queries": len(records),
        "nDCG@10": float(np.mean(ndcgs)),
        "MRR@10": float(np.mean(mrrs)),
        "Recall@10": float(np.mean(recalls)),
    }


# ------------------------------------------------------------
# 4. DEV / TEST
# ------------------------------------------------------------
dev_indices = [
    i for i, q in enumerate(frozen_benchmark)
    if q["split"] == "DEV"
]

test_indices = [
    i for i, q in enumerate(frozen_benchmark)
    if q["split"] == "TEST"
]

dev_results_1b = evaluate_native_own_language(dev_indices)
test_results_1b = evaluate_native_own_language(test_indices)

print("\n" + "=" * 60)
print("DIAGNOSTIC 1b — NATIVE / OWN-LANGUAGE RETRIEVAL")
print("=" * 60)

print("\nDEV")
for k, v in dev_results_1b.items():
    print(f"{k:>12}: {v:.6f}" if isinstance(v, float) else f"{k:>12}: {v}")

print("\nTEST")
for k, v in test_results_1b.items():
    print(f"{k:>12}: {v:.6f}" if isinstance(v, float) else f"{k:>12}: {v}")

print("\nDone.")

Native passages by language:
  bengali: 997
  gujarati: 997
  hindi: 997
  kannada: 997
  malayalam: 997
  marathi: 997
  punjabi: 997
  tamil: 997
  telugu: 997

DIAGNOSTIC 1b — NATIVE / OWN-LANGUAGE RETRIEVAL

DEV
   n_queries: 0
     nDCG@10: nan
      MRR@10: nan
   Recall@10: nan

TEST
   n_queries: 0
     nDCG@10: nan
      MRR@10: nan
   Recall@10: nan

Done.


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
print("Number of frozen records:", len(frozen_benchmark))

print("\nFirst record keys:")
print(frozen_benchmark[0].keys())

print("\nFirst record:")
print(frozen_benchmark[0])

print("\nUnique split values, if present:")
if "split" in frozen_benchmark[0]:
    print(sorted(set(q["split"] for q in frozen_benchmark)))
else:
    print("NO 'split' FIELD")

Number of frozen records: 459

First record keys:
dict_keys(['language', 'query_id', 'query', 'passages', 'is_selected', 'split'])

First record:
{'language': 'hindi', 'query_id': 1090350, 'query': 'वयस्कों में विस्थापन में सीमावर्ती व्यक्तित्व विकार के लक्षण', 'passages': ['सीमावर्ती विकार के लिए नौ डीएसएम-IV मानदंडों में से तीन इस समूह में आते हैं: 1. प्रभावी (भावनात्मक) अस्थिरता, जिसमें तीव्र, प्रकरणिक भावनात्मक पीड़ा, चिड़चिड़ापन और चिंता/घबराहट के दौरे शामिल हैं। 2. अनुचित, तीव्र और नियंत्रित करने में कठिन क्रोध। 3. खालीपन की पुरानी भावनाएं।', 'सीमावर्ती व्यक्तित्व विकार के उपचार में आमतौर पर एक चिकित्सक के साथ दीर्घकालिक मनोचिकित्सा शामिल होती है जिसे इस प्रकार के व्यक्तित्व विकार के उपचार में अनुभव होता है। विशिष्ट परेशान करने वाले और अक्षम करने वाले लक्षणों को कम करने में मदद के लिए दवाएं भी निर्धारित की जा सकती हैं।', 'सीमावर्ती व्यक्तित्व विकार के लक्षणों में शामिल हैंः रिश्तों में अस्थिरता का आवर्ती प्रतिरूप, परित्याग से बचने के प्रयास, पहचान में गड़बड़ी, आवेगशीलता, भावनात्म

In [ ]:
# ============================================================
# DIAGNOSTIC 1b — Native retrieval restricted to query language
# ============================================================

import numpy as np
from collections import defaultdict

TOP_K = 10

# ------------------------------------------------------------
# 1. Native corpus indices by language
# ------------------------------------------------------------
lang_to_indices = defaultdict(list)

for i, p in enumerate(native_corpus):
    lang_to_indices[p["language"]].append(i)

print("Native passages by language:")
for lang in sorted(lang_to_indices):
    print(f"  {lang}: {len(lang_to_indices[lang])}")

assert all(len(v) == 997 for v in lang_to_indices.values())

# ------------------------------------------------------------
# 2. Metric helpers
# ------------------------------------------------------------
def dcg_at_k(rel):
    return sum(
        r / np.log2(i + 2)
        for i, r in enumerate(rel[:TOP_K])
    )

def ndcg_at_k(rel):
    actual = dcg_at_k(rel)
    ideal = dcg_at_k(sorted(rel, reverse=True))
    return actual / ideal if ideal > 0 else 0.0

def mrr_at_k(rel):
    for i, r in enumerate(rel[:TOP_K]):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.0

def recall_at_k(rel, num_relevant):
    return sum(rel[:TOP_K]) / num_relevant if num_relevant > 0 else 0.0

# ------------------------------------------------------------
# 3. Evaluate within query language
# ------------------------------------------------------------
def evaluate_native_own_language(record_indices):

    ndcgs = []
    mrrs = []
    recalls = []

    for q_idx in record_indices:

        q = frozen_benchmark[q_idx]
        lang = q["language"]
        qid = q["query_id"]

        # Query embedding
        q_emb = query_embeddings[q_idx]

        # Restrict retrieval to the query's own language
        candidate_indices = np.asarray(
            lang_to_indices[lang],
            dtype=np.int32
        )

        scores = native_emb[candidate_indices] @ q_emb

        order = np.argsort(-scores)[:TOP_K]
        ranked_indices = candidate_indices[order]

        # Relevant parent IDs
        relevant_parent_ids = {
            native_lookup[(lang, qid, passage_idx)]
            for passage_idx, label in enumerate(q["is_selected"])
            if label == 1
        }

        # Parent-level binary relevance
        rel = np.array([
            1.0 if native_ids[idx] in relevant_parent_ids else 0.0
            for idx in ranked_indices
        ])

        ndcgs.append(ndcg_at_k(rel))
        mrrs.append(mrr_at_k(rel))
        recalls.append(
            recall_at_k(rel, len(relevant_parent_ids))
        )

    return {
        "n_queries": len(record_indices),
        "nDCG@10": float(np.mean(ndcgs)),
        "MRR@10": float(np.mean(mrrs)),
        "Recall@10": float(np.mean(recalls)),
    }

# ------------------------------------------------------------
# 4. Correct split names: 'dev' / 'test'
# ------------------------------------------------------------
dev_indices = [
    i for i, q in enumerate(frozen_benchmark)
    if q["split"] == "dev"
]

test_indices = [
    i for i, q in enumerate(frozen_benchmark)
    if q["split"] == "test"
]

print(f"\nDEV queries : {len(dev_indices)}")
print(f"TEST queries: {len(test_indices)}")

# ------------------------------------------------------------
# 5. Run diagnostic
# ------------------------------------------------------------
dev_results_1b = evaluate_native_own_language(dev_indices)
test_results_1b = evaluate_native_own_language(test_indices)

print("\n" + "=" * 60)
print("DIAGNOSTIC 1b — NATIVE / OWN-LANGUAGE RETRIEVAL")
print("=" * 60)

print("\nDEV")
for k, v in dev_results_1b.items():
    if isinstance(v, float):
        print(f"{k:>12}: {v:.6f}")
    else:
        print(f"{k:>12}: {v}")

print("\nTEST")
for k, v in test_results_1b.items():
    if isinstance(v, float):
        print(f"{k:>12}: {v:.6f}")
    else:
        print(f"{k:>12}: {v}")

Native passages by language:
  bengali: 997
  gujarati: 997
  hindi: 997
  kannada: 997
  malayalam: 997
  marathi: 997
  punjabi: 997
  tamil: 997
  telugu: 997

DEV queries : 225
TEST queries: 234

DIAGNOSTIC 1b — NATIVE / OWN-LANGUAGE RETRIEVAL

DEV
   n_queries: 225
     nDCG@10: 0.551198
      MRR@10: 0.452651
   Recall@10: 0.846667

TEST
   n_queries: 234
     nDCG@10: 0.551068
      MRR@10: 0.453867
   Recall@10: 0.858974


In [ ]:
# ============================================================
# DIAGNOSTIC 1c — Cross-language passage alignment
# ============================================================

from collections import defaultdict
import numpy as np

# ------------------------------------------------------------
# Build lookup:
# (language, query_id, passage_idx) -> passage text
# ------------------------------------------------------------
passage_lookup = {}

for q in frozen_benchmark:
    lang = q["language"]
    qid = q["query_id"]

    for idx, text in enumerate(q["passages"]):
        passage_lookup[(lang, qid, idx)] = text.strip()


# ------------------------------------------------------------
# Group passages by (query_id, passage_idx)
# ------------------------------------------------------------
groups = defaultdict(list)

for (lang, qid, idx), text in passage_lookup.items():
    groups[(qid, idx)].append((lang, text))


# ------------------------------------------------------------
# Compare cross-language versions
# ------------------------------------------------------------
total_groups = 0
multi_language_groups = 0
exact_same = 0
different_text = 0

examples = []

for (qid, idx), versions in groups.items():

    # Only groups represented in multiple languages
    if len(versions) < 2:
        continue

    total_groups += 1
    multi_language_groups += 1

    texts = [text for _, text in versions]

    if len(set(texts)) == 1:
        exact_same += 1
    else:
        different_text += 1

        # Keep a few examples for manual inspection
        if len(examples) < 10:
            examples.append({
                "query_id": qid,
                "passage_idx": idx,
                "languages": [lang for lang, _ in versions],
                "texts": texts
            })


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 65)
print("DIAGNOSTIC 1c — CROSS-LANGUAGE PASSAGE ALIGNMENT")
print("=" * 65)

print(f"\nMulti-language aligned groups : {total_groups}")
print(f"Exact same text               : {exact_same}")
print(f"Different text                : {different_text}")

if total_groups > 0:
    print(
        f"Exact-text proportion         : "
        f"{exact_same / total_groups:.4%}"
    )
    print(
        f"Different-text proportion     : "
        f"{different_text / total_groups:.4%}"
    )


# ------------------------------------------------------------
# Show examples of different-language versions
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("SAMPLE CROSS-LANGUAGE VERSIONS")
print("=" * 65)

for i, ex in enumerate(examples, 1):
    print(f"\n--- Example {i} ---")
    print(f"query_id     : {ex['query_id']}")
    print(f"passage_idx  : {ex['passage_idx']}")

    for lang, text in zip(ex["languages"], ex["texts"]):
        print(f"\n[{lang}]")
        print(text[:500])

DIAGNOSTIC 1c — CROSS-LANGUAGE PASSAGE ALIGNMENT

Multi-language aligned groups : 507
Exact same text               : 0
Different text                : 507
Exact-text proportion         : 0.0000%
Different-text proportion     : 100.0000%

SAMPLE CROSS-LANGUAGE VERSIONS

--- Example 1 ---
query_id     : 1090350
passage_idx  : 0

[hindi]
सीमावर्ती विकार के लिए नौ डीएसएम-IV मानदंडों में से तीन इस समूह में आते हैं: 1. प्रभावी (भावनात्मक) अस्थिरता, जिसमें तीव्र, प्रकरणिक भावनात्मक पीड़ा, चिड़चिड़ापन और चिंता/घबराहट के दौरे शामिल हैं। 2. अनुचित, तीव्र और नियंत्रित करने में कठिन क्रोध। 3. खालीपन की पुरानी भावनाएं।

[bengali]
সীমান্তবর্তী ব্যাধির জন্য নয়টি ডিএসএম-IV মানদণ্ডের মধ্যে তিনটি এই গ্রুপে পড়ে: ১ তীব্র, পর্বকালীন আবেগজনিত যন্ত্রণা, চিড়চিড়ে মেজাজ এবং উদ্বেগ/আতঙ্কের আক্রমণ সহ আবেগজনিত (আবেগজনিত) অস্থিরতা। ২ অনুপযুক্ত, তীব্র এবং নিয়ন্ত্রণ করা কঠিন ক্রোধ। ৩ দীর্ঘস্থায়ী শূন্যতার অনুভূতি।

[tamil]
எல்லைக்கும் அப்பால் உள்ள கோளாறுகளுக்கான ஒன்பது டிஎஸ்எம்-IV அளவுகோல்களில், மூன்று இந்த குழ

In [ ]:
# ============================================================
# DIAGNOSTIC 1d — Final embedding / ID sanity check
# ============================================================

import numpy as np
import torch

# ------------------------------------------------------------
# 1. Basic cache alignment checks
# ------------------------------------------------------------
print("=" * 65)
print("DIAGNOSTIC 1d — EMBEDDING / ID SANITY CHECK")
print("=" * 65)

print("\nCache shapes:")
print("native_emb :", native_emb.shape)
print("native_ids :", len(native_ids))
print("native_corpus:", len(native_corpus))

assert len(native_emb) == len(native_ids) == len(native_corpus)

# Verify IDs correspond to corpus rows
for i in [0, len(native_ids)//2, len(native_ids)-1]:
    assert native_ids[i] == native_corpus[i]["parent_id"]

print("\n✓ Native embedding rows, IDs, and corpus rows are aligned.")

# ------------------------------------------------------------
# 2. Check cached embedding normalization
# ------------------------------------------------------------
sample_norms = np.linalg.norm(
    native_emb[np.linspace(
        0,
        len(native_emb) - 1,
        100,
        dtype=int
    )],
    axis=1
)

print("\nCached embedding norm check:")
print(f"  min  : {sample_norms.min():.9f}")
print(f"  mean : {sample_norms.mean():.9f}")
print(f"  max  : {sample_norms.max():.9f}")

assert np.allclose(sample_norms, 1.0, atol=1e-5)

print("✓ Cached embeddings are L2-normalized.")

# ------------------------------------------------------------
# 3. Fresh one-at-a-time FP32 embeddings
# ------------------------------------------------------------
# Use the already-loaded Qwen model.
# Do NOT alter the cached embeddings.

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    len(native_corpus),
    size=20,
    replace=False
)

fresh_vectors = []
cached_vectors = []

model_was_training = qwen_model.training
qwen_model.eval()

with torch.no_grad():
    for idx in sample_indices:

        text = native_corpus[idx]["text"]

        # Fresh FP32 embedding
        output = qwen_model.encode(
            [text],
            normalize_embeddings=False
        )

        vec = np.asarray(output[0], dtype=np.float32)

        # Explicit FP32 L2 normalization
        vec = vec / np.linalg.norm(vec)

        fresh_vectors.append(vec)
        cached_vectors.append(
            native_emb[idx].astype(np.float32)
        )

if model_was_training:
    qwen_model.train()

fresh_vectors = np.vstack(fresh_vectors)
cached_vectors = np.vstack(cached_vectors)

# ------------------------------------------------------------
# 4. Compare cached vs fresh vectors
# ------------------------------------------------------------
cosines = np.sum(
    fresh_vectors * cached_vectors,
    axis=1
)

max_abs_diff = np.max(
    np.abs(fresh_vectors - cached_vectors),
    axis=1
)

print("\nFresh FP32 vs cached embedding comparison:")
print(f"  cosine min  : {cosines.min():.9f}")
print(f"  cosine mean : {cosines.mean():.9f}")
print(f"  cosine max  : {cosines.max():.9f}")
print(f"  max vector absolute difference: {max_abs_diff.max():.9e}")

print("\nSample results:")
for idx, cos, diff in zip(sample_indices[:10], cosines[:10], max_abs_diff[:10]):
    print(
        f"  row {idx:5d} | "
        f"cosine={cos:.9f} | "
        f"max_abs_diff={diff:.3e}"
    )

# Very high cosine means the cache reproduces the same embedding.
assert cosines.min() > 0.999

print("\n✓ Independent FP32 re-embedding agrees with cached vectors.")
print("\n" + "=" * 65)
print("DIAGNOSTIC 1d COMPLETE")
print("=" * 65)

DIAGNOSTIC 1d — EMBEDDING / ID SANITY CHECK

Cache shapes:
native_emb : (8973, 1024)
native_ids : 8973
native_corpus: 8973

✓ Native embedding rows, IDs, and corpus rows are aligned.

Cached embedding norm check:
  min  : 0.999999881
  mean : 1.000000000
  max  : 1.000000119
✓ Cached embeddings are L2-normalized.


NameError: name 'qwen_model' is not defined

In [ ]:
# Find the currently loaded Qwen model variable
import gc

print("Possible model variables:")

for name, obj in list(globals().items()):
    if hasattr(obj, "encode") and hasattr(obj, "eval"):
        print(f"  {name}: {type(obj)}")

Possible model variables:
  SentenceTransformer: <class 'abc.ABCMeta'>
  model: <class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [ ]:
# ============================================================
# DIAGNOSTIC 1d — Independent FP32 embedding verification
# ============================================================

import numpy as np
import torch

rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(native_corpus),
    size=20,
    replace=False
)

fresh_vectors = []
cached_vectors = []

model.eval()

with torch.no_grad():
    for idx in sample_indices:

        text = native_corpus[idx]["text"]

        # Fresh embedding, explicitly without normalization
        output = model.encode(
            [text],
            normalize_embeddings=False
        )

        vec = np.asarray(output[0], dtype=np.float32)

        # Explicit FP32 L2 normalization
        vec = vec / np.linalg.norm(vec)

        fresh_vectors.append(vec)
        cached_vectors.append(
            native_emb[idx].astype(np.float32)
        )

fresh_vectors = np.vstack(fresh_vectors)
cached_vectors = np.vstack(cached_vectors)

# ------------------------------------------------------------
# Compare fresh vs cached
# ------------------------------------------------------------
cosines = np.sum(
    fresh_vectors * cached_vectors,
    axis=1
)

max_abs_diff = np.max(
    np.abs(fresh_vectors - cached_vectors),
    axis=1
)

print("=" * 65)
print("DIAGNOSTIC 1d — FRESH FP32 vs CACHED EMBEDDINGS")
print("=" * 65)

print(f"\nCosine similarity:")
print(f"  min  : {cosines.min():.9f}")
print(f"  mean : {cosines.mean():.9f}")
print(f"  max  : {cosines.max():.9f}")

print(f"\nMaximum vector absolute difference:")
print(f"  max : {max_abs_diff.max():.9e}")

print("\nFirst 10 samples:")
for idx, cos, diff in zip(
    sample_indices[:10],
    cosines[:10],
    max_abs_diff[:10]
):
    print(
        f"  row {idx:5d} | "
        f"cosine={cos:.9f} | "
        f"max_abs_diff={diff:.3e}"
    )

assert cosines.min() > 0.999

print("\n✓ Fresh FP32 embeddings agree with cached embeddings.")
print("✓ DIAGNOSTIC 1d COMPLETE")
print("\nAll Native baseline sanity checks are now complete.")

DIAGNOSTIC 1d — FRESH FP32 vs CACHED EMBEDDINGS

Cosine similarity:
  min  : 0.999999404
  mean : 1.000000000
  max  : 1.000000119

Maximum vector absolute difference:
  max : 2.451539040e-04

First 10 samples:
  row  6930 | cosine=0.999999881 | max_abs_diff=0.000e+00
  row  6434 | cosine=1.000000000 | max_abs_diff=1.490e-08
  row  1805 | cosine=1.000000119 | max_abs_diff=0.000e+00
  row  8746 | cosine=1.000000000 | max_abs_diff=0.000e+00
  row  7534 | cosine=0.999999940 | max_abs_diff=0.000e+00
  row  6825 | cosine=1.000000119 | max_abs_diff=0.000e+00
  row  1149 | cosine=0.999999940 | max_abs_diff=0.000e+00
  row   799 | cosine=1.000000000 | max_abs_diff=7.451e-09
  row  5862 | cosine=0.999999940 | max_abs_diff=0.000e+00
  row  4603 | cosine=0.999999821 | max_abs_diff=0.000e+00

✓ Fresh FP32 embeddings agree with cached embeddings.
✓ DIAGNOSTIC 1d COMPLETE

All Native baseline sanity checks are now complete.


In [ ]:
# ============================================================
# METHOD 2 — FIXED 2-SENTENCE CHUNKING
# Cell 1: Build chunk corpus
# ============================================================

fixed2_corpus = []

for passage in native_corpus:
    text = passage["text"]
    parent_id = passage["parent_id"]
    language = passage["language"]

    sentences = split_sentences(text)

    # Group consecutive sentences in pairs
    chunks = [
        " ".join(sentences[i:i+2])
        for i in range(0, len(sentences), 2)
    ]

    for chunk_idx, chunk_text in enumerate(chunks):
        fixed2_corpus.append({
            "chunk_id": f"{parent_id}_fixed2_{chunk_idx}",
            "parent_id": parent_id,
            "language": language,
            "chunk_idx": chunk_idx,
            "text": chunk_text,
            "num_sentences": min(2, len(sentences) - chunk_idx * 2)
        })

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("=" * 65)
print("METHOD 2 — FIXED 2-SENTENCE CHUNKING")
print("=" * 65)

print(f"\nNative passages : {len(native_corpus)}")
print(f"Fixed-2 chunks  : {len(fixed2_corpus)}")

# Every chunk must have a valid parent
assert all(
    c["parent_id"] in native_ids
    for c in fixed2_corpus
)

# Every chunk must contain 1 or 2 sentences
assert all(
    c["num_sentences"] in (1, 2)
    for c in fixed2_corpus
)

# Language distribution
from collections import Counter

lang_counts = Counter(
    c["language"] for c in fixed2_corpus
)

print("\nChunks by language:")
for lang in sorted(lang_counts):
    print(f"  {lang}: {lang_counts[lang]}")

# ------------------------------------------------------------
# Show a few examples
# ------------------------------------------------------------

print("\nSample chunks:")

for c in fixed2_corpus[:5]:
    print(f"\n[{c['language']}] {c['chunk_id']}")
    print(f"sentences: {c['num_sentences']}")
    print(c["text"][:400])

print("\n✓ Fixed-2 chunk corpus constructed successfully.")

METHOD 2 — FIXED 2-SENTENCE CHUNKING

Native passages : 8973
Fixed-2 chunks  : 18171

Chunks by language:
  bengali: 1875
  gujarati: 2028
  hindi: 2038
  kannada: 2086
  malayalam: 2021
  marathi: 2111
  punjabi: 2029
  tamil: 1995
  telugu: 1988

Sample chunks:

[hindi] hindi_1102432_p00_fixed2_0
sentences: 2
एक कंपनी एक विशिष्ट देश में निगमित होती है, अक्सर उस देश के एक छोटे उपसमूह, जैसे कि एक राज्य या प्रांत, की सीमाओं के भीतर। निगम तब उस राज्य में निगमन के कानूनों द्वारा शासित होता है।

[hindi] hindi_1102432_p00_fixed2_1
sentences: 2
एक निगम या तो निजी या सार्वजनिक स्टॉक जारी कर सकता है, या इसे गैर-स्टॉक निगम के रूप में वर्गीकृत किया जा सकता है। यदि स्टॉक जारी किया जाता है, तो निगम आमतौर पर अपने शेयरधारकों द्वारा सीधे या परोक्ष रूप से शासित होगा।

[hindi] hindi_1102432_p01_fixed2_0
sentences: 2
आज, 50 देशों से 2,100 से अधिक प्रमाणित बी कोर का एक बढ़ता हुआ समुदाय है और 130 से अधिक उद्योग एक एकीकृत लक्ष्य की दिशा में मिलकर काम कर रहे हैं: व्यवसाय में सफलता को पुनः परिभाषित करना। आंद

In [ ]:
# ============================================================
# FINAL CHECKPOINT — VERIFY EVERYTHING IS ON GOOGLE DRIVE
# ============================================================

from pathlib import Path
import json
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/maxmin_research")

assert DRIVE_ROOT.exists(), f"Drive path not found: {DRIVE_ROOT}"

print("=" * 70)
print("FINAL DRIVE CHECKPOINT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Expected important artifacts
# ------------------------------------------------------------

expected = {
    "Native embeddings":
        DRIVE_ROOT / "cache/qwen_native_embeddings/qwen_native_embeddings.npy",

    "Native IDs":
        DRIVE_ROOT / "cache/qwen_native_embeddings/parent_ids.json",

    "Qwen sentence embeddings":
        DRIVE_ROOT / "cache/qwen_sentence_embeddings/qwen_sentence_embeddings.npy",

    "Sentence IDs":
        DRIVE_ROOT / "cache/qwen_sentence_embeddings/sentence_ids.json",

    "Qwen query embeddings":
        DRIVE_ROOT / "cache/qwen_query_embeddings/qwen_query_embeddings.npy",

    "Frozen benchmark":
        DRIVE_ROOT / "frozen/msmarco_xi_frozen_benchmark.json",

    "Native corpus":
        DRIVE_ROOT / "frozen/msmarco_xi_native_corpus.json",

    "Experiment manifest":
        DRIVE_ROOT / "frozen/experiment_001_manifest.json",

    "Sentence corpus":
        DRIVE_ROOT / "frozen/msmarco_xi_sentence_corpus.json",
}

print("\nARTIFACT STATUS\n")

missing = []

for name, path in expected.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✓ {name:<25} {size_mb:8.2f} MB")
    else:
        print(f"✗ MISSING: {name}")
        missing.append(name)

# ------------------------------------------------------------
# 2. Save the exact sentence splitter used
# ------------------------------------------------------------

splitter_code = r'''import re

SENTENCE_PATTERN = re.compile(r'(?<=[.!?।॥！？])\s+')

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    sentences = SENTENCE_PATTERN.split(text)
    return [s.strip() for s in sentences if s.strip()]
'''

splitter_path = DRIVE_ROOT / "frozen/sentence_splitter.py"
splitter_path.parent.mkdir(parents=True, exist_ok=True)
splitter_path.write_text(splitter_code, encoding="utf-8")

print(f"\n✓ Sentence splitter saved:")
print(f"  {splitter_path}")

# ------------------------------------------------------------
# 3. Save the corrected Native results
# ------------------------------------------------------------

native_results = {
    "experiment": "Native baseline — corrected own-language retrieval",
    "dataset": "AI4Bharat MSMARCO-XI",
    "model": "Qwen/Qwen3-Embedding-0.6B",
    "languages": [
        "hindi", "bengali", "tamil", "telugu", "malayalam",
        "kannada", "marathi", "gujarati", "punjabi"
    ],
    "total_queries": 459,
    "dev_queries": 225,
    "test_queries": 234,
    "native_passages_per_language": 997,
    "total_native_passages": 8973,
    "retrieval_candidate_pool": "query language only",
    "top_k": 10,
    "metrics": {
        "dev": {
            "nDCG@10": 0.551198,
            "MRR@10": 0.452651,
            "Recall@10": 0.846667
        },
        "test": {
            "nDCG@10": 0.551068,
            "MRR@10": 0.453867,
            "Recall@10": 0.858974
        }
    }
}

results_dir = DRIVE_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

json_path = results_dir / "native_baseline_corrected.json"
txt_path = results_dir / "native_baseline_corrected.txt"

json_path.write_text(
    json.dumps(native_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

txt_content = """NATIVE BASELINE — CORRECTED OWN-LANGUAGE RETRIEVAL

Dataset: AI4Bharat MSMARCO-XI
Model: Qwen/Qwen3-Embedding-0.6B
Queries: 459
DEV: 225
TEST: 234
Native passages/language: 997
Total native passages: 8973
Retrieval pool: query language only
Top-K: 10

DEV
nDCG@10: 0.551198
MRR@10: 0.452651
Recall@10: 0.846667

TEST
nDCG@10: 0.551068
MRR@10: 0.453867
Recall@10: 0.858974

This is the frozen Native reference baseline for subsequent experiments.
"""

txt_path.write_text(txt_content, encoding="utf-8")

print(f"\n✓ Native results saved:")
print(f"  {json_path}")
print(f"  {txt_path}")

# ------------------------------------------------------------
# 4. Save a simple checkpoint manifest
# ------------------------------------------------------------

checkpoint = {
    "status": "Native baseline validated and frozen",
    "drive_root": str(DRIVE_ROOT),
    "missing_artifacts": missing,
    "native_results_file": str(json_path),
    "sentence_splitter_file": str(splitter_path),
}

checkpoint_path = results_dir / "checkpoint_native_complete.json"

checkpoint_path.write_text(
    json.dumps(checkpoint, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)

if missing:
    print("WARNING — SOME EXPECTED ARTIFACTS ARE MISSING:")
    for item in missing:
        print("  ✗", item)
else:
    print("✓ ALL IMPORTANT ARTIFACTS VERIFIED ON GOOGLE DRIVE")

print("✓ Native results saved")
print("✓ Sentence splitter saved")
print("✓ Checkpoint manifest saved")
print("=" * 70)

FINAL DRIVE CHECKPOINT

ARTIFACT STATUS

✓ Native embeddings            35.05 MB
✓ Native IDs                    0.19 MB
✓ Qwen sentence embeddings    124.19 MB
✓ Sentence IDs                  0.81 MB
✓ Qwen query embeddings         1.79 MB
✓ Frozen benchmark              3.98 MB
✓ Native corpus                 8.64 MB
✓ Experiment manifest           0.00 MB
✓ Sentence corpus              11.39 MB

✓ Sentence splitter saved:
  /content/drive/MyDrive/maxmin_research/frozen/sentence_splitter.py

✓ Native results saved:
  /content/drive/MyDrive/maxmin_research/results/native_baseline_corrected.json
  /content/drive/MyDrive/maxmin_research/results/native_baseline_corrected.txt

✓ ALL IMPORTANT ARTIFACTS VERIFIED ON GOOGLE DRIVE
✓ Native results saved
✓ Sentence splitter saved
✓ Checkpoint manifest saved


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import re
import time
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/maxmin_research"
CACHE = os.path.join(BASE, "cache")
FROZEN = os.path.join(BASE, "frozen")
RESULTS = os.path.join(BASE, "results")

print("Drive mounted.")
print("BASE:", BASE)
print("GPU:", os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read().strip())

Mounted at /content/drive
Drive mounted.
BASE: /content/drive/MyDrive/maxmin_research
GPU: Tesla T4


In [ ]:
with open(os.path.join(FROZEN, "msmarco_xi_frozen_benchmark.json"), "r", encoding="utf-8") as f:
    frozen_benchmark = json.load(f)

with open(os.path.join(FROZEN, "msmarco_xi_native_corpus.json"), "r", encoding="utf-8") as f:
    native_corpus = json.load(f)

with open(os.path.join(FROZEN, "msmarco_xi_sentence_corpus.json"), "r", encoding="utf-8") as f:
    sentence_corpus = json.load(f)

print("Frozen benchmark:", len(frozen_benchmark))
print("Native corpus:", len(native_corpus))
print("Sentence corpus:", len(sentence_corpus))

print("Splits:", sorted(set(q["split"] for q in frozen_benchmark)))
print("Languages:", sorted(set(q["language"] for q in frozen_benchmark)))

Frozen benchmark: 459
Native corpus: 8973
Sentence corpus: 31792
Splits: ['dev', 'test']
Languages: ['bengali', 'gujarati', 'hindi', 'kannada', 'malayalam', 'marathi', 'punjabi', 'tamil', 'telugu']


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

print("Loading Qwen3-Embedding-0.6B...")

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

query_emb_path = os.path.join(
    CACHE, "qwen_query_embeddings", "qwen_query_embeddings.npy"
)

query_embeddings = np.load(query_emb_path)

# Explicit L2 normalization, matching the frozen protocol
query_embeddings = query_embeddings / np.linalg.norm(
    query_embeddings, axis=1, keepdims=True
)

print("Model loaded.")
print("Device:", model.device)
print("Query embeddings:", query_embeddings.shape)
print(
    "Norms:",
    query_embeddings.min(axis=1).min(),
    query_embeddings.mean(axis=1).mean(),
    query_embeddings.max(axis=1).max()
)

Loading Qwen3-Embedding-0.6B...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Model loaded.
Device: cuda:0
Query embeddings: (459, 1024)
Norms: -0.17905982 6.376247e-05 0.16562322


In [ ]:
query_norms = np.linalg.norm(query_embeddings, axis=1)

print("Query embeddings:", query_embeddings.shape)
print(
    "Actual norms:",
    query_norms.min(),
    query_norms.mean(),
    query_norms.max()
)

assert np.allclose(query_norms, 1.0, atol=1e-6)

print("✓ Query embeddings are correctly L2-normalized.")

Query embeddings: (459, 1024)
Actual norms: 0.9999999 1.0 1.0000001
✓ Query embeddings are correctly L2-normalized.


In [ ]:
fixed2_corpus = []

for passage in native_corpus:
    text = passage["text"]
    parent_id = passage["parent_id"]
    language = passage["language"]

    sentences = split_sentences(text)

    chunks = [
        " ".join(sentences[i:i+2])
        for i in range(0, len(sentences), 2)
    ]

    for chunk_idx, chunk_text in enumerate(chunks):
        fixed2_corpus.append({
            "chunk_id": f"{parent_id}_fixed2_{chunk_idx}",
            "parent_id": parent_id,
            "language": language,
            "chunk_idx": chunk_idx,
            "text": chunk_text,
            "num_sentences": min(2, len(sentences) - chunk_idx * 2)
        })

print("Native passages:", len(native_corpus))
print("Fixed-2 chunks:", len(fixed2_corpus))

print("\nChunks by language:")
print(
    pd.Series([x["language"] for x in fixed2_corpus])
    .value_counts()
    .sort_index()
)

print("\nSample:")
for x in fixed2_corpus[:3]:
    print(
        x["language"],
        "|",
        x["num_sentences"],
        "sentences |",
        x["text"][:150]
    )

Native passages: 8973
Fixed-2 chunks: 18171

Chunks by language:
bengali      1875
gujarati     2028
hindi        2038
kannada      2086
malayalam    2021
marathi      2111
punjabi      2029
tamil        1995
telugu       1988
Name: count, dtype: int64

Sample:
hindi | 2 sentences | एक कंपनी एक विशिष्ट देश में निगमित होती है, अक्सर उस देश के एक छोटे उपसमूह, जैसे कि एक राज्य या प्रांत, की सीमाओं के भीतर। निगम तब उस राज्य में निगमन 
hindi | 2 sentences | एक निगम या तो निजी या सार्वजनिक स्टॉक जारी कर सकता है, या इसे गैर-स्टॉक निगम के रूप में वर्गीकृत किया जा सकता है। यदि स्टॉक जारी किया जाता है, तो निगम
hindi | 2 sentences | आज, 50 देशों से 2,100 से अधिक प्रमाणित बी कोर का एक बढ़ता हुआ समुदाय है और 130 से अधिक उद्योग एक एकीकृत लक्ष्य की दिशा में मिलकर काम कर रहे हैं: व्यवस


In [ ]:
splitter_path = os.path.join(FROZEN, "sentence_splitter.py")

with open(splitter_path, "r", encoding="utf-8") as f:
    splitter_code = f.read()

exec(splitter_code)

print("Loaded frozen sentence splitter.")
print(splitter_code)

Loaded frozen sentence splitter.
import re

SENTENCE_PATTERN = re.compile(r'(?<=[.!?।॥！？])\s+')

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    sentences = SENTENCE_PATTERN.split(text)
    return [s.strip() for s in sentences if s.strip()]



In [ ]:
fixed2_cache_dir = os.path.join(CACHE, "qwen_fixed2_embeddings")
fixed2_emb_path = os.path.join(fixed2_cache_dir, "qwen_fixed2_embeddings.npy")
fixed2_ids_path = os.path.join(fixed2_cache_dir, "fixed2_chunk_ids.json")

print("Fixed-2 embedding cache exists:", os.path.exists(fixed2_emb_path))
print("Fixed-2 ID cache exists:", os.path.exists(fixed2_ids_path))

if os.path.exists(fixed2_emb_path):
    print("Embedding file size:",
          round(os.path.getsize(fixed2_emb_path) / (1024**2), 2), "MB")

Fixed-2 embedding cache exists: False
Fixed-2 ID cache exists: False


In [ ]:
import gc
import torch

os.makedirs(fixed2_cache_dir, exist_ok=True)

texts = [x["text"] for x in fixed2_corpus]
chunk_ids = [x["chunk_id"] for x in fixed2_corpus]

batch_size = 4
checkpoint_every = 1000

all_embeddings = []
all_ids = []

start_time = time.time()

for start in range(0, len(texts), checkpoint_every):
    end = min(start + checkpoint_every, len(texts))

    print(f"\nEmbedding chunks {start} → {end} / {len(texts)}")

    chunk_embeddings = []

    for batch_start in range(start, end, batch_size):
        batch_end = min(batch_start + batch_size, end)

        emb = model.encode(
            texts[batch_start:batch_end],
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )

        emb = emb / np.linalg.norm(
            emb, axis=1, keepdims=True
        )

        chunk_embeddings.append(emb.astype(np.float32))

        del emb

    checkpoint_embeddings = np.vstack(chunk_embeddings)

    all_embeddings.append(checkpoint_embeddings)
    all_ids.extend(chunk_ids[start:end])

    # Save checkpoint
    current_embeddings = np.vstack(all_embeddings)

    np.save(
        os.path.join(
            fixed2_cache_dir,
            "fixed2_checkpoint.npy"
        ),
        current_embeddings
    )

    with open(
        os.path.join(
            fixed2_cache_dir,
            "fixed2_checkpoint_ids.json"
        ),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(all_ids, f)

    del checkpoint_embeddings
    del current_embeddings
    del chunk_embeddings

    gc.collect()
    torch.cuda.empty_cache()

    elapsed = (time.time() - start_time) / 60

    print(
        f"✓ Saved checkpoint: {end}/{len(texts)} "
        f"({100*end/len(texts):.1f}%)"
        f" | elapsed: {elapsed:.1f} min"
    )

# Final result
fixed2_embeddings = np.vstack(all_embeddings)

np.save(
    fixed2_emb_path,
    fixed2_embeddings
)

with open(fixed2_ids_path, "w", encoding="utf-8") as f:
    json.dump(chunk_ids, f)

norms = np.linalg.norm(fixed2_embeddings, axis=1)

print("\n" + "=" * 50)
print("FIXED-2 EMBEDDING COMPLETE")
print("=" * 50)
print("Shape:", fixed2_embeddings.shape)
print("IDs:", len(chunk_ids))
print(
    "Norms:",
    norms.min(),
    norms.mean(),
    norms.max()
)
print(
    "Total time:",
    round((time.time() - start_time) / 60, 2),
    "minutes"
)


Embedding chunks 0 → 1000 / 18171


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.64 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.55 GiB is free. Including non-PyTorch memory, this process has 10.01 GiB memory in use. Of the allocated memory 9.64 GiB is allocated by PyTorch, and 253.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU memory allocated: 7.15 GB
GPU memory reserved: 8.55 GB


In [ ]:
tokenizer = model.tokenizer

lengths = []

for text in texts:
    tokenized = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False
    )
    lengths.append(len(tokenized["input_ids"]))

lengths = np.array(lengths)

print("Fixed-2 chunks:", len(lengths))
print("Token length statistics:")
print("  min :", lengths.min())
print("  mean:", round(lengths.mean(), 2))
print("  median:", np.median(lengths))
print("  p90:", np.percentile(lengths, 90))
print("  p95:", np.percentile(lengths, 95))
print("  p99:", np.percentile(lengths, 99))
print("  max :", lengths.max())

print("\nChunks > 512 tokens:", np.sum(lengths > 512))
print("Chunks > 1024 tokens:", np.sum(lengths > 1024))
print("Chunks > 2048 tokens:", np.sum(lengths > 2048))

Fixed-2 chunks: 18171
Token length statistics:
  min : 3
  mean: 200.81
  median: 180.0
  p90: 350.0
  p95: 404.0
  p99: 501.2999999999993
  max : 7282

Chunks > 512 tokens: 153
Chunks > 1024 tokens: 19
Chunks > 2048 tokens: 19


In [ ]:
print("Model max sequence length:", model.max_seq_length)

print(
    "Transformer max position embeddings:",
    getattr(
        model[0].auto_model.config,
        "max_position_embeddings",
        "unknown"
    )
)

print(
    "Tokenizer model max length:",
    tokenizer.model_max_length
)

Model max sequence length: 32768
Transformer max position embeddings: 32768
Tokenizer model max length: 32768


In [ ]:
longest_idx = int(np.argmax(lengths))
longest_text = texts[longest_idx]

print("Longest chunk index:", longest_idx)
print("Token length:", lengths[longest_idx])
print("Chunk ID:", chunk_ids[longest_idx])

# Clear any cached GPU memory
gc.collect()
torch.cuda.empty_cache()

start = time.time()

test_emb = model.encode(
    [longest_text],
    batch_size=1,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=False
)

test_emb = test_emb / np.linalg.norm(
    test_emb, axis=1, keepdims=True
)

print("\n✓ Longest chunk embedded successfully.")
print("Embedding shape:", test_emb.shape)
print("Norm:", np.linalg.norm(test_emb[0]))
print("Time:", round(time.time() - start, 2), "seconds")

del test_emb
gc.collect()
torch.cuda.empty_cache()

Longest chunk index: 426
Token length: 7282
Chunk ID: hindi_165349_p00_fixed2_0

✓ Longest chunk embedded successfully.
Embedding shape: (1, 1024)
Norm: 0.99999994
Time: 10.33 seconds


In [ ]:
import gc
import torch

os.makedirs(fixed2_cache_dir, exist_ok=True)

texts = [x["text"] for x in fixed2_corpus]
chunk_ids = [x["chunk_id"] for x in fixed2_corpus]

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
CHECKPOINT_EVERY = 1000

# Token-length-aware batch limits
BATCH_SHORT = 16       # <= 256 tokens
BATCH_MEDIUM = 8       # <= 512 tokens
BATCH_LONG = 2         # <= 1024 tokens
BATCH_VERY_LONG = 1    # > 1024 tokens

# ---------------------------------------------------------
# Create token lengths if not already available
# ---------------------------------------------------------
if "lengths" not in globals() or len(lengths) != len(texts):
    print("Computing token lengths...")
    lengths = np.array([
        len(tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False
        )["input_ids"])
        for text in texts
    ])

print("Total chunks:", len(texts))
print("Max tokens:", lengths.max())

# ---------------------------------------------------------
# Helper: choose batch size from token length
# ---------------------------------------------------------
def choose_batch_size(token_length):
    if token_length <= 256:
        return BATCH_SHORT
    elif token_length <= 512:
        return BATCH_MEDIUM
    elif token_length <= 1024:
        return BATCH_LONG
    else:
        return BATCH_VERY_LONG

# ---------------------------------------------------------
# Embedding
# ---------------------------------------------------------
all_embeddings = []
all_ids = []

start_time = time.time()

for checkpoint_start in range(
    0,
    len(texts),
    CHECKPOINT_EVERY
):
    checkpoint_end = min(
        checkpoint_start + CHECKPOINT_EVERY,
        len(texts)
    )

    print(
        f"\nEmbedding chunks "
        f"{checkpoint_start} → {checkpoint_end} / {len(texts)}"
    )

    checkpoint_embeddings = []

    i = checkpoint_start

    while i < checkpoint_end:

        token_length = int(lengths[i])
        batch_size = choose_batch_size(token_length)

        batch_end = min(
            i + batch_size,
            checkpoint_end
        )

        batch_texts = texts[i:batch_end]

        # If any item in the batch is substantially longer,
        # reduce the batch size to the safest option.
        max_batch_tokens = int(lengths[i:batch_end].max())

        batch_size = choose_batch_size(max_batch_tokens)

        if batch_end - i > batch_size:
            batch_end = i + batch_size
            batch_texts = texts[i:batch_end]

        emb = model.encode(
            batch_texts,
            batch_size=len(batch_texts),
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )

        # Explicit L2 normalization
        emb = emb / np.linalg.norm(
            emb,
            axis=1,
            keepdims=True
        )

        checkpoint_embeddings.append(
            emb.astype(np.float32)
        )

        i = batch_end

        del emb
        gc.collect()

    checkpoint_embeddings = np.vstack(
        checkpoint_embeddings
    )

    all_embeddings.append(checkpoint_embeddings)
    all_ids.extend(
        chunk_ids[
            checkpoint_start:checkpoint_end
        ]
    )

    # -----------------------------------------------------
    # Save checkpoint to Drive
    # -----------------------------------------------------
    current_embeddings = np.vstack(
        all_embeddings
    )

    np.save(
        os.path.join(
            fixed2_cache_dir,
            "fixed2_checkpoint.npy"
        ),
        current_embeddings
    )

    with open(
        os.path.join(
            fixed2_cache_dir,
            "fixed2_checkpoint_ids.json"
        ),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            all_ids,
            f,
            ensure_ascii=False
        )

    elapsed = (
        time.time() - start_time
    ) / 60

    print(
        f"✓ Checkpoint saved: "
        f"{checkpoint_end}/{len(texts)} "
        f"({100 * checkpoint_end / len(texts):.1f}%)"
    )

    print(
        f"  Elapsed: {elapsed:.1f} min"
    )

    del current_embeddings
    del checkpoint_embeddings

    gc.collect()
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# Final save
# ---------------------------------------------------------
fixed2_embeddings = np.vstack(
    all_embeddings
)

np.save(
    fixed2_emb_path,
    fixed2_embeddings
)

with open(
    fixed2_ids_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        chunk_ids,
        f,
        ensure_ascii=False
    )

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------
norms = np.linalg.norm(
    fixed2_embeddings,
    axis=1
)

print("\n" + "=" * 60)
print("FIXED-2 EMBEDDING COMPLETE")
print("=" * 60)

print(
    "Embedding shape:",
    fixed2_embeddings.shape
)

print(
    "Chunk IDs:",
    len(chunk_ids)
)

print(
    "Norms:",
    norms.min(),
    norms.mean(),
    norms.max()
)

print(
    "Total time:",
    round(
        (time.time() - start_time) / 60,
        2
    ),
    "minutes"
)

assert fixed2_embeddings.shape == (
    len(fixed2_corpus),
    1024
)

assert len(chunk_ids) == len(fixed2_corpus)

assert np.allclose(
    norms,
    1.0,
    atol=1e-5
)

print("✓ Shape verified")
print("✓ IDs verified")
print("✓ L2 normalization verified")
print("✓ Fixed-2 embeddings saved to Drive")

Total chunks: 18171
Max tokens: 7282

Embedding chunks 0 → 1000 / 18171
✓ Checkpoint saved: 1000/18171 (5.5%)
  Elapsed: 2.7 min

Embedding chunks 1000 → 2000 / 18171
✓ Checkpoint saved: 2000/18171 (11.0%)
  Elapsed: 5.0 min

Embedding chunks 2000 → 3000 / 18171
✓ Checkpoint saved: 3000/18171 (16.5%)
  Elapsed: 8.0 min

Embedding chunks 3000 → 4000 / 18171
✓ Checkpoint saved: 4000/18171 (22.0%)
  Elapsed: 11.0 min

Embedding chunks 4000 → 5000 / 18171
✓ Checkpoint saved: 5000/18171 (27.5%)
  Elapsed: 14.3 min

Embedding chunks 5000 → 6000 / 18171
✓ Checkpoint saved: 6000/18171 (33.0%)
  Elapsed: 17.8 min

Embedding chunks 6000 → 7000 / 18171
✓ Checkpoint saved: 7000/18171 (38.5%)
  Elapsed: 21.9 min

Embedding chunks 7000 → 8000 / 18171
✓ Checkpoint saved: 8000/18171 (44.0%)
  Elapsed: 25.8 min

Embedding chunks 8000 → 9000 / 18171
✓ Checkpoint saved: 9000/18171 (49.5%)
  Elapsed: 29.7 min

Embedding chunks 9000 → 10000 / 18171
✓ Checkpoint saved: 10000/18171 (55.0%)
  Elapsed: 33.7 mi

In [ ]:
fixed2_embeddings_check = np.load(fixed2_emb_path)

with open(fixed2_ids_path, "r", encoding="utf-8") as f:
    fixed2_ids_check = json.load(f)

expected_ids = [x["chunk_id"] for x in fixed2_corpus]

print("Saved embeddings:", fixed2_embeddings_check.shape)
print("Saved IDs:", len(fixed2_ids_check))
print("Expected IDs:", len(expected_ids))

print(
    "ID order identical:",
    fixed2_ids_check == expected_ids
)

norms = np.linalg.norm(
    fixed2_embeddings_check,
    axis=1
)

print(
    "Norm range:",
    norms.min(),
    "→",
    norms.max()
)

assert fixed2_embeddings_check.shape == (18171, 1024)
assert fixed2_ids_check == expected_ids
assert np.allclose(norms, 1.0, atol=1e-5)

print("\n✓ Fixed-2 saved artifact is fully verified.")

Saved embeddings: (18171, 1024)
Saved IDs: 18171
Expected IDs: 18171
ID order identical: True
Norm range: 0.9999998 → 1.0000001

✓ Fixed-2 saved artifact is fully verified.


In [ ]:
# Use the verified saved embeddings
fixed2_embeddings = fixed2_embeddings_check
fixed2_ids = fixed2_ids_check

# Map each language to the row indices belonging to that language
fixed2_lang_indices = {}

for lang in sorted(set(x["language"] for x in fixed2_corpus)):
    indices = np.array(
        [
            i
            for i, x in enumerate(fixed2_corpus)
            if x["language"] == lang
        ],
        dtype=np.int32
    )

    fixed2_lang_indices[lang] = indices

print("Fixed-2 candidates by language:")

for lang, indices in fixed2_lang_indices.items():
    print(f"  {lang:10s}: {len(indices)}")

print("\nTotal:", sum(len(x) for x in fixed2_lang_indices.values()))

assert sum(len(x) for x in fixed2_lang_indices.values()) == 18171

print("\n✓ Fixed-2 language indexes constructed.")

Fixed-2 candidates by language:
  bengali   : 1875
  gujarati  : 2028
  hindi     : 2038
  kannada   : 2086
  malayalam : 2021
  marathi   : 2111
  punjabi   : 2029
  tamil     : 1995
  telugu    : 1988

Total: 18171

✓ Fixed-2 language indexes constructed.


In [ ]:
from sklearn.metrics import ndcg_score

def evaluate_fixed2(split_name):
    split_queries = [
        (i, q)
        for i, q in enumerate(frozen_benchmark)
        if q["split"] == split_name
    ]

    ndcg_scores = []
    mrr_scores = []
    recall_scores = []

    for q_idx, q in split_queries:
        lang = q["language"]

        # Query embedding
        query_vec = query_embeddings[q_idx]

        # Fixed-2 candidates in the query's language
        candidate_idx = fixed2_lang_indices[lang]

        candidate_emb = fixed2_embeddings[candidate_idx]

        # Exact cosine similarity because both vectors are normalized
        scores = candidate_emb @ query_vec

        # Sort chunks by similarity
        ranked_local = np.argsort(-scores)

        ranked_global = candidate_idx[ranked_local]

        # -------------------------------------------------
        # Collapse chunks to parent passages
        # Keep the highest-scoring chunk for each parent.
        # -------------------------------------------------
        parent_best_score = {}

        for idx in ranked_global:
            parent_id = fixed2_corpus[idx]["parent_id"]
            score = float(
                fixed2_embeddings[idx] @ query_vec
            )

            if (
                parent_id not in parent_best_score
                or score > parent_best_score[parent_id]
            ):
                parent_best_score[parent_id] = score

        ranked_parents = sorted(
            parent_best_score,
            key=parent_best_score.get,
            reverse=True
        )

        # -------------------------------------------------
        # Relevance mapping
        # Correct frozen key:
        # (language, query_id, passage_idx)
        # -------------------------------------------------
        relevance = {}

        for passage_idx, label in enumerate(q["is_selected"]):
            key = (
                lang,
                q["query_id"],
                passage_idx
            )

            # Find corresponding parent_id through frozen corpus
            passage_text = q["passages"][passage_idx]

            # Match exact passage text within native corpus
            # using language + text.
            matches = [
                x["parent_id"]
                for x in native_corpus
                if (
                    x["language"] == lang
                    and x["text"] == passage_text
                )
            ]

            if len(matches) != 1:
                raise ValueError(
                    f"Expected exactly one parent for {key}, "
                    f"found {len(matches)}"
                )

            relevance[matches[0]] = int(label)

        # Top-10 parent relevance
        top10 = ranked_parents[:10]

        gains = np.array(
            [relevance.get(pid, 0) for pid in top10],
            dtype=float
        )

        # nDCG@10
        if len(gains) < 10:
            gains = np.pad(
                gains,
                (0, 10 - len(gains))
            )

        ideal = np.sort(
            np.array(
                list(relevance.values()),
                dtype=float
            )
        )[::-1][:10]

        if len(ideal) < 10:
            ideal = np.pad(
                ideal,
                (0, 10 - len(ideal))
            )

        ndcg = ndcg_score(
            ideal.reshape(1, -1),
            gains.reshape(1, -1),
            k=10
        )

        # MRR@10
        rr = 0.0
        for rank, pid in enumerate(top10, start=1):
            if relevance.get(pid, 0) > 0:
                rr = 1.0 / rank
                break

        # Recall@10
        total_relevant = sum(relevance.values())

        retrieved_relevant = sum(
            relevance.get(pid, 0)
            for pid in top10
        )

        recall = (
            retrieved_relevant / total_relevant
            if total_relevant > 0
            else 0.0
        )

        ndcg_scores.append(ndcg)
        mrr_scores.append(rr)
        recall_scores.append(recall)

    return {
        "split": split_name,
        "queries": len(split_queries),
        "nDCG@10": float(np.mean(ndcg_scores)),
        "MRR@10": float(np.mean(mrr_scores)),
        "Recall@10": float(np.mean(recall_scores))
    }


print("Evaluating Fixed-2...\n")

fixed2_dev = evaluate_fixed2("dev")
fixed2_test = evaluate_fixed2("test")

print("DEV")
print(fixed2_dev)

print("\nTEST")
print(fixed2_test)

Evaluating Fixed-2...



ValueError: Expected exactly one parent for ('hindi', 126172, 5), found 2

In [ ]:
# Build the verified frozen mapping:
# (language, query_id, passage_idx) -> parent_id

passage_key_to_parent = {}

for q in frozen_benchmark:
    lang = q["language"]
    query_id = q["query_id"]

    for passage_idx, passage_text in enumerate(q["passages"]):
        key = (
            lang,
            query_id,
            passage_idx
        )

        # Find the native passage by matching the frozen
        # query's passage against the language-specific corpus.
        #
        # We allow duplicate text, but verify that the
        # corresponding corpus occurrence is consistent.
        matches = [
            x["parent_id"]
            for x in native_corpus
            if (
                x["language"] == lang
                and x["text"] == passage_text
            )
        ]

        if len(matches) == 0:
            raise ValueError(
                f"No native parent found for {key}"
            )

        # Temporarily retain all matching parents.
        passage_key_to_parent[key] = matches

print("Mapping entries:", len(passage_key_to_parent))

duplicate_keys = [
    k for k, v in passage_key_to_parent.items()
    if len(v) > 1
]

print("Keys with duplicate text matches:", len(duplicate_keys))

print("\nExample duplicate:")
if duplicate_keys:
    k = duplicate_keys[0]
    print(k)
    print("Matching parents:", passage_key_to_parent[k])

Mapping entries: 4563
Keys with duplicate text matches: 42

Example duplicate:
('hindi', 126172, 5)
Matching parents: ['hindi_126172_p05', 'hindi_126172_p07']


In [ ]:
print("Frozen benchmark keys:")
print(frozen_benchmark[0].keys())

print("\nNative corpus keys:")
print(native_corpus[0].keys())

print("\nExample frozen query:")
print(frozen_benchmark[0])

print("\nExample native passage:")
print(native_corpus[0])

Frozen benchmark keys:
dict_keys(['language', 'query_id', 'query', 'passages', 'is_selected', 'split'])

Native corpus keys:
dict_keys(['parent_id', 'language', 'source_query_id', 'query_idx', 'passage_idx', 'text', 'label'])

Example frozen query:
{'language': 'hindi', 'query_id': 1090350, 'query': 'वयस्कों में विस्थापन में सीमावर्ती व्यक्तित्व विकार के लक्षण', 'passages': ['सीमावर्ती विकार के लिए नौ डीएसएम-IV मानदंडों में से तीन इस समूह में आते हैं: 1. प्रभावी (भावनात्मक) अस्थिरता, जिसमें तीव्र, प्रकरणिक भावनात्मक पीड़ा, चिड़चिड़ापन और चिंता/घबराहट के दौरे शामिल हैं। 2. अनुचित, तीव्र और नियंत्रित करने में कठिन क्रोध। 3. खालीपन की पुरानी भावनाएं।', 'सीमावर्ती व्यक्तित्व विकार के उपचार में आमतौर पर एक चिकित्सक के साथ दीर्घकालिक मनोचिकित्सा शामिल होती है जिसे इस प्रकार के व्यक्तित्व विकार के उपचार में अनुभव होता है। विशिष्ट परेशान करने वाले और अक्षम करने वाले लक्षणों को कम करने में मदद के लिए दवाएं भी निर्धारित की जा सकती हैं।', 'सीमावर्ती व्यक्तित्व विकार के लक्षणों में शामिल हैंः रिश्

In [ ]:
# Exact mapping from the native corpus.
# No text matching is used.

passage_key_to_parent = {}

for passage in native_corpus:
    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"]
    )

    parent_id = passage["parent_id"]

    if key in passage_key_to_parent:
        raise ValueError(
            f"Duplicate native key found: {key}"
        )

    passage_key_to_parent[key] = parent_id

print(
    "Native mapping entries:",
    len(passage_key_to_parent)
)

# Verify every frozen benchmark passage resolves uniquely.
missing = []

for q in frozen_benchmark:
    lang = q["language"]
    query_id = q["query_id"]

    for passage_idx in range(len(q["passages"])):
        key = (
            lang,
            query_id,
            passage_idx
        )

        if key not in passage_key_to_parent:
            missing.append(key)

print(
    "Missing frozen benchmark mappings:",
    len(missing)
)

if missing:
    print("First missing:", missing[:5])

assert len(missing) == 0

print(
    "✓ Every frozen benchmark passage maps "
    "to exactly one native parent_id."
)

Native mapping entries: 8973
Missing frozen benchmark mappings: 0
✓ Every frozen benchmark passage maps to exactly one native parent_id.


In [ ]:
from sklearn.metrics import ndcg_score

def evaluate_fixed2(split_name):
    split_queries = [
        (i, q)
        for i, q in enumerate(frozen_benchmark)
        if q["split"] == split_name
    ]

    ndcg_scores = []
    mrr_scores = []
    recall_scores = []

    for q_idx, q in split_queries:
        lang = q["language"]
        query_id = q["query_id"]

        # Query embedding
        query_vec = query_embeddings[q_idx]

        # Fixed-2 candidates from the query's language only
        candidate_idx = fixed2_lang_indices[lang]
        candidate_emb = fixed2_embeddings[candidate_idx]

        # Exact cosine similarity
        scores = candidate_emb @ query_vec

        # Rank chunks
        ranked_local = np.argsort(-scores)
        ranked_global = candidate_idx[ranked_local]

        # -------------------------------------------------
        # Collapse chunks -> parent passages
        # Keep highest-scoring chunk for each parent.
        # -------------------------------------------------
        parent_best_score = {}

        for idx, score in zip(ranked_global, scores[ranked_local]):
            parent_id = fixed2_corpus[idx]["parent_id"]

            if (
                parent_id not in parent_best_score
                or score > parent_best_score[parent_id]
            ):
                parent_best_score[parent_id] = float(score)

        ranked_parents = sorted(
            parent_best_score,
            key=parent_best_score.get,
            reverse=True
        )

        # -------------------------------------------------
        # Exact frozen relevance mapping
        # (language, query_id, passage_idx) -> parent_id
        # -------------------------------------------------
        relevance = {}

        for passage_idx, label in enumerate(q["is_selected"]):
            key = (
                lang,
                query_id,
                passage_idx
            )

            parent_id = passage_key_to_parent[key]
            relevance[parent_id] = int(label)

        # -------------------------------------------------
        # Top-10 parent ranking
        # -------------------------------------------------
        top10 = ranked_parents[:10]

        gains = np.array(
            [
                relevance.get(pid, 0)
                for pid in top10
            ],
            dtype=float
        )

        gains = np.pad(
            gains,
            (0, max(0, 10 - len(gains)))
        )[:10]

        ideal = np.sort(
            np.array(
                list(relevance.values()),
                dtype=float
            )
        )[::-1][:10]

        ideal = np.pad(
            ideal,
            (0, max(0, 10 - len(ideal)))
        )[:10]

        # nDCG@10
        ndcg = ndcg_score(
            ideal.reshape(1, -1),
            gains.reshape(1, -1),
            k=10
        )

        # MRR@10
        rr = 0.0

        for rank, pid in enumerate(top10, start=1):
            if relevance.get(pid, 0) > 0:
                rr = 1.0 / rank
                break

        # Recall@10
        total_relevant = sum(relevance.values())

        retrieved_relevant = sum(
            relevance.get(pid, 0)
            for pid in top10
        )

        recall = (
            retrieved_relevant / total_relevant
            if total_relevant > 0
            else 0.0
        )

        ndcg_scores.append(ndcg)
        mrr_scores.append(rr)
        recall_scores.append(recall)

    return {
        "split": split_name,
        "queries": len(split_queries),
        "nDCG@10": float(np.mean(ndcg_scores)),
        "MRR@10": float(np.mean(mrr_scores)),
        "Recall@10": float(np.mean(recall_scores))
    }


print("Evaluating Fixed-2...\n")

fixed2_dev = evaluate_fixed2("dev")
fixed2_test = evaluate_fixed2("test")

print("DEV")
print(fixed2_dev)

print("\nTEST")
print(fixed2_test)

Evaluating Fixed-2...

DEV
{'split': 'dev', 'queries': 225, 'nDCG@10': 0.5751810541939495, 'MRR@10': 0.44926984126984126, 'Recall@10': 0.8266666666666667}

TEST
{'split': 'test', 'queries': 234, 'nDCG@10': 0.5676296894610642, 'MRR@10': 0.44905541988875325, 'Recall@10': 0.8311965811965812}


In [ ]:
fixed2_result = {
    "method": "fixed_2_sentence",
    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
    "split_protocol": "frozen_seed_42",
    "retrieval_scope": "query_language_only",
    "metrics": {
        "dev": fixed2_dev,
        "test": fixed2_test
    }
}

fixed2_result_path = os.path.join(
    RESULTS,
    "fixed2_results.json"
)

with open(
    fixed2_result_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        fixed2_result,
        f,
        indent=2
    )

print("Saved:", fixed2_result_path)
print("✓ Fixed-2 result checkpointed to Drive.")

Saved: /content/drive/MyDrive/maxmin_research/results/fixed2_results.json
✓ Fixed-2 result checkpointed to Drive.


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 1: Load cached sentence embeddings and verify alignment

import os
import json
import numpy as np

BASE = "/content/drive/MyDrive/maxmin_research"
CACHE = os.path.join(BASE, "cache")
FROZEN = os.path.join(BASE, "frozen")

# Load sentence corpus
with open(os.path.join(FROZEN, "msmarco_xi_sentence_corpus.json"), "r", encoding="utf-8") as f:
    sentence_corpus = json.load(f)

# Load cached Qwen sentence embeddings
sentence_emb_path = os.path.join(
    CACHE,
    "qwen_sentence_embeddings",
    "qwen_sentence_embeddings.npy"
)

sentence_ids_path = os.path.join(
    CACHE,
    "qwen_sentence_embeddings",
    "sentence_ids.json"
)

sentence_embeddings = np.load(sentence_emb_path)

with open(sentence_ids_path, "r", encoding="utf-8") as f:
    cached_sentence_ids = json.load(f)

print("Sentence corpus:", len(sentence_corpus))
print("Cached embeddings:", sentence_embeddings.shape)
print("Cached sentence IDs:", len(cached_sentence_ids))

# Verify count
assert len(sentence_corpus) == len(cached_sentence_ids)
assert sentence_embeddings.shape[0] == len(cached_sentence_ids)

# Verify exact sentence-ID ordering
corpus_sentence_ids = [x["sentence_id"] for x in sentence_corpus]

assert corpus_sentence_ids == cached_sentence_ids, \
    "Sentence ID ordering does not match cached embeddings."

# Verify normalized embeddings
norms = np.linalg.norm(sentence_embeddings, axis=1)

print(f"Norm min:  {norms.min():.8f}")
print(f"Norm mean: {norms.mean():.8f}")
print(f"Norm max:  {norms.max():.8f}")

assert np.allclose(norms, 1.0, atol=1e-5)

print("\n✓ Sentence corpus ↔ embedding alignment verified")
print("✓ Embeddings are L2-normalized")
print("✓ Ready for Max–Min structural segmentation")

Sentence corpus: 31792
Cached embeddings: (31792, 1024)
Cached sentence IDs: 31792
Norm min:  0.99999982
Norm mean: 1.00000000
Norm max:  1.00000012

✓ Sentence corpus ↔ embedding alignment verified
✓ Embeddings are L2-normalized
✓ Ready for Max–Min structural segmentation


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 2: Define the official-code Max–Min chunker

import math
import numpy as np

# Paper / official implementation defaults
HARD_THR = 0.6
INIT_CONST = 1.5
C_PARAM = 0.9

# Official code uses sigmoid(|C| - 1)
SIGMOID_OFFSET = 1

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


def maxmin_chunk_embeddings(
    sentence_embeddings_for_passage,
    hard_thr=HARD_THR,
    init_const=INIT_CONST,
    c=C_PARAM,
    sigmoid_offset=SIGMOID_OFFSET
):
    """
    Official-code Max–Min chunking.

    Parameters
    ----------
    sentence_embeddings_for_passage : np.ndarray
        Shape: (num_sentences, embedding_dim)
        Expected to be L2-normalized.

    hard_thr : float
        Fixed lower threshold.

    init_const : float
        Multiplier used when the current chunk has one sentence.

    c : float
        Scaling constant for the adaptive threshold.

    sigmoid_offset : int
        1 -> official implementation: sigmoid(|C| - 1)
        0 -> paper-text variant: sigmoid(|C|)

    Returns
    -------
    chunks : list[list[int]]
        Sentence indices belonging to each chunk.
    """

    n_sentences = len(sentence_embeddings_for_passage)

    if n_sentences == 0:
        return []

    if n_sentences == 1:
        return [[0]]

    chunks = []
    current_chunk = [0]

    # Running minimum pairwise cosine similarity within current chunk.
    # For |C| == 1, this is not used.
    min_pairwise_sim = None

    for k in range(1, n_sentences):

        candidate = sentence_embeddings_for_passage[k]

        # ---------------------------------------------------------
        # Case 1: current chunk contains exactly one sentence
        # ---------------------------------------------------------
        if len(current_chunk) == 1:

            previous = sentence_embeddings_for_passage[current_chunk[-1]]
            adjacent_sim = float(np.dot(previous, candidate))

            if init_const * adjacent_sim > hard_thr:
                current_chunk.append(k)
            else:
                chunks.append(current_chunk)
                current_chunk = [k]
                min_pairwise_sim = None

        # ---------------------------------------------------------
        # Case 2: current chunk contains >= 2 sentences
        # ---------------------------------------------------------
        else:

            current = sentence_embeddings_for_passage[current_chunk]

            # Maximum similarity between candidate and ANY sentence
            # already in the current chunk.
            candidate_sims = np.dot(current, candidate)
            max_sim = float(np.max(candidate_sims))

            # Pairwise similarities involving the new sentence.
            # Since min_pairwise_sim is already the minimum over
            # existing pairs, update it with the new pairs.
            new_min = float(np.min(candidate_sims))

            if min_pairwise_sim is None:
                # Should not normally occur because |C| >= 2,
                # but keeps the state explicit.
                pairwise = np.dot(current, current.T)
                np.fill_diagonal(pairwise, np.inf)
                min_pairwise_sim = float(np.min(pairwise))

            min_pairwise_sim = min(min_pairwise_sim, new_min)

            # Official implementation:
            # sigmoid(|C| - 1)
            adaptive_thr = (
                c
                * min_pairwise_sim
                * sigmoid(len(current_chunk) - sigmoid_offset)
            )

            threshold = max(adaptive_thr, hard_thr)

            if max_sim > threshold:
                current_chunk.append(k)
            else:
                chunks.append(current_chunk)
                current_chunk = [k]
                min_pairwise_sim = None

    # End-of-document handling:
    # append the final open chunk.
    chunks.append(current_chunk)

    return chunks


print("✓ Max–Min chunker defined")
print(f"hard_thr       = {HARD_THR}")
print(f"init_const     = {INIT_CONST}")
print(f"c              = {C_PARAM}")
print("sigmoid        = sigmoid(|C| - 1)  [official-code version]")
print("max chunk size = none")
print("similarity window = none")
print("final open chunk = appended")

✓ Max–Min chunker defined
hard_thr       = 0.6
init_const     = 1.5
c              = 0.9
sigmoid        = sigmoid(|C| - 1)  [official-code version]
max chunk size = none
similarity window = none
final open chunk = appended


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 3: Run structural segmentation on all native passages

from collections import defaultdict
import os
import json
import numpy as np

# ---------------------------------------------------------
# Build sentence index lookup by parent_id
# ---------------------------------------------------------

sentences_by_parent = defaultdict(list)

for global_idx, record in enumerate(sentence_corpus):
    sentences_by_parent[record["parent_id"]].append({
        "global_idx": global_idx,
        "sentence_idx": record["sentence_idx"],
    })

# Ensure deterministic sentence order
for parent_id in sentences_by_parent:
    sentences_by_parent[parent_id].sort(key=lambda x: x["sentence_idx"])


# ---------------------------------------------------------
# Map parent_id -> native corpus record
# ---------------------------------------------------------

native_by_parent = {
    passage["parent_id"]: passage
    for passage in native_corpus
}

assert len(native_by_parent) == len(native_corpus)


# ---------------------------------------------------------
# Run Max–Min
# ---------------------------------------------------------

maxmin_segmentation = []

total_passages = 0
total_chunks = 0
total_sentences = 0
singleton_chunks = 0
one_chunk_passages = 0

for passage in native_corpus:

    parent_id = passage["parent_id"]

    sentence_records = sentences_by_parent[parent_id]

    # Verify sentence ordering and count
    sentence_indices = [x["sentence_idx"] for x in sentence_records]
    expected_indices = list(range(len(sentence_records)))

    assert sentence_indices == expected_indices, (
        f"Sentence ordering mismatch for {parent_id}"
    )

    global_indices = [x["global_idx"] for x in sentence_records]

    passage_embeddings = sentence_embeddings[global_indices]

    # Run official-code Max–Min
    chunks = maxmin_chunk_embeddings(passage_embeddings)

    # Basic validity checks
    assert sum(len(chunk) for chunk in chunks) == len(sentence_records)

    flattened = [idx for chunk in chunks for idx in chunk]
    assert flattened == list(range(len(sentence_records)))

    # Convert local sentence indices to actual sentence indices
    chunk_records = []

    for chunk_idx, chunk in enumerate(chunks):

        chunk_sentence_indices = [
            sentence_records[i]["sentence_idx"]
            for i in chunk
        ]

        chunk_records.append({
            "chunk_idx": chunk_idx,
            "sentence_indices": chunk_sentence_indices,
            "num_sentences": len(chunk_sentence_indices),
        })

        total_chunks += 1
        total_sentences += len(chunk_sentence_indices)

        if len(chunk_sentence_indices) == 1:
            singleton_chunks += 1

    num_chunks = len(chunk_records)

    if num_chunks == 1:
        one_chunk_passages += 1

    total_passages += 1

    maxmin_segmentation.append({
        "parent_id": parent_id,
        "language": passage["language"],
        "source_query_id": passage["source_query_id"],
        "passage_idx": passage["passage_idx"],
        "num_sentences": len(sentence_records),
        "num_chunks": num_chunks,
        "chunks": chunk_records,
    })


# ---------------------------------------------------------
# Sanity checks
# ---------------------------------------------------------

assert total_passages == len(native_corpus)
assert total_sentences == len(sentence_corpus)

print("✓ Max–Min segmentation complete")
print(f"Passages:              {total_passages:,}")
print(f"Total sentences:       {total_sentences:,}")
print(f"Total chunks:          {total_chunks:,}")
print(f"Chunks / passage:      {total_chunks / total_passages:.4f}")
print(f"Sentences / chunk:     {total_sentences / total_chunks:.4f}")
print(f"Singleton chunks:      {singleton_chunks:,}")
print(
    f"Singleton-chunk share: "
    f"{100 * singleton_chunks / total_chunks:.2f}%"
)
print(
    f"One-chunk passages:    {one_chunk_passages:,}"
)
print(
    f"One-chunk passage share:"
    f" {100 * one_chunk_passages / total_passages:.2f}%"
)


# ---------------------------------------------------------
# Chunk-count distribution
# ---------------------------------------------------------

chunk_counts = np.array([
    x["num_chunks"]
    for x in maxmin_segmentation
])

print("\nChunk-count distribution:")
print(f"  min:    {chunk_counts.min()}")
print(f"  median: {np.median(chunk_counts):.1f}")
print(f"  mean:   {chunk_counts.mean():.4f}")
print(f"  p90:    {np.percentile(chunk_counts, 90):.1f}")
print(f"  p95:    {np.percentile(chunk_counts, 95):.1f}")
print(f"  max:    {chunk_counts.max()}")


# ---------------------------------------------------------
# Save segmentation
# ---------------------------------------------------------

segmentation_path = os.path.join(
    BASE,
    "frozen",
    "maxmin_segmentation_qwen_official.json"
)

with open(segmentation_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "method": "max_min",
            "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
            "hard_thr": HARD_THR,
            "init_const": INIT_CONST,
            "c": C_PARAM,
            "sigmoid_variant": "sigmoid(|C| - 1)",
            "implementation_reference": "official_author_code",
            "max_chunk_size": None,
            "similarity_window": None,
            "num_passages": total_passages,
            "num_sentences": total_sentences,
            "num_chunks": total_chunks,
            "segmentation": maxmin_segmentation,
        },
        f,
        ensure_ascii=False,
    )

print(f"\n✓ Saved segmentation to:")
print(segmentation_path)

✓ Max–Min segmentation complete
Passages:              8,973
Total sentences:       31,792
Total chunks:          22,341
Chunks / passage:      2.4898
Sentences / chunk:     1.4230
Singleton chunks:      14,737
Singleton-chunk share: 65.96%
One-chunk passages:    2,865
One-chunk passage share: 31.93%

Chunk-count distribution:
  min:    1
  median: 2.0
  mean:   2.4898
  p90:    4.0
  p95:    6.0
  max:    135

✓ Saved segmentation to:
/content/drive/MyDrive/maxmin_research/frozen/maxmin_segmentation_qwen_official.json


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 4: Detailed structural diagnostics

from collections import defaultdict
import numpy as np

# ---------------------------------------------------------
# Overall chunk-size distribution
# ---------------------------------------------------------

sentences_per_chunk = np.array([
    chunk["num_sentences"]
    for passage in maxmin_segmentation
    for chunk in passage["chunks"]
])

print("Overall sentences/chunk distribution")
print("-------------------------------------")
print(f"Min:     {sentences_per_chunk.min()}")
print(f"Mean:    {sentences_per_chunk.mean():.4f}")
print(f"Median:  {np.median(sentences_per_chunk):.1f}")
print(f"P75:     {np.percentile(sentences_per_chunk, 75):.1f}")
print(f"P90:     {np.percentile(sentences_per_chunk, 90):.1f}")
print(f"P95:     {np.percentile(sentences_per_chunk, 95):.1f}")
print(f"P99:     {np.percentile(sentences_per_chunk, 99):.1f}")
print(f"Max:     {sentences_per_chunk.max()}")

print("\nChunk size counts")
print("-----------------")

for size in sorted(set(sentences_per_chunk)):
    count = int(np.sum(sentences_per_chunk == size))
    share = 100 * count / len(sentences_per_chunk)
    print(f"{size:>3} sentence(s): {count:>6,}  ({share:>6.2f}%)")


# ---------------------------------------------------------
# Language-level diagnostics
# ---------------------------------------------------------

language_stats = defaultdict(lambda: {
    "passages": 0,
    "chunks": 0,
    "sentences": 0,
    "singleton_chunks": 0,
    "one_chunk_passages": 0,
})

for passage in maxmin_segmentation:

    lang = passage["language"]

    stats = language_stats[lang]

    stats["passages"] += 1
    stats["chunks"] += passage["num_chunks"]
    stats["sentences"] += passage["num_sentences"]

    if passage["num_chunks"] == 1:
        stats["one_chunk_passages"] += 1

    for chunk in passage["chunks"]:
        if chunk["num_sentences"] == 1:
            stats["singleton_chunks"] += 1


print("\n\nLanguage-level structural statistics")
print("--------------------------------------")

languages_sorted = sorted(language_stats)

print(
    f"{'Language':<12}"
    f"{'Passages':>10}"
    f"{'Chunks':>10}"
    f"{'Ch/Psg':>10}"
    f"{'Sent/Ch':>10}"
    f"{'Singleton%':>12}"
    f"{'1-Ch%':>10}"
)

for lang in languages_sorted:

    s = language_stats[lang]

    chunks_per_passage = s["chunks"] / s["passages"]
    sentences_per_chunk = s["sentences"] / s["chunks"]
    singleton_share = (
        100 * s["singleton_chunks"] / s["chunks"]
    )
    one_chunk_share = (
        100 * s["one_chunk_passages"] / s["passages"]
    )

    print(
        f"{lang:<12}"
        f"{s['passages']:>10,}"
        f"{s['chunks']:>10,}"
        f"{chunks_per_passage:>10.3f}"
        f"{sentences_per_chunk:>10.3f}"
        f"{singleton_share:>11.2f}%"
        f"{one_chunk_share:>9.2f}%"
    )


print("\n✓ Structural diagnostics complete")
print("✓ No embeddings or retrieval evaluation performed")

Overall sentences/chunk distribution
-------------------------------------
Min:     1
Mean:    1.4230
Median:  1.0
P75:     2.0
P90:     2.0
P95:     3.0
P99:     4.0
Max:     68

Chunk size counts
-----------------
  1 sentence(s): 14,737  ( 65.96%)
  2 sentence(s):  6,336  ( 28.36%)
  3 sentence(s):    933  (  4.18%)
  4 sentence(s):    250  (  1.12%)
  5 sentence(s):     60  (  0.27%)
  6 sentence(s):     17  (  0.08%)
  7 sentence(s):      1  (  0.00%)
  9 sentence(s):      1  (  0.00%)
 11 sentence(s):      2  (  0.01%)
 22 sentence(s):      1  (  0.00%)
 26 sentence(s):      1  (  0.00%)
 28 sentence(s):      1  (  0.00%)
 68 sentence(s):      1  (  0.00%)


Language-level structural statistics
--------------------------------------
Language      Passages    Chunks    Ch/Psg   Sent/Ch  Singleton%     1-Ch%
bengali            997     2,271     2.278     1.426      64.29%    34.90%
gujarati           997     2,480     2.487     1.425      65.28%    29.29%
hindi              997    

In [ ]:
# METHOD 3 — MAX–MIN
# Cell 5: Build chunk corpus and verify embedding reuse

import os
import json
import hashlib
import numpy as np

# ---------------------------------------------------------
# Fast lookup: sentence text + embedding index
# ---------------------------------------------------------

sentence_lookup = {}

for idx, record in enumerate(sentence_corpus):
    sentence_lookup[record["sentence_id"]] = {
        "global_idx": idx,
        "text": record["text"],
        "parent_id": record["parent_id"],
        "sentence_idx": record["sentence_idx"],
    }

assert len(sentence_lookup) == len(sentence_corpus)


# ---------------------------------------------------------
# Build exact parent lookup
# ---------------------------------------------------------

native_key_to_parent = {}

for passage in native_corpus:
    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"],
    )

    if key in native_key_to_parent:
        raise ValueError(f"Duplicate native key: {key}")

    native_key_to_parent[key] = passage["parent_id"]


# ---------------------------------------------------------
# Construct Max–Min chunks
# ---------------------------------------------------------

maxmin_chunks = []

singleton_count = 0
multisentence_count = 0
one_chunk_passage_count = 0

for passage in maxmin_segmentation:

    parent_id = passage["parent_id"]

    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"],
    )

    # Exact parent mapping — NEVER text matching
    assert native_key_to_parent[key] == parent_id

    parent_sentence_records = sentences_by_parent[parent_id]

    for chunk in passage["chunks"]:

        sentence_indices = chunk["sentence_indices"]

        # Recover exact sentence records in order
        selected = [
            parent_sentence_records[i]
            for i in sentence_indices
        ]

        sentence_ids = [
            sentence_corpus[x["global_idx"]]["sentence_id"]
            for x in selected
        ]

        sentence_texts = [
            sentence_corpus[x["global_idx"]]["text"]
            for x in selected
        ]

        chunk_text = " ".join(sentence_texts)

        # Deterministic ID based on parent + chunk index
        chunk_id = (
            f"{parent_id}_maxmin_{chunk['chunk_idx']}"
        )

        if len(sentence_indices) == 1:
            singleton_count += 1
            embedding_source = "sentence_cache"
        else:
            multisentence_count += 1
            embedding_source = "new_chunk_embedding"

        maxmin_chunks.append({
            "chunk_id": chunk_id,
            "parent_id": parent_id,
            "language": passage["language"],
            "source_query_id": passage["source_query_id"],
            "passage_idx": passage["passage_idx"],
            "chunk_idx": chunk["chunk_idx"],
            "sentence_indices": sentence_indices,
            "sentence_ids": sentence_ids,
            "text": chunk_text,
            "num_sentences": len(sentence_indices),
            "embedding_source": embedding_source,
        })

    if passage["num_chunks"] == 1:
        one_chunk_passage_count += 1


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(maxmin_chunks) == sum(
    x["num_chunks"] for x in maxmin_segmentation
)

assert singleton_count + multisentence_count == len(maxmin_chunks)

chunk_ids = [x["chunk_id"] for x in maxmin_chunks]
assert len(chunk_ids) == len(set(chunk_ids))

# Verify every chunk has a valid exact parent mapping
for chunk in maxmin_chunks:

    key = (
        chunk["language"],
        chunk["source_query_id"],
        chunk["passage_idx"],
    )

    assert native_key_to_parent[key] == chunk["parent_id"]

print("✓ Max–Min chunk corpus constructed")
print(f"Total chunks:              {len(maxmin_chunks):,}")
print(f"Singleton chunks:          {singleton_count:,}")
print(f"Multi-sentence chunks:     {multisentence_count:,}")
print(f"One-chunk passages:        {one_chunk_passage_count:,}")
print(
    f"New chunk embeddings needed:"
    f" {multisentence_count:,}"
)

print("\nEmbedding reuse:")
print(
    f"  Singleton chunks → cached sentence embeddings: "
    f"{singleton_count:,}"
)
print(
    f"  One-chunk passages → native embedding available: "
    f"{one_chunk_passage_count:,}"
)


# ---------------------------------------------------------
# Save deterministic chunk corpus
# ---------------------------------------------------------

chunk_corpus_path = os.path.join(
    FROZEN,
    "maxmin_chunk_corpus_qwen_official.json"
)

with open(chunk_corpus_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "method": "max_min",
            "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
            "hard_thr": HARD_THR,
            "init_const": INIT_CONST,
            "c": C_PARAM,
            "sigmoid_variant": "sigmoid(|C| - 1)",
            "implementation_reference": "official_author_code",
            "num_chunks": len(maxmin_chunks),
            "num_singleton_chunks": singleton_count,
            "num_multisentence_chunks": multisentence_count,
            "num_one_chunk_passages": one_chunk_passage_count,
            "chunks": maxmin_chunks,
        },
        f,
        ensure_ascii=False,
    )

print("\n✓ Saved chunk corpus:")
print(chunk_corpus_path)

✓ Max–Min chunk corpus constructed
Total chunks:              22,341
Singleton chunks:          14,737
Multi-sentence chunks:     7,604
One-chunk passages:        2,865
New chunk embeddings needed: 7,604

Embedding reuse:
  Singleton chunks → cached sentence embeddings: 14,737
  One-chunk passages → native embedding available: 2,865

✓ Saved chunk corpus:
/content/drive/MyDrive/maxmin_research/frozen/maxmin_chunk_corpus_qwen_official.json


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 6: Corrected verification of new embedding workload

new_chunks = [
    chunk
    for chunk in maxmin_chunks
    if chunk["embedding_source"] == "new_chunk_embedding"
]

assert len(new_chunks) == 7604

# Every new chunk must be multi-sentence
assert all(chunk["num_sentences"] >= 2 for chunk in new_chunks)

# Every singleton must use the cached sentence embedding
assert all(
    chunk["embedding_source"] == "sentence_cache"
    for chunk in maxmin_chunks
    if chunk["num_sentences"] == 1
)

# No empty chunk text
assert all(chunk["text"].strip() for chunk in new_chunks)

# Chunk IDs must be unique even when chunk text is duplicated
new_chunk_ids = [chunk["chunk_id"] for chunk in new_chunks]
assert len(new_chunk_ids) == len(set(new_chunk_ids))

# Exact parent mapping must remain valid
for chunk in new_chunks:
    key = (
        chunk["language"],
        chunk["source_query_id"],
        chunk["passage_idx"],
    )
    assert native_key_to_parent[key] == chunk["parent_id"]

# Duplicate TEXT is allowed.
# We report it only as a diagnostic.
new_texts = [chunk["text"] for chunk in new_chunks]
unique_new_texts = set(new_texts)
duplicate_count = len(new_texts) - len(unique_new_texts)

# Sentence-count diagnostics
new_sentence_counts = np.array([
    chunk["num_sentences"]
    for chunk in new_chunks
])

print("✓ New embedding workload verified")
print(f"New chunk instances:       {len(new_chunks):,}")
print(f"Unique chunk IDs:           {len(set(new_chunk_ids)):,}")
print(f"Unique chunk texts:         {len(unique_new_texts):,}")
print(f"Repeated text instances:    {duplicate_count:,}")
print(
    f"Sentence count range:       "
    f"{new_sentence_counts.min()}–{new_sentence_counts.max()}"
)
print(
    f"Mean sentences/chunk:       "
    f"{new_sentence_counts.mean():.4f}"
)

print("\n✓ Duplicate text is allowed and preserved")
print("✓ Every chunk retains its unique parent/chunk identity")
print("✓ Exact parent mapping verified")
print("✓ Ready for Qwen chunk embedding")

✓ New embedding workload verified
New chunk instances:       7,604
Unique chunk IDs:           7,604
Unique chunk texts:         7,513
Repeated text instances:    91
Sentence count range:       2–68
Mean sentences/chunk:       2.2429

✓ Duplicate text is allowed and preserved
✓ Every chunk retains its unique parent/chunk identity
✓ Exact parent mapping verified
✓ Ready for Qwen chunk embedding


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 7: Embed the 7,604 new multi-sentence chunks

import os
import json
import time
import numpy as np
import torch

EMB_DIR = os.path.join(
    CACHE,
    "qwen_maxmin_embeddings"
)
os.makedirs(EMB_DIR, exist_ok=True)

EMB_PATH = os.path.join(
    EMB_DIR,
    "qwen_maxmin_multisentence_embeddings.npy"
)

IDS_PATH = os.path.join(
    EMB_DIR,
    "maxmin_multisentence_chunk_ids.json"
)

CHECKPOINT_PATH = os.path.join(
    EMB_DIR,
    "maxmin_embedding_checkpoint.json"
)

# ---------------------------------------------------------
# Prepare deterministic workload
# ---------------------------------------------------------

new_chunks = [
    chunk
    for chunk in maxmin_chunks
    if chunk["embedding_source"] == "new_chunk_embedding"
]

new_chunk_ids = [chunk["chunk_id"] for chunk in new_chunks]
new_chunk_texts = [chunk["text"] for chunk in new_chunks]

assert len(new_chunks) == 7604
assert len(new_chunk_ids) == len(set(new_chunk_ids))

# ---------------------------------------------------------
# Resume support
# ---------------------------------------------------------

start_idx = 0
embeddings_list = []

if os.path.exists(EMB_PATH) and os.path.exists(IDS_PATH):

    existing_embeddings = np.load(EMB_PATH)

    with open(IDS_PATH, "r", encoding="utf-8") as f:
        existing_ids = json.load(f)

    if (
        existing_embeddings.shape[0] == len(existing_ids)
        and existing_ids == new_chunk_ids[:len(existing_ids)]
    ):
        start_idx = len(existing_ids)
        embeddings_list = [
            existing_embeddings[i]
            for i in range(existing_embeddings.shape[0])
        ]

        print(f"✓ Resuming from {start_idx:,}/{len(new_chunks):,}")

    else:
        print("⚠ Existing checkpoint does not match current workload.")
        print("Starting fresh.")

# ---------------------------------------------------------
# Token-length-aware batching
# ---------------------------------------------------------

def choose_batch_size(token_count):
    if token_count <= 256:
        return 16
    elif token_count <= 512:
        return 8
    elif token_count <= 1024:
        return 2
    else:
        return 1


# ---------------------------------------------------------
# Embedding loop
# ---------------------------------------------------------

remaining = len(new_chunks) - start_idx
start_time = time.time()

i = start_idx

while i < len(new_chunks):

    # Estimate token length using tokenizer
    token_lengths = []

    for j in range(i, min(i + 16, len(new_chunks))):
        tokenized = model.tokenizer(
            new_chunk_texts[j],
            add_special_tokens=True,
            truncation=False,
        )
        token_lengths.append(len(tokenized["input_ids"]))

    current_tokens = max(token_lengths)

    batch_size = choose_batch_size(current_tokens)

    batch_end = min(
        i + batch_size,
        len(new_chunks)
    )

    batch_texts = new_chunk_texts[i:batch_end]

    embeddings = model.encode(
        batch_texts,
        batch_size=batch_size,
        normalize_embeddings=False,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    # Explicit L2 normalization
    norms = np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True
    )

    embeddings = embeddings / norms

    embeddings_list.extend(embeddings)

    i = batch_end

    # -----------------------------------------------------
    # Checkpoint every 500 chunks
    # -----------------------------------------------------

    if i % 500 < batch_size or i == len(new_chunks):

        arr = np.asarray(
            embeddings_list,
            dtype=np.float32
        )

        np.save(EMB_PATH, arr)

        with open(IDS_PATH, "w", encoding="utf-8") as f:
            json.dump(new_chunk_ids[:i], f)

        elapsed = time.time() - start_time

        print(
            f"{i:,}/{len(new_chunks):,} "
            f"({100*i/len(new_chunks):.1f}%) | "
            f"elapsed {elapsed/60:.1f} min"
        )


# ---------------------------------------------------------
# Final verification
# ---------------------------------------------------------

final_embeddings = np.asarray(
    embeddings_list,
    dtype=np.float32
)

assert final_embeddings.shape == (
    len(new_chunks),
    1024
)

final_norms = np.linalg.norm(
    final_embeddings,
    axis=1
)

assert np.allclose(
    final_norms,
    1.0,
    atol=1e-5
)

np.save(EMB_PATH, final_embeddings)

with open(IDS_PATH, "w", encoding="utf-8") as f:
    json.dump(new_chunk_ids, f)

with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "method": "max_min",
            "model": "Qwen/Qwen3-Embedding-0.6B",
            "num_embeddings": len(new_chunks),
            "dimension": 1024,
            "completed": True,
            "normalized": True,
        },
        f,
        indent=2,
    )

print("\n✓ Max–Min multi-sentence embeddings complete")
print(f"Shape: {final_embeddings.shape}")
print(
    f"Norm range: "
    f"{final_norms.min():.8f} → {final_norms.max():.8f}"
)
print(f"Saved: {EMB_PATH}")

502/7,604 (6.6%) | elapsed 1.1 min
1,002/7,604 (13.2%) | elapsed 2.2 min
1,500/7,604 (19.7%) | elapsed 3.5 min
2,000/7,604 (26.3%) | elapsed 5.0 min
2,507/7,604 (33.0%) | elapsed 6.7 min
3,001/7,604 (39.5%) | elapsed 8.4 min
3,503/7,604 (46.1%) | elapsed 10.1 min
4,001/7,604 (52.6%) | elapsed 11.9 min
4,503/7,604 (59.2%) | elapsed 13.5 min
5,006/7,604 (65.8%) | elapsed 15.3 min
5,504/7,604 (72.4%) | elapsed 16.6 min
6,000/7,604 (78.9%) | elapsed 17.9 min
6,506/7,604 (85.6%) | elapsed 19.5 min
7,000/7,604 (92.1%) | elapsed 21.2 min
7,500/7,604 (98.6%) | elapsed 22.8 min
7,604/7,604 (100.0%) | elapsed 23.2 min

✓ Max–Min multi-sentence embeddings complete
Shape: (7604, 1024)
Norm range: 0.99999982 → 1.00000012
Saved: /content/drive/MyDrive/maxmin_research/cache/qwen_maxmin_embeddings/qwen_maxmin_multisentence_embeddings.npy


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 8: Assemble and verify complete Max–Min embedding matrix

import os
import json
import numpy as np

MAXMIN_EMB_DIR = os.path.join(
    CACHE,
    "qwen_maxmin_embeddings"
)

MULTI_EMB_PATH = os.path.join(
    MAXMIN_EMB_DIR,
    "qwen_maxmin_multisentence_embeddings.npy"
)

MULTI_IDS_PATH = os.path.join(
    MAXMIN_EMB_DIR,
    "maxmin_multisentence_chunk_ids.json"
)

# ---------------------------------------------------------
# Load newly embedded multi-sentence chunks
# ---------------------------------------------------------

multi_embeddings = np.load(MULTI_EMB_PATH)

with open(MULTI_IDS_PATH, "r", encoding="utf-8") as f:
    multi_ids = json.load(f)

assert multi_embeddings.shape == (7604, 1024)
assert len(multi_ids) == 7604
assert len(set(multi_ids)) == 7604

multi_embedding_lookup = {
    chunk_id: multi_embeddings[i]
    for i, chunk_id in enumerate(multi_ids)
}

# ---------------------------------------------------------
# Cached embedding lookups
# ---------------------------------------------------------

sentence_embedding_lookup = {
    sentence_corpus[i]["sentence_id"]: sentence_embeddings[i]
    for i in range(len(sentence_corpus))
}

native_embedding_lookup = {
    native_corpus[i]["parent_id"]: native_embeddings[i]
    for i in range(len(native_corpus))
}

segmentation_lookup = {
    item["parent_id"]: item
    for item in maxmin_segmentation
}

assert len(segmentation_lookup) == len(native_corpus)

# ---------------------------------------------------------
# Assemble in exact Max–Min chunk order
# ---------------------------------------------------------

assembled_embeddings = []
assembled_ids = []

source_counts = {
    "native_embedding": 0,
    "sentence_embedding": 0,
    "new_multisentence_embedding": 0,
}

for chunk in maxmin_chunks:

    chunk_id = chunk["chunk_id"]
    parent_id = chunk["parent_id"]

    parent_seg = segmentation_lookup[parent_id]

    # -----------------------------------------------------
    # One-chunk passage:
    # the entire passage is unchanged, so reuse Native.
    # This takes priority even if it has one sentence.
    # -----------------------------------------------------

    if parent_seg["num_chunks"] == 1:

        emb = native_embedding_lookup[parent_id]

        source_counts["native_embedding"] += 1

    # -----------------------------------------------------
    # Multi-chunk passage + singleton chunk:
    # reuse cached sentence embedding.
    # -----------------------------------------------------

    elif chunk["num_sentences"] == 1:

        sentence_id = chunk["sentence_ids"][0]

        emb = sentence_embedding_lookup[sentence_id]

        source_counts["sentence_embedding"] += 1

    # -----------------------------------------------------
    # Multi-chunk passage + multi-sentence chunk:
    # use newly generated embedding.
    # -----------------------------------------------------

    else:

        assert chunk_id in multi_embedding_lookup

        emb = multi_embedding_lookup[chunk_id]

        source_counts["new_multisentence_embedding"] += 1

    assembled_embeddings.append(emb)
    assembled_ids.append(chunk_id)


assembled_embeddings = np.asarray(
    assembled_embeddings,
    dtype=np.float32
)

# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert assembled_embeddings.shape == (
    len(maxmin_chunks),
    1024
)

expected_ids = [
    chunk["chunk_id"]
    for chunk in maxmin_chunks
]

assert assembled_ids == expected_ids
assert len(set(assembled_ids)) == len(assembled_ids)

assert sum(source_counts.values()) == len(maxmin_chunks)

norms = np.linalg.norm(
    assembled_embeddings,
    axis=1
)

assert np.allclose(
    norms,
    1.0,
    atol=1e-5
)

print("✓ Complete Max–Min embedding matrix assembled")
print(f"Shape: {assembled_embeddings.shape}")

print("\nEmbedding sources:")
print(
    f"  Native embeddings:       "
    f"{source_counts['native_embedding']:,}"
)
print(
    f"  Singleton embeddings:    "
    f"{source_counts['sentence_embedding']:,}"
)
print(
    f"  New multi-sentence:      "
    f"{source_counts['new_multisentence_embedding']:,}"
)

print(
    f"\nNorm range: "
    f"{norms.min():.8f} → {norms.max():.8f}"
)

# ---------------------------------------------------------
# Save complete matrix
# ---------------------------------------------------------

COMPLETE_EMB_PATH = os.path.join(
    MAXMIN_EMB_DIR,
    "qwen_maxmin_complete_embeddings.npy"
)

COMPLETE_IDS_PATH = os.path.join(
    MAXMIN_EMB_DIR,
    "maxmin_complete_chunk_ids.json"
)

np.save(
    COMPLETE_EMB_PATH,
    assembled_embeddings
)

with open(COMPLETE_IDS_PATH, "w", encoding="utf-8") as f:
    json.dump(assembled_ids, f)

print("\n✓ Saved complete embedding matrix:")
print(COMPLETE_EMB_PATH)
print(COMPLETE_IDS_PATH)

✓ Complete Max–Min embedding matrix assembled
Shape: (22341, 1024)

Embedding sources:
  Native embeddings:       2,865
  Singleton embeddings:    13,755
  New multi-sentence:      5,721

Norm range: 0.99999982 → 1.00000012

✓ Saved complete embedding matrix:
/content/drive/MyDrive/maxmin_research/cache/qwen_maxmin_embeddings/qwen_maxmin_complete_embeddings.npy
/content/drive/MyDrive/maxmin_research/cache/qwen_maxmin_embeddings/maxmin_complete_chunk_ids.json


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 8A: Reload cached Native embeddings into memory

NATIVE_EMB_DIR = os.path.join(
    CACHE,
    "qwen_native_embeddings"
)

NATIVE_EMB_PATH = os.path.join(
    NATIVE_EMB_DIR,
    "qwen_native_embeddings.npy"
)

NATIVE_IDS_PATH = os.path.join(
    NATIVE_EMB_DIR,
    "parent_ids.json"
)

native_embeddings = np.load(NATIVE_EMB_PATH)

with open(NATIVE_IDS_PATH, "r", encoding="utf-8") as f:
    native_embedding_ids = json.load(f)

print("Native embeddings:", native_embeddings.shape)
print("Native IDs:", len(native_embedding_ids))

assert native_embeddings.shape == (len(native_corpus), 1024)

expected_native_ids = [
    passage["parent_id"]
    for passage in native_corpus
]

assert native_embedding_ids == expected_native_ids

native_norms = np.linalg.norm(
    native_embeddings,
    axis=1
)

assert np.allclose(native_norms, 1.0, atol=1e-5)

print(
    f"Norm range: "
    f"{native_norms.min():.8f} → {native_norms.max():.8f}"
)

print("✓ Cached Native embeddings reloaded")
print("✓ Native IDs aligned with native corpus")
print("✓ No Native recomputation performed")

Native embeddings: (8973, 1024)
Native IDs: 8973
Norm range: 0.99999988 → 1.00000012
✓ Cached Native embeddings reloaded
✓ Native IDs aligned with native corpus
✓ No Native recomputation performed


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 9: Build exact retrieval indexes and verify relevance mapping

from collections import defaultdict
import numpy as np

# ---------------------------------------------------------
# Verify query embeddings
# ---------------------------------------------------------

assert query_embeddings.shape == (459, 1024)

query_norms = np.linalg.norm(
    query_embeddings,
    axis=1
)

assert np.allclose(
    query_norms,
    1.0,
    atol=1e-5
)

# ---------------------------------------------------------
# Build exact Max–Min chunk index by language
# ---------------------------------------------------------

maxmin_language_indices = defaultdict(list)

for idx, chunk in enumerate(maxmin_chunks):
    maxmin_language_indices[
        chunk["language"]
    ].append(idx)

print("Max–Min language indexes:")

for lang in sorted(maxmin_language_indices):
    print(
        f"  {lang:<12} "
        f"{len(maxmin_language_indices[lang]):,} chunks"
    )

assert sum(
    len(v) for v in maxmin_language_indices.values()
) == len(maxmin_chunks)

# ---------------------------------------------------------
# Build exact frozen relevance mapping
#
# Key:
# (language, source_query_id, passage_idx) -> parent_id
# ---------------------------------------------------------

passage_key_to_parent = {}

for passage in native_corpus:

    key = (
        passage["language"],
        passage["source_query_id"],
        passage["passage_idx"],
    )

    assert key not in passage_key_to_parent

    passage_key_to_parent[key] = passage["parent_id"]

# ---------------------------------------------------------
# Verify every frozen benchmark passage maps exactly
# ---------------------------------------------------------

mapped_passages = 0

for item in frozen_benchmark:

    language = item["language"]
    query_id = item["query_id"]

    for passage_idx in range(len(item["passages"])):

        key = (
            language,
            query_id,
            passage_idx,
        )

        assert key in passage_key_to_parent

        mapped_passages += 1

print(
    f"\n✓ Frozen benchmark passage mappings verified: "
    f"{mapped_passages:,}"
)

# ---------------------------------------------------------
# Query lookup
# ---------------------------------------------------------

query_lookup = {
    (item["language"], item["query_id"]): idx
    for idx, item in enumerate(frozen_benchmark)
}

assert len(query_lookup) == len(frozen_benchmark)

print(f"✓ Frozen queries: {len(query_lookup):,}")
print("✓ Exact parent mapping verified")
print("✓ Query-language-only indexes ready")
print("✓ No text-based relevance matching")
print("✓ No ANN / FAISS")

Max–Min language indexes:
  bengali      2,271 chunks
  gujarati     2,480 chunks
  hindi        2,580 chunks
  kannada      2,665 chunks
  malayalam    2,513 chunks
  marathi      2,780 chunks
  punjabi      2,392 chunks
  tamil        2,349 chunks
  telugu       2,311 chunks

✓ Frozen benchmark passage mappings verified: 4,563
✓ Frozen queries: 459
✓ Exact parent mapping verified
✓ Query-language-only indexes ready
✓ No text-based relevance matching
✓ No ANN / FAISS


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 10: Frozen DEV/TEST retrieval evaluation

import os
import json
import math
import numpy as np


# ---------------------------------------------------------
# Metric helpers
# ---------------------------------------------------------

def dcg_at_k(relevances, k=10):
    relevances = relevances[:k]

    return sum(
        (2 ** rel - 1) / math.log2(i + 2)
        for i, rel in enumerate(relevances)
    )


def ndcg_at_k(relevances, k=10):
    actual_dcg = dcg_at_k(relevances, k)

    ideal = sorted(
        relevances,
        reverse=True
    )

    ideal_dcg = dcg_at_k(ideal, k)

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg


def mrr_at_k(relevances, k=10):
    for i, rel in enumerate(relevances[:k]):
        if rel > 0:
            return 1.0 / (i + 1)

    return 0.0


def recall_at_k(retrieved_parent_ids, relevant_parent_ids, k=10):
    retrieved = set(retrieved_parent_ids[:k])
    relevant = set(relevant_parent_ids)

    if not relevant:
        return 0.0

    return len(retrieved & relevant) / len(relevant)


# ---------------------------------------------------------
# Evaluate one split
# ---------------------------------------------------------

def evaluate_split(split_name):

    split_items = [
        item
        for item in frozen_benchmark
        if item["split"] == split_name
    ]

    per_query = []

    for query_idx, item in enumerate(split_items):

        language = item["language"]
        query_id = item["query_id"]

        # -------------------------------------------------
        # Exact relevance mapping
        # -------------------------------------------------

        relevant_parent_ids = set()

        for passage_idx, label in enumerate(item["is_selected"]):

            if label == 1:

                key = (
                    language,
                    query_id,
                    passage_idx,
                )

                parent_id = passage_key_to_parent[key]

                relevant_parent_ids.add(parent_id)

        assert len(relevant_parent_ids) > 0

        # -------------------------------------------------
        # Query embedding
        # -------------------------------------------------

        global_query_idx = query_lookup[
            (language, query_id)
        ]

        query_vector = query_embeddings[
            global_query_idx
        ]

        # -------------------------------------------------
        # Own-language candidate pool
        # -------------------------------------------------

        candidate_indices = np.asarray(
            maxmin_language_indices[language],
            dtype=np.int32
        )

        candidate_embeddings = assembled_embeddings[
            candidate_indices
        ]

        # Exact cosine because all vectors are normalized
        scores = candidate_embeddings @ query_vector

        # Sort descending
        order = np.argsort(
            -scores,
            kind="stable"
        )

        ranked_chunk_indices = candidate_indices[order]
        ranked_scores = scores[order]

        # -------------------------------------------------
        # Collapse chunks -> parents by MAX chunk score
        # -------------------------------------------------

        parent_best_score = {}

        for chunk_idx, score in zip(
            ranked_chunk_indices,
            ranked_scores
        ):

            parent_id = maxmin_chunks[
                int(chunk_idx)
            ]["parent_id"]

            if (
                parent_id not in parent_best_score
                or score > parent_best_score[parent_id]
            ):
                parent_best_score[parent_id] = float(score)

        ranked_parents = sorted(
            parent_best_score.items(),
            key=lambda x: x[1],
            reverse=True
        )

        ranked_parent_ids = [
            parent_id
            for parent_id, _ in ranked_parents
        ]

        ranked_parent_scores = [
            score
            for _, score in ranked_parents
        ]

        # -------------------------------------------------
        # Relevance vector
        # -------------------------------------------------

        relevance = [
            1 if parent_id in relevant_parent_ids else 0
            for parent_id in ranked_parent_ids[:10]
        ]

        # Pad to 10 if necessary
        relevance += [0] * (10 - len(relevance))

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        ndcg10 = ndcg_at_k(relevance, 10)
        mrr10 = mrr_at_k(relevance, 10)
        recall1 = recall_at_k(
            ranked_parent_ids,
            relevant_parent_ids,
            1
        )
        recall10 = recall_at_k(
            ranked_parent_ids,
            relevant_parent_ids,
            10
        )

        per_query.append({
            "language": language,
            "query_id": query_id,
            "split": split_name,
            "ndcg10": ndcg10,
            "mrr10": mrr10,
            "recall1": recall1,
            "recall10": recall10,
            "num_relevant_parents": len(
                relevant_parent_ids
            ),
            "top10_parent_ids": ranked_parent_ids[:10],
            "top10_scores": ranked_parent_scores[:10],
        })

    # -----------------------------------------------------
    # Aggregate
    # -----------------------------------------------------

    metrics = {
        "n_queries": len(per_query),
        "ndcg10": float(np.mean([
            x["ndcg10"]
            for x in per_query
        ])),
        "mrr10": float(np.mean([
            x["mrr10"]
            for x in per_query
        ])),
        "recall1": float(np.mean([
            x["recall1"]
            for x in per_query
        ])),
        "recall10": float(np.mean([
            x["recall10"]
            for x in per_query
        ])),
    }

    return metrics, per_query


# ---------------------------------------------------------
# Run DEV and TEST
# ---------------------------------------------------------

maxmin_dev, maxmin_dev_queries = evaluate_split("dev")
maxmin_test, maxmin_test_queries = evaluate_split("test")


# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------

print("MAX–MIN RESULTS")
print("=" * 50)

print("\nDEV")
print(f"Queries:    {maxmin_dev['n_queries']}")
print(f"nDCG@10:    {maxmin_dev['ndcg10']:.6f}")
print(f"MRR@10:     {maxmin_dev['mrr10']:.6f}")
print(f"Recall@1:   {maxmin_dev['recall1']:.6f}")
print(f"Recall@10:  {maxmin_dev['recall10']:.6f}")

print("\nTEST")
print(f"Queries:    {maxmin_test['n_queries']}")
print(f"nDCG@10:    {maxmin_test['ndcg10']:.6f}")
print(f"MRR@10:     {maxmin_test['mrr10']:.6f}")
print(f"Recall@1:   {maxmin_test['recall1']:.6f}")
print(f"Recall@10:  {maxmin_test['recall10']:.6f}")


# ---------------------------------------------------------
# Save raw results
# ---------------------------------------------------------

result = {
    "method": "max_min",
    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
    "implementation_reference": "official_author_code",
    "sigmoid_variant": "sigmoid(|C| - 1)",
    "hard_thr": HARD_THR,
    "init_const": INIT_CONST,
    "c": C_PARAM,
    "split_protocol": "frozen_seed_42",
    "retrieval_scope": "query_language_only",
    "retrieval": "exact_cosine_dot_product",
    "parent_collapse": "max_chunk_score",
    "metrics": {
        "dev": maxmin_dev,
        "test": maxmin_test,
    },
    "per_query": {
        "dev": maxmin_dev_queries,
        "test": maxmin_test_queries,
    },
}

result_path = os.path.join(
    BASE,
    "results",
    "maxmin_qwen_official_results.json"
)

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"\n✓ Results saved to:")
print(result_path)

MAX–MIN RESULTS

DEV
Queries:    225
nDCG@10:    0.539095
MRR@10:     0.445926
Recall@1:   0.246667
Recall@10:  0.824444

TEST
Queries:    234
nDCG@10:    0.540578
MRR@10:     0.452110
Recall@1:   0.243590
Recall@10:  0.826923

✓ Results saved to:
/content/drive/MyDrive/maxmin_research/results/maxmin_qwen_official_results.json


In [ ]:
# METHOD 3 — MAX–MIN
# Cell 11: Paired bootstrap comparison, 10,000 replicates

import os
import json
import numpy as np

N_BOOT = 10_000
BOOT_SEED = 20261003

RESULTS_DIR = os.path.join(BASE, "results")

# ---------------------------------------------------------
# Load saved per-query results
# ---------------------------------------------------------

with open(
    os.path.join(RESULTS_DIR, "maxmin_qwen_official_results.json"),
    "r",
    encoding="utf-8"
) as f:
    maxmin_saved = json.load(f)

with open(
    os.path.join(RESULTS_DIR, "native_baseline_corrected.json"),
    "r",
    encoding="utf-8"
) as f:
    native_saved = json.load(f)

with open(
    os.path.join(RESULTS_DIR, "fixed2_results.json"),
    "r",
    encoding="utf-8"
) as f:
    fixed2_saved = json.load(f)


# ---------------------------------------------------------
# Extract per-query metrics
# ---------------------------------------------------------

METRICS = [
    "ndcg10",
    "mrr10",
    "recall1",
    "recall10",
]


def make_metric_dict(saved_result, split):

    rows = saved_result["per_query"][split]

    return {
        (row["language"], row["query_id"]): {
            metric: row[metric]
            for metric in METRICS
        }
        for row in rows
    }


native = {
    split: make_metric_dict(native_saved, split)
    for split in ["dev", "test"]
}

fixed2 = {
    split: make_metric_dict(fixed2_saved, split)
    for split in ["dev", "test"]
}

maxmin = {
    split: make_metric_dict(maxmin_saved, split)
    for split in ["dev", "test"]
}


# ---------------------------------------------------------
# Verify identical query sets
# ---------------------------------------------------------

for split in ["dev", "test"]:

    assert set(native[split]) == set(fixed2[split])
    assert set(native[split]) == set(maxmin[split])

    print(
        f"{split.upper()}: "
        f"{len(native[split])} paired queries verified"
    )


# ---------------------------------------------------------
# Paired bootstrap function
# ---------------------------------------------------------

def paired_bootstrap(
    method_a,
    method_b,
    split,
    metric,
    n_boot=N_BOOT,
    seed=BOOT_SEED,
):
    """
    Difference = method_a - method_b.

    Resamples the same query indices jointly for both methods.
    """

    keys = sorted(native[split].keys())

    a = np.array([
        method_a[split][key][metric]
        for key in keys
    ], dtype=np.float64)

    b = np.array([
        method_b[split][key][metric]
        for key in keys
    ], dtype=np.float64)

    observed = float(np.mean(a - b))

    rng = np.random.default_rng(seed)

    n = len(keys)

    bootstrap_diffs = np.empty(
        n_boot,
        dtype=np.float64
    )

    for i in range(n_boot):

        sample_idx = rng.integers(
            0,
            n,
            size=n
        )

        bootstrap_diffs[i] = np.mean(
            a[sample_idx] - b[sample_idx]
        )

    ci_low, ci_high = np.percentile(
        bootstrap_diffs,
        [2.5, 97.5]
    )

    return {
        "difference": observed,
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
    }


# ---------------------------------------------------------
# Run all comparisons
# ---------------------------------------------------------

comparisons = {
    "native_minus_maxmin": (native, maxmin),
    "fixed2_minus_maxmin": (fixed2, maxmin),
    "native_minus_fixed2": (native, fixed2),
}

bootstrap_results = {}

for split in ["dev", "test"]:

    bootstrap_results[split] = {}

    for comparison_name, (method_a, method_b) in comparisons.items():

        bootstrap_results[split][comparison_name] = {}

        for metric in METRICS:

            bootstrap_results[split][comparison_name][metric] = (
                paired_bootstrap(
                    method_a,
                    method_b,
                    split,
                    metric,
                )
            )


# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------

print("\nPAIRED BOOTSTRAP RESULTS")
print("=" * 80)
print("10,000 replicates | 95% percentile CI")
print("Difference = first method − second method")
print()

for split in ["dev", "test"]:

    print(f"\n{split.upper()}")
    print("-" * 80)

    for comparison_name in comparisons:

        print(f"\n{comparison_name}")

        for metric in METRICS:

            result = (
                bootstrap_results[
                    split
                ][comparison_name][metric]
            )

            print(
                f"  {metric:<9} "
                f"{result['difference']:+.6f} "
                f"[{result['ci_low']:+.6f}, "
                f"{result['ci_high']:+.6f}]"
            )


# ---------------------------------------------------------
# Save bootstrap results
# ---------------------------------------------------------

bootstrap_output = {
    "n_bootstrap": N_BOOT,
    "seed": BOOT_SEED,
    "confidence_level": 0.95,
    "method": "paired_query_bootstrap",
    "difference_definition": "first_method_minus_second_method",
    "results": bootstrap_results,
}

bootstrap_path = os.path.join(
    RESULTS_DIR,
    "paired_bootstrap_10000_results.json"
)

with open(
    bootstrap_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        bootstrap_output,
        f,
        indent=2
    )

print(f"\n✓ Bootstrap results saved to:")
print(bootstrap_path)

KeyError: 'per_query'

In [ ]:
# Bootstrap diagnostic: inspect saved result structures

import os
import json

RESULTS_DIR = os.path.join(BASE, "results")

result_files = {
    "native": "native_baseline_corrected.json",
    "fixed2": "fixed2_results.json",
    "maxmin": "maxmin_qwen_official_results.json",
}

for name, filename in result_files.items():

    path = os.path.join(RESULTS_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("\n" + "=" * 60)
    print(name.upper())
    print("File:", filename)
    print("Top-level keys:", list(data.keys()))

    if "metrics" in data:
        print("Metric splits:", list(data["metrics"].keys()))

    if "per_query" in data:
        print(
            "Per-query splits:",
            list(data["per_query"].keys())
        )

    print("Sample structure:")
    for key in data.keys():
        value = data[key]
        if isinstance(value, dict):
            print(f"  {key}: dict keys = {list(value.keys())[:10]}")
        elif isinstance(value, list):
            print(f"  {key}: list length = {len(value)}")
        else:
            print(f"  {key}: {type(value).__name__}")


NATIVE
File: native_baseline_corrected.json
Top-level keys: ['experiment', 'dataset', 'model', 'languages', 'total_queries', 'dev_queries', 'test_queries', 'native_passages_per_language', 'total_native_passages', 'retrieval_candidate_pool', 'top_k', 'metrics']
Metric splits: ['dev', 'test']
Sample structure:
  experiment: str
  dataset: str
  model: str
  languages: list length = 9
  total_queries: int
  dev_queries: int
  test_queries: int
  native_passages_per_language: int
  total_native_passages: int
  retrieval_candidate_pool: str
  top_k: int
  metrics: dict keys = ['dev', 'test']

FIXED2
File: fixed2_results.json
Top-level keys: ['method', 'embedding_model', 'split_protocol', 'retrieval_scope', 'metrics']
Metric splits: ['dev', 'test']
Sample structure:
  method: str
  embedding_model: str
  split_protocol: str
  retrieval_scope: str
  metrics: dict keys = ['dev', 'test']

MAXMIN
File: maxmin_qwen_official_results.json
Top-level keys: ['method', 'embedding_model', 'implementati

In [ ]:
# Bootstrap diagnostic: check whether per-query results
# from Native / Fixed-2 are still available in memory.

candidates = [
    "native_dev_queries",
    "native_test_queries",
    "fixed2_dev_queries",
    "fixed2_test_queries",
    "native_dev_per_query",
    "native_test_per_query",
    "fixed2_dev_per_query",
    "fixed2_test_per_query",
]

print("Checking notebook memory...\n")

found = []

for name in candidates:
    if name in globals():
        value = globals()[name]
        try:
            length = len(value)
        except Exception:
            length = "N/A"

        print(f"✓ {name}: exists, length = {length}")
        found.append(name)
    else:
        print(f"✗ {name}: not present")

print("\nTotal candidate objects found:", len(found))

Checking notebook memory...

✗ native_dev_queries: not present
✗ native_test_queries: not present
✗ fixed2_dev_queries: not present
✗ fixed2_test_queries: not present
✗ native_dev_per_query: not present
✗ native_test_per_query: not present
✗ fixed2_dev_per_query: not present
✗ fixed2_test_per_query: not present

Total candidate objects found: 0


In [ ]:
# REGENERATE NATIVE PER-QUERY RESULTS
# No embedding or model computation.

import os
import json
import math
import numpy as np

def dcg_at_k(relevances, k=10):
    return sum(
        (2 ** rel - 1) / math.log2(i + 2)
        for i, rel in enumerate(relevances[:k])
    )

def ndcg_at_k(relevances, k=10):
    actual = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)

    return 0.0 if ideal == 0 else actual / ideal

def mrr_at_k(relevances, k=10):
    for i, rel in enumerate(relevances[:k]):
        if rel > 0:
            return 1.0 / (i + 1)
    return 0.0

def recall_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    if not relevant_ids:
        return 0.0

    return len(
        set(retrieved_ids[:k]) & relevant_ids
    ) / len(relevant_ids)


# ---------------------------------------------------------
# Native index by language
# ---------------------------------------------------------

native_language_indices = defaultdict(list)

for idx, passage in enumerate(native_corpus):
    native_language_indices[
        passage["language"]
    ].append(idx)

# ---------------------------------------------------------
# Evaluate Native
# ---------------------------------------------------------

native_per_query = {
    "dev": [],
    "test": []
}

for item in frozen_benchmark:

    split = item["split"]
    language = item["language"]
    query_id = item["query_id"]

    relevant_parent_ids = set()

    for passage_idx, label in enumerate(item["is_selected"]):

        if label == 1:
            key = (
                language,
                query_id,
                passage_idx
            )

            relevant_parent_ids.add(
                passage_key_to_parent[key]
            )

    query_idx = query_lookup[
        (language, query_id)
    ]

    query_vector = query_embeddings[query_idx]

    candidate_indices = np.asarray(
        native_language_indices[language],
        dtype=np.int32
    )

    candidate_embeddings = native_embeddings[
        candidate_indices
    ]

    scores = candidate_embeddings @ query_vector

    order = np.argsort(
        -scores,
        kind="stable"
    )

    ranked_indices = candidate_indices[order]
    ranked_scores = scores[order]

    ranked_parent_ids = [
        native_corpus[int(idx)]["parent_id"]
        for idx in ranked_indices
    ]

    # Native has one representation per parent, so no
    # chunk-to-parent collapse is required.

    relevance = [
        1 if parent_id in relevant_parent_ids else 0
        for parent_id in ranked_parent_ids[:10]
    ]

    relevance += [0] * (10 - len(relevance))

    row = {
        "language": language,
        "query_id": query_id,
        "split": split,
        "ndcg10": ndcg_at_k(relevance, 10),
        "mrr10": mrr_at_k(relevance, 10),
        "recall1": recall_at_k(
            ranked_parent_ids,
            relevant_parent_ids,
            1
        ),
        "recall10": recall_at_k(
            ranked_parent_ids,
            relevant_parent_ids,
            10
        ),
    }

    native_per_query[split].append(row)


# ---------------------------------------------------------
# Verify counts
# ---------------------------------------------------------

assert len(native_per_query["dev"]) == 225
assert len(native_per_query["test"]) == 234

# ---------------------------------------------------------
# Aggregate check
# ---------------------------------------------------------

for split in ["dev", "test"]:

    rows = native_per_query[split]

    print(f"\n{split.upper()}")
    print(f"Queries:   {len(rows)}")
    print(
        f"nDCG@10:   "
        f"{np.mean([r['ndcg10'] for r in rows]):.6f}"
    )
    print(
        f"MRR@10:    "
        f"{np.mean([r['mrr10'] for r in rows]):.6f}"
    )
    print(
        f"Recall@1:  "
        f"{np.mean([r['recall1'] for r in rows]):.6f}"
    )
    print(
        f"Recall@10: "
        f"{np.mean([r['recall10'] for r in rows]):.6f}"
    )


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

native_per_query_path = os.path.join(
    RESULTS_DIR,
    "native_per_query_results.json"
)

with open(
    native_per_query_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        native_per_query,
        f,
        indent=2
    )

print("\n✓ Native per-query results regenerated")
print(native_per_query_path)


DEV
Queries:   225
nDCG@10:   0.551198
MRR@10:    0.452651
Recall@1:  0.260000
Recall@10: 0.846667

TEST
Queries:   234
nDCG@10:   0.551068
MRR@10:    0.453867
Recall@1:  0.258547
Recall@10: 0.858974

✓ Native per-query results regenerated
/content/drive/MyDrive/maxmin_research/results/native_per_query_results.json


In [ ]:
# REGENERATE FIXED-2 PER-QUERY RESULTS
# No embedding computation.

import os
import json
import numpy as np
from collections import defaultdict

# ---------------------------------------------------------
# Paths — explicitly restored after runtime restart
# ---------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/maxmin_research"

CACHE_DIR = os.path.join(BASE_DIR, "cache")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

# ---------------------------------------------------------
# Load cached Fixed-2 embeddings
# ---------------------------------------------------------

fixed2_emb_path = os.path.join(
    CACHE_DIR,
    "qwen_fixed2_embeddings",
    "qwen_fixed2_embeddings.npy"
)

fixed2_ids_path = os.path.join(
    CACHE_DIR,
    "qwen_fixed2_embeddings",
    "fixed2_chunk_ids.json"
)

fixed2_embeddings = np.load(fixed2_emb_path)

with open(fixed2_ids_path, "r", encoding="utf-8") as f:
    fixed2_chunk_ids = json.load(f)

print("Fixed-2 embeddings:", fixed2_embeddings.shape)
print("Fixed-2 chunk IDs:", len(fixed2_chunk_ids))

assert fixed2_embeddings.shape[0] == len(fixed2_chunk_ids)
assert fixed2_embeddings.shape[1] == 1024


# ---------------------------------------------------------
# Build chunk lookup
# ---------------------------------------------------------

fixed2_chunk_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in fixed2_corpus
}

assert len(fixed2_chunk_by_id) == len(fixed2_corpus)

for i, chunk_id in enumerate(fixed2_chunk_ids[:100]):
    assert fixed2_corpus[i]["chunk_id"] == chunk_id

print("✓ Fixed-2 embedding/ID alignment verified")


# ---------------------------------------------------------
# Build language-specific indices
# ---------------------------------------------------------

fixed2_language_indices = defaultdict(list)

for idx, chunk_id in enumerate(fixed2_chunk_ids):

    chunk = fixed2_chunk_by_id[chunk_id]

    fixed2_language_indices[
        chunk["language"]
    ].append(idx)

print("\nFixed-2 chunks by language:")

for language in sorted(fixed2_language_indices):
    print(
        f"{language}: "
        f"{len(fixed2_language_indices[language])}"
    )


# ---------------------------------------------------------
# Evaluate Fixed-2 per query
# ---------------------------------------------------------

fixed2_per_query = {
    "dev": [],
    "test": []
}

for item in frozen_benchmark:

    split = item["split"]
    language = item["language"]
    query_id = item["query_id"]

    relevant_parent_ids = set()

    for passage_idx, label in enumerate(item["is_selected"]):

        if label == 1:

            key = (
                language,
                query_id,
                passage_idx
            )

            relevant_parent_ids.add(
                passage_key_to_parent[key]
            )

    query_idx = query_lookup[
        (language, query_id)
    ]

    query_vector = query_embeddings[query_idx]

    candidate_indices = np.asarray(
        fixed2_language_indices[language],
        dtype=np.int32
    )

    candidate_embeddings = fixed2_embeddings[
        candidate_indices
    ]

    scores = candidate_embeddings @ query_vector

    order = np.argsort(
        -scores,
        kind="stable"
    )

    ranked_indices = candidate_indices[order]
    ranked_scores = scores[order]

    # Collapse chunk ranking -> parent ranking
    parent_best_score = {}

    for idx, score in zip(
        ranked_indices,
        ranked_scores
    ):

        chunk_id = fixed2_chunk_ids[int(idx)]

        parent_id = fixed2_chunk_by_id[
            chunk_id
        ]["parent_id"]

        if (
            parent_id not in parent_best_score
            or score > parent_best_score[parent_id]
        ):
            parent_best_score[parent_id] = float(score)

    ranked_parents = sorted(
        parent_best_score.items(),
        key=lambda x: -x[1]
    )

    ranked_parent_ids = [
        parent_id
        for parent_id, score in ranked_parents
    ]

    relevance = [
        1 if parent_id in relevant_parent_ids else 0
        for parent_id in ranked_parent_ids[:10]
    ]

    relevance += [0] * (10 - len(relevance))

    row = {
        "language": language,
        "query_id": query_id,
        "split": split,
        "ndcg10": ndcg_at_k(relevance, 10),
        "mrr10": mrr_at_k(relevance, 10),
        "recall1": recall_at_k(
            ranked_parent_ids,
            relevant_parent_ids,
            1
        ),
        "recall10": recall_at_k(
            ranked_parent_ids,
            relevant_parent_ids,
            10
        )
    }

    fixed2_per_query[split].append(row)


# ---------------------------------------------------------
# Verify counts
# ---------------------------------------------------------

assert len(fixed2_per_query["dev"]) == 225
assert len(fixed2_per_query["test"]) == 234


# ---------------------------------------------------------
# Aggregate check
# ---------------------------------------------------------

for split in ["dev", "test"]:

    rows = fixed2_per_query[split]

    print(f"\n{split.upper()}")
    print(f"Queries:   {len(rows)}")

    print(
        f"nDCG@10:   "
        f"{np.mean([r['ndcg10'] for r in rows]):.6f}"
    )

    print(
        f"MRR@10:    "
        f"{np.mean([r['mrr10'] for r in rows]):.6f}"
    )

    print(
        f"Recall@1:  "
        f"{np.mean([r['recall1'] for r in rows]):.6f}"
    )

    print(
        f"Recall@10: "
        f"{np.mean([r['recall10'] for r in rows]):.6f}"
    )


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

fixed2_per_query_path = os.path.join(
    RESULTS_DIR,
    "fixed2_per_query_results.json"
)

with open(
    fixed2_per_query_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        fixed2_per_query,
        f,
        indent=2
    )

print("\n✓ Fixed-2 per-query results regenerated")
print(fixed2_per_query_path)

Fixed-2 embeddings: (18171, 1024)
Fixed-2 chunk IDs: 18171
✓ Fixed-2 embedding/ID alignment verified

Fixed-2 chunks by language:
bengali: 1875
gujarati: 2028
hindi: 2038
kannada: 2086
malayalam: 2021
marathi: 2111
punjabi: 2029
tamil: 1995
telugu: 1988

DEV
Queries:   225
nDCG@10:   0.543190
MRR@10:    0.449418
Recall@1:  0.264444
Recall@10: 0.826667

TEST
Queries:   234
nDCG@10:   0.541111
MRR@10:    0.449412
Recall@1:  0.258547
Recall@10: 0.831197

✓ Fixed-2 per-query results regenerated
/content/drive/MyDrive/maxmin_research/results/fixed2_per_query_results.json


In [ ]:
import json
import os

path = "/content/drive/MyDrive/maxmin_research/results/fixed2_results.json"

with open(path, "r", encoding="utf-8") as f:
    old_fixed2 = json.load(f)

print("Top-level keys:")
print(old_fixed2.keys())

print("\nMetrics:")
print(json.dumps(old_fixed2["metrics"], indent=2))

Top-level keys:
dict_keys(['method', 'embedding_model', 'split_protocol', 'retrieval_scope', 'metrics'])

Metrics:
{
  "dev": {
    "split": "dev",
    "queries": 225,
    "nDCG@10": 0.5751810541939495,
    "MRR@10": 0.44926984126984126,
    "Recall@10": 0.8266666666666667
  },
  "test": {
    "split": "test",
    "queries": 234,
    "nDCG@10": 0.5676296894610642,
    "MRR@10": 0.44905541988875325,
    "Recall@10": 0.8311965811965812
  }
}


In [ ]:
import inspect

print("Functions currently available that may be relevant:\n")

for name in [
    "ndcg_at_k",
    "mrr_at_k",
    "recall_at_k"
]:
    if name in globals():
        print(f"\n===== {name} =====")
        print(inspect.getsource(globals()[name]))

Functions currently available that may be relevant:


===== ndcg_at_k =====
def ndcg_at_k(relevances, k=10):
    actual = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)

    return 0.0 if ideal == 0 else actual / ideal


===== mrr_at_k =====
def mrr_at_k(relevances, k=10):
    for i, rel in enumerate(relevances[:k]):
        if rel > 0:
            return 1.0 / (i + 1)
    return 0.0


===== recall_at_k =====
def recall_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    if not relevant_ids:
        return 0.0

    return len(
        set(retrieved_ids[:k]) & relevant_ids
    ) / len(relevant_ids)



In [ ]:
# Inspect notebook history for the original Fixed-2 evaluation code.
# No computation or retrieval is performed.

from IPython import get_ipython

history = get_ipython().history_manager

matches = []

for session, line_no, source in history.get_range():
    if (
        "fixed2" in source.lower()
        and (
            "ndcg" in source.lower()
            or "recall" in source.lower()
            or "mrr" in source.lower()
        )
    ):
        matches.append((session, line_no, source))

print(f"Found {len(matches)} matching history entries.\n")

for session, line_no, source in matches[-10:]:
    print("=" * 80)
    print(f"Session {session}, cell {line_no}")
    print("=" * 80)
    print(source[:12000])

Found 6 matching history entries.

Session 0, cell 18
from sklearn.metrics import ndcg_score

def evaluate_fixed2(split_name):
    split_queries = [
        (i, q)
        for i, q in enumerate(frozen_benchmark)
        if q["split"] == split_name
    ]

    ndcg_scores = []
    mrr_scores = []
    recall_scores = []

    for q_idx, q in split_queries:
        lang = q["language"]

        # Query embedding
        query_vec = query_embeddings[q_idx]

        # Fixed-2 candidates in the query's language
        candidate_idx = fixed2_lang_indices[lang]

        candidate_emb = fixed2_embeddings[candidate_idx]

        # Exact cosine similarity because both vectors are normalized
        scores = candidate_emb @ query_vec

        # Sort chunks by similarity
        ranked_local = np.argsort(-scores)

        ranked_global = candidate_idx[ranked_local]

        # -------------------------------------------------
        # Collapse chunks to parent passages
        # Keep the highest-sco

In [ ]:
# EXACT ORIGINAL FIXED-2 EVALUATION
# Reproduces the original aggregate protocol and saves per-query scores.
# No embeddings are recomputed.

import os
import json
import numpy as np
from sklearn.metrics import ndcg_score

BASE_DIR = "/content/drive/MyDrive/maxmax_research"

In [ ]:
# EXACT ORIGINAL FIXED-2 EVALUATION
# Saves per-query metrics for paired bootstrap.
# No embedding computation.

import os
import json
import numpy as np
from sklearn.metrics import ndcg_score

BASE_DIR = "/content/drive/MyDrive/maxmin_research"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

# ---------------------------------------------------------
# Load cached Fixed-2 embeddings
# ---------------------------------------------------------

fixed2_emb_path = os.path.join(
    CACHE_DIR,
    "qwen_fixed2_embeddings",
    "qwen_fixed2_embeddings.npy"
)

fixed2_ids_path = os.path.join(
    CACHE_DIR,
    "qwen_fixed2_embeddings",
    "fixed2_chunk_ids.json"
)

fixed2_embeddings = np.load(fixed2_emb_path)

with open(fixed2_ids_path, "r", encoding="utf-8") as f:
    fixed2_chunk_ids = json.load(f)

print("Fixed-2 embeddings:", fixed2_embeddings.shape)
print("Fixed-2 chunk IDs:", len(fixed2_chunk_ids))

assert fixed2_embeddings.shape == (18171, 1024)
assert len(fixed2_chunk_ids) == 18171


# ---------------------------------------------------------
# Build exact chunk lookup
# ---------------------------------------------------------

fixed2_chunk_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in fixed2_corpus
}

assert len(fixed2_chunk_by_id) == len(fixed2_corpus)

for i, chunk_id in enumerate(fixed2_chunk_ids):
    assert fixed2_corpus[i]["chunk_id"] == chunk_id

print("✓ Fixed-2 embedding/ID alignment verified")


# ---------------------------------------------------------
# Language-specific candidate indices
# ---------------------------------------------------------

fixed2_lang_indices = {}

for language in sorted(
    set(chunk["language"] for chunk in fixed2_corpus)
):
    fixed2_lang_indices[language] = np.array(
        [
            i
            for i, chunk_id in enumerate(fixed2_chunk_ids)
            if fixed2_chunk_by_id[chunk_id]["language"] == language
        ],
        dtype=np.int32
    )

print("\nFixed-2 chunks by language:")
for language in sorted(fixed2_lang_indices):
    print(
        f"{language}: "
        f"{len(fixed2_lang_indices[language])}"
    )


# ---------------------------------------------------------
# Exact original evaluation
# ---------------------------------------------------------

fixed2_per_query = {
    "dev": [],
    "test": []
}

for q_idx, q in enumerate(frozen_benchmark):

    split = q["split"]
    lang = q["language"]
    query_id = q["query_id"]

    query_vec = query_embeddings[
        query_lookup[(lang, query_id)]
    ]

    # Query-language candidates only
    candidate_idx = fixed2_lang_indices[lang]
    candidate_emb = fixed2_embeddings[candidate_idx]

    # Exact cosine
    scores = candidate_emb @ query_vec

    # Rank chunks
    ranked_local = np.argsort(-scores)
    ranked_global = candidate_idx[ranked_local]

    # -----------------------------------------------------
    # Collapse chunks -> parent passages
    # Keep highest-scoring chunk per parent
    # -----------------------------------------------------

    parent_best_score = {}

    for idx, score in zip(
        ranked_global,
        scores[ranked_local]
    ):

        chunk_id = fixed2_chunk_ids[int(idx)]
        parent_id = fixed2_chunk_by_id[
            chunk_id
        ]["parent_id"]

        if (
            parent_id not in parent_best_score
            or score > parent_best_score[parent_id]
        ):
            parent_best_score[parent_id] = float(score)

    ranked_parents = sorted(
        parent_best_score,
        key=parent_best_score.get,
        reverse=True
    )

    # -----------------------------------------------------
    # Exact frozen relevance mapping
    # -----------------------------------------------------

    relevance = {}

    for passage_idx, label in enumerate(
        q["is_selected"]
    ):

        key = (
            lang,
            query_id,
            passage_idx
        )

        parent_id = passage_key_to_parent[key]

        relevance[parent_id] = int(label)

    # -----------------------------------------------------
    # Top-10
    # -----------------------------------------------------

    top10 = ranked_parents[:10]

    gains = np.array(
        [
            relevance.get(pid, 0)
            for pid in top10
        ],
        dtype=float
    )

    gains = np.pad(
        gains,
        (0, max(0, 10 - len(gains)))
    )[:10]

    ideal = np.sort(
        np.array(
            list(relevance.values()),
            dtype=float
        )
    )[::-1][:10]

    ideal = np.pad(
        ideal,
        (0, max(0, 10 - len(ideal)))
    )[:10]

    # EXACT ORIGINAL nDCG IMPLEMENTATION
    ndcg = ndcg_score(
        ideal.reshape(1, -1),
        gains.reshape(1, -1),
        k=10
    )

    # MRR@10
    rr = 0.0

    for rank, pid in enumerate(
        top10,
        start=1
    ):
        if relevance.get(pid, 0) > 0:
            rr = 1.0 / rank
            break

    # Recall@1
    total_relevant = sum(relevance.values())

    recall1 = (
        1.0
        if top10 and relevance.get(top10[0], 0) > 0
        else 0.0
    )

    # Recall@10
    retrieved_relevant = sum(
        relevance.get(pid, 0)
        for pid in top10
    )

    recall10 = (
        retrieved_relevant / total_relevant
        if total_relevant > 0
        else 0.0
    )

    fixed2_per_query[split].append({
        "language": lang,
        "query_id": query_id,
        "split": split,
        "ndcg10": float(ndcg),
        "mrr10": float(rr),
        "recall1": float(recall1),
        "recall10": float(recall10)
    })


# ---------------------------------------------------------
# Verify query counts
# ---------------------------------------------------------

assert len(fixed2_per_query["dev"]) == 225
assert len(fixed2_per_query["test"]) == 234


# ---------------------------------------------------------
# Aggregate results
# ---------------------------------------------------------

for split in ["dev", "test"]:

    rows = fixed2_per_query[split]

    print(f"\n{split.upper()}")
    print(f"Queries:   {len(rows)}")

    print(
        f"nDCG@10:   "
        f"{np.mean([r['ndcg10'] for r in rows]):.6f}"
    )

    print(
        f"MRR@10:    "
        f"{np.mean([r['mrr10'] for r in rows]):.6f}"
    )

    print(
        f"Recall@1:  "
        f"{np.mean([r['recall1'] for r in rows]):.6f}"
    )

    print(
        f"Recall@10: "
        f"{np.mean([r['recall10'] for r in rows]):.6f}"
    )


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

output_path = os.path.join(
    RESULTS_DIR,
    "fixed2_per_query_results.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        fixed2_per_query,
        f,
        indent=2
    )

print("\n✓ Exact Fixed-2 per-query results saved")
print(output_path)

Fixed-2 embeddings: (18171, 1024)
Fixed-2 chunk IDs: 18171
✓ Fixed-2 embedding/ID alignment verified

Fixed-2 chunks by language:
bengali: 1875
gujarati: 2028
hindi: 2038
kannada: 2086
malayalam: 2021
marathi: 2111
punjabi: 2029
tamil: 1995
telugu: 1988

DEV
Queries:   225
nDCG@10:   0.575181
MRR@10:    0.449270
Recall@1:  0.271111
Recall@10: 0.826667

TEST
Queries:   234
nDCG@10:   0.567630
MRR@10:    0.449055
Recall@1:  0.273504
Recall@10: 0.831197

✓ Exact Fixed-2 per-query results saved
/content/drive/MyDrive/maxmin_research/results/fixed2_per_query_results.json


In [ ]:
import os
import json
import numpy as np

BASE_DIR = "/content/drive/MyDrive/maxmin_research"
RESULTS_DIR = os.path.join(BASE_DIR, "results")

N_BOOT = 10_000
BOOT_SEED = 20261003

METRICS = [
    "ndcg10",
    "mrr10",
    "recall1",
    "recall10",
]


# ---------------------------------------------------------
# Load per-query results
# ---------------------------------------------------------

with open(
    os.path.join(
        RESULTS_DIR,
        "native_per_query_results.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    native_saved = json.load(f)

with open(
    os.path.join(
        RESULTS_DIR,
        "fixed2_per_query_results.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    fixed2_saved = json.load(f)

with open(
    os.path.join(
        RESULTS_DIR,
        "maxmin_qwen_official_results.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    maxmin_saved = json.load(f)


# ---------------------------------------------------------
# Extract per-query rows
# ---------------------------------------------------------

native_rows = {
    split: native_saved[split]
    for split in ["dev", "test"]
}

fixed2_rows = {
    split: fixed2_saved[split]
    for split in ["dev", "test"]
}

maxmin_rows = {
    split: maxmin_saved["per_query"][split]
    for split in ["dev", "test"]
}


# ---------------------------------------------------------
# Convert to:
# (language, query_id) -> metrics
# ---------------------------------------------------------

def make_metric_dict(rows):

    return {
        (row["language"], row["query_id"]): {
            metric: row[metric]
            for metric in METRICS
        }
        for row in rows
    }


native = {
    split: make_metric_dict(native_rows[split])
    for split in ["dev", "test"]
}

fixed2 = {
    split: make_metric_dict(fixed2_rows[split])
    for split in ["dev", "test"]
}

maxmin = {
    split: make_metric_dict(maxmin_rows[split])
    for split in ["dev", "test"]
}


# ---------------------------------------------------------
# Verify identical query sets
# ---------------------------------------------------------

for split in ["dev", "test"]:

    native_keys = set(native[split])
    fixed2_keys = set(fixed2[split])
    maxmin_keys = set(maxmin[split])

    assert native_keys == fixed2_keys
    assert native_keys == maxmin_keys

    print(
        f"{split.upper()}: "
        f"{len(native_keys)} paired queries verified"
    )


# ---------------------------------------------------------
# Paired bootstrap
# ---------------------------------------------------------

def paired_bootstrap(
    method_a,
    method_b,
    split,
    metric,
    n_boot=N_BOOT,
    seed=BOOT_SEED,
):

    keys = sorted(native[split].keys())

    a = np.array(
        [
            method_a[split][key][metric]
            for key in keys
        ],
        dtype=np.float64
    )

    b = np.array(
        [
            method_b[split][key][metric]
            for key in keys
        ],
        dtype=np.float64
    )

    # Observed paired difference
    observed = float(
        np.mean(a - b)
    )

    rng = np.random.default_rng(seed)

    n = len(keys)

    bootstrap_diffs = np.empty(
        n_boot,
        dtype=np.float64
    )

    for i in range(n_boot):

        sample_idx = rng.integers(
            0,
            n,
            size=n
        )

        bootstrap_diffs[i] = np.mean(
            a[sample_idx] - b[sample_idx]
        )

    ci_low, ci_high = np.percentile(
        bootstrap_diffs,
        [2.5, 97.5]
    )

    return {
        "difference": observed,
        "ci_low": float(ci_low),
        "ci_high": float(ci_high)
    }


# ---------------------------------------------------------
# Comparisons
# ---------------------------------------------------------

comparisons = {
    "native_minus_maxmin": (native, maxmin),
    "fixed2_minus_maxmin": (fixed2, maxmin),
    "native_minus_fixed2": (native, fixed2),
}


# ---------------------------------------------------------
# Run all comparisons
# ---------------------------------------------------------

bootstrap_results = {}

for split in ["dev", "test"]:

    bootstrap_results[split] = {}

    for comparison_name, (
        method_a,
        method_b
    ) in comparisons.items():

        bootstrap_results[split][
            comparison_name
        ] = {}

        for metric in METRICS:

            bootstrap_results[split][
                comparison_name
            ][metric] = paired_bootstrap(
                method_a,
                method_b,
                split,
                metric
            )


# ---------------------------------------------------------
# Print
# ---------------------------------------------------------

print("\nPAIRED BOOTSTRAP RESULTS")
print("=" * 80)
print("10,000 replicates | 95% percentile CI")
print("Difference = first method − second method")

for split in ["dev", "test"]:

    print(f"\n{split.upper()}")
    print("-" * 80)

    for comparison_name in comparisons:

        print(f"\n{comparison_name}")

        for metric in METRICS:

            result = (
                bootstrap_results[
                    split
                ][comparison_name][metric]
            )

            print(
                f"  {metric:<9} "
                f"{result['difference']:+.6f} "
                f"[{result['ci_low']:+.6f}, "
                f"{result['ci_high']:+.6f}]"
            )


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

bootstrap_output = {
    "n_bootstrap": N_BOOT,
    "seed": BOOT_SEED,
    "confidence_level": 0.95,
    "method": "paired_query_bootstrap",
    "difference_definition":
        "first_method_minus_second_method",
    "results": bootstrap_results,
}

bootstrap_path = os.path.join(
    RESULTS_DIR,
    "paired_bootstrap_10000_results.json"
)

with open(
    bootstrap_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        bootstrap_output,
        f,
        indent=2
    )

print(
    f"\n✓ Bootstrap results saved to:\n"
    f"{bootstrap_path}"
)

DEV: 225 paired queries verified
TEST: 234 paired queries verified

PAIRED BOOTSTRAP RESULTS
10,000 replicates | 95% percentile CI
Difference = first method − second method

DEV
--------------------------------------------------------------------------------

native_minus_maxmin
  ndcg10    +0.012103 [-0.010630, +0.034572]
  mrr10     +0.006725 [-0.022209, +0.035031]
  recall1   +0.013333 [-0.031111, +0.057778]
  recall10  +0.022222 [+0.000000, +0.046667]

fixed2_minus_maxmin
  ndcg10    +0.036086 [+0.005456, +0.066641]
  mrr10     +0.003344 [-0.017447, +0.024690]
  recall1   +0.024444 [-0.008889, +0.060000]
  recall10  +0.002222 [-0.015556, +0.020000]

native_minus_fixed2
  ndcg10    -0.023983 [-0.057987, +0.009356]
  mrr10     +0.003381 [-0.023736, +0.029696]
  recall1   -0.011111 [-0.055556, +0.031111]
  recall10  +0.020000 [-0.002222, +0.044444]

TEST
--------------------------------------------------------------------------------

native_minus_maxmin
  ndcg10    +0.010490 [-0.0122

In [ ]:
import os
import json

BASE_DIR = "/content/drive/MyDrive/maxmin_research"
RESULTS_DIR = os.path.join(BASE_DIR, "results")

files_to_check = [
    "native_per_query_results.json",
    "fixed2_per_query_results.json",
    "maxmin_qwen_official_results.json",
]

for filename in files_to_check:

    path = os.path.join(RESULTS_DIR, filename)

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level keys:", list(data.keys()))

    if "per_query" in data:
        print("per_query splits:", list(data["per_query"].keys()))

        for split in ["dev", "test"]:
            if split in data["per_query"]:
                print(
                    f"{split}: "
                    f"{len(data['per_query'][split])} rows"
                )


native_per_query_results.json
Top-level keys: ['dev', 'test']

fixed2_per_query_results.json
Top-level keys: ['dev', 'test']

maxmin_qwen_official_results.json
Top-level keys: ['method', 'embedding_model', 'implementation_reference', 'sigmoid_variant', 'hard_thr', 'init_const', 'c', 'split_protocol', 'retrieval_scope', 'retrieval', 'parent_collapse', 'metrics', 'per_query']
per_query splits: ['dev', 'test']
dev: 225 rows
test: 234 rows


In [ ]:
# Cell 1 — Load Max–Min vs Native per-query results + structural data

import json
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/maxmin_research")

# Load per-query retrieval results
with open(BASE / "results/native_per_query_results.json", "r") as f:
    native_results = json.load(f)

with open(BASE / "results/maxmin_qwen_official_results.json", "r") as f:
    maxmin_results = json.load(f)

# Load frozen Max–Min segmentation
with open(BASE / "frozen/maxmin_segmentation_qwen_official.json", "r") as f:
    maxmin_seg = json.load(f)

# Load frozen benchmark
with open(BASE / "frozen/msmarco_xi_frozen_benchmark.json", "r") as f:
    benchmark = json.load(f)

print("Native keys:", native_results.keys())
print("Max–Min keys:", maxmin_results.keys())
print("Max–Min segmentation records:", len(maxmin_seg))
print("Frozen benchmark queries:", len(benchmark))

# Extract per-query results
native_dev = native_results["dev"]
native_test = native_results["test"]

maxmin_dev = maxmin_results["per_query"]["dev"]
maxmin_test = maxmin_results["per_query"]["test"]

print("\nPer-query counts:")
print("DEV  — Native:", len(native_dev), "| Max–Min:", len(maxmin_dev))
print("TEST — Native:", len(native_test), "| Max–Min:", len(maxmin_test))

# Check that the query keys match exactly
def get_key(row):
    return (row["language"], row["query_id"])

native_dev_keys = {get_key(x) for x in native_dev}
maxmin_dev_keys = {get_key(x) for x in maxmin_dev}

native_test_keys = {get_key(x) for x in native_test}
maxmin_test_keys = {get_key(x) for x in maxmin_test}

assert native_dev_keys == maxmin_dev_keys
assert native_test_keys == maxmin_test_keys

print("\n✓ DEV query sets match exactly")
print("✓ TEST query sets match exactly")
print("✓ Ready for ΔnDCG analysis")

Native keys: dict_keys(['dev', 'test'])
Max–Min keys: dict_keys(['method', 'embedding_model', 'implementation_reference', 'sigmoid_variant', 'hard_thr', 'init_const', 'c', 'split_protocol', 'retrieval_scope', 'retrieval', 'parent_collapse', 'metrics', 'per_query'])
Max–Min segmentation records: 13
Frozen benchmark queries: 459

Per-query counts:
DEV  — Native: 225 | Max–Min: 225
TEST — Native: 234 | Max–Min: 234

✓ DEV query sets match exactly
✓ TEST query sets match exactly
✓ Ready for ΔnDCG analysis


In [ ]:
# Cell 2 — Per-query ΔnDCG@10: Max–Min − Native

def build_delta_df(native_rows, maxmin_rows):
    native_map = {
        (r["language"], r["query_id"]): r
        for r in native_rows
    }
    maxmin_map = {
        (r["language"], r["query_id"]): r
        for r in maxmin_rows
    }

    rows = []

    for key in sorted(native_map):
        n = native_map[key]
        m = maxmin_map[key]

        rows.append({
            "language": key[0],
            "query_id": key[1],
            "query": n.get("query", ""),
            "native_ndcg": n["ndcg10"],
            "maxmin_ndcg": m["ndcg10"],
            "delta_ndcg": m["ndcg10"] - n["ndcg10"],
            "native_mrr": n["mrr10"],
            "maxmin_mrr": m["mrr10"],
            "delta_mrr": m["mrr10"] - n["mrr10"],
            "native_recall10": n["recall10"],
            "maxmin_recall10": m["recall10"],
            "delta_recall10": m["recall10"] - n["recall10"],
        })

    return pd.DataFrame(rows)


dev_delta = build_delta_df(native_dev, maxmin_dev)
test_delta = build_delta_df(native_test, maxmin_test)

def summarize_delta(df, name):
    d = df["delta_ndcg"]

    print(f"\n{name}")
    print("=" * 50)
    print(f"Queries:              {len(df)}")
    print(f"Mean ΔnDCG:           {d.mean():+.6f}")
    print(f"Median ΔnDCG:         {d.median():+.6f}")
    print(f"Std ΔnDCG:            {d.std():.6f}")
    print(f"Max improvement:      {d.max():+.6f}")
    print(f"Max degradation:      {d.min():+.6f}")
    print(f"Improved queries:     {(d > 0).sum()} ({(d > 0).mean()*100:.1f}%)")
    print(f"Equal queries:        {(d == 0).sum()} ({(d == 0).mean()*100:.1f}%)")
    print(f"Degraded queries:     {(d < 0).sum()} ({(d < 0).mean()*100:.1f}%)")


summarize_delta(dev_delta, "DEV")
summarize_delta(test_delta, "TEST")

print("\n✓ Per-query ΔnDCG calculated")


DEV
Queries:              225
Mean ΔnDCG:           -0.012103
Median ΔnDCG:         +0.000000
Std ΔnDCG:            0.171238
Max improvement:      +0.643793
Max degradation:      -0.613147
Improved queries:     41 (18.2%)
Equal queries:        127 (56.4%)
Degraded queries:     57 (25.3%)

TEST
Queries:              234
Mean ΔnDCG:           -0.010490
Median ΔnDCG:         +0.000000
Std ΔnDCG:            0.178380
Max improvement:      +1.000000
Max degradation:      -0.643793
Improved queries:     49 (20.9%)
Equal queries:        133 (56.8%)
Degraded queries:     52 (22.2%)

✓ Per-query ΔnDCG calculated


In [ ]:
# Cell 3 — Largest Max–Min gains and degradations

def show_extremes(df, name, n=10):
    print(f"\n{'='*80}")
    print(f"{name} — TOP {n} IMPROVEMENTS")
    print("=" * 80)

    cols = [
        "language", "query_id",
        "native_ndcg", "maxmin_ndcg", "delta_ndcg",
        "query"
    ]

    print(
        df.sort_values("delta_ndcg", ascending=False)[cols]
        .head(n)
        .to_string(index=False)
    )

    print(f"\n{'='*80}")
    print(f"{name} — TOP {n} DEGRADATIONS")
    print("=" * 80)

    print(
        df.sort_values("delta_ndcg", ascending=True)[cols]
        .head(n)
        .to_string(index=False)
    )


show_extremes(dev_delta, "DEV")
show_extremes(test_delta, "TEST")


DEV — TOP 10 IMPROVEMENTS
 language  query_id  native_ndcg  maxmin_ndcg  delta_ndcg query
   telugu    265717     0.356207          1.0    0.643793      
  kannada    211441     0.430677          1.0    0.569323      
  kannada    232017     0.500000          1.0    0.500000      
 gujarati    113570     0.500000          1.0    0.500000      
 gujarati     44760     0.630930          1.0    0.369070      
  kannada   1060361     0.630930          1.0    0.369070      
    tamil    165349     0.630930          1.0    0.369070      
malayalam    158720     0.630930          1.0    0.369070      
  bengali     53335     0.630930          1.0    0.369070      
    tamil   1102432     0.630930          1.0    0.369070      

DEV — TOP 10 DEGRADATIONS
 language  query_id  native_ndcg  maxmin_ndcg  delta_ndcg query
  bengali    373460     1.000000     0.386853   -0.613147      
malayalam    328321     1.000000     0.430677   -0.569323      
    hindi   1060361     1.000000     0.580279   -0

In [ ]:
# Cell 4A — Inspect Max–Min segmentation JSON structure

print("Type:", type(maxmin_seg))

if isinstance(maxmin_seg, dict):
    print("Top-level keys:", list(maxmin_seg.keys())[:20])

    for k, v in list(maxmin_seg.items())[:3]:
        print("\nKEY:", k)
        print("VALUE TYPE:", type(v))
        if isinstance(v, list):
            print("LIST LENGTH:", len(v))
            if len(v) > 0:
                print("FIRST ITEM:", v[0])
        elif isinstance(v, dict):
            print("DICT KEYS:", list(v.keys())[:20])
        else:
            print("VALUE:", v)

elif isinstance(maxmin_seg, list):
    print("List length:", len(maxmin_seg))
    print("First item:", maxmin_seg[0])

Type: <class 'dict'>
Top-level keys: ['method', 'embedding_model', 'hard_thr', 'init_const', 'c', 'sigmoid_variant', 'implementation_reference', 'max_chunk_size', 'similarity_window', 'num_passages', 'num_sentences', 'num_chunks', 'segmentation']

KEY: method
VALUE TYPE: <class 'str'>
VALUE: max_min

KEY: embedding_model
VALUE TYPE: <class 'str'>
VALUE: Qwen/Qwen3-Embedding-0.6B

KEY: hard_thr
VALUE TYPE: <class 'float'>
VALUE: 0.6


In [ ]:
# Cell 4B — Build per-query structural features from frozen Max–Min segmentation

# Map parent -> Max–Min chunks
parent_chunks = {}

for row in maxmin_seg["segmentation"]:
    parent_chunks[row["parent_id"]] = row["chunks"]

print("Max–Min segmentation records:", len(parent_chunks))

# Map parent -> sentence records
parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(row["parent_id"], []).append(row)

print("Sentence-corpus parent records:", len(parent_sentences))


def structural_features(parent_id):
    sentences = parent_sentences[parent_id]
    chunks = parent_chunks[parent_id]

    n_sentences = len(sentences)
    n_chunks = len(chunks)

    chunk_sizes = [len(c["sentence_ids"]) for c in chunks]

    singleton_pct = (
        sum(s == 1 for s in chunk_sizes) / n_chunks
        if n_chunks else np.nan
    )

    avg_sent_per_chunk = (
        np.mean(chunk_sizes) if chunk_sizes else np.nan
    )

    # Adjacent-sentence cosine similarity statistics
    if n_sentences >= 2:
        vecs = np.vstack([
            sentence_embeddings[sentence_id_to_idx[s["sentence_id"]]]
            for s in sentences
        ])

        adjacent_sims = np.sum(vecs[:-1] * vecs[1:], axis=1)

        adjacent_sim_std = float(np.std(adjacent_sims))
        adjacent_sim_mean = float(np.mean(adjacent_sims))
    else:
        adjacent_sim_std = 0.0
        adjacent_sim_mean = np.nan

    return {
        "n_sentences": n_sentences,
        "n_chunks": n_chunks,
        "avg_sentences_per_chunk": avg_sent_per_chunk,
        "singleton_pct": singleton_pct,
        "was_split": int(n_chunks > 1),
        "adjacent_sim_std": adjacent_sim_std,
        "adjacent_sim_mean": adjacent_sim_mean,
    }


# Compute once for every parent
structural = {
    parent_id: structural_features(parent_id)
    for parent_id in parent_sentences
}

print("\nStructural records:", len(structural))

example_parent = next(iter(structural))

print("\nExample parent:", example_parent)
print("Features:", structural[example_parent])

assert len(structural) == 8973
assert set(structural) == set(parent_chunks)
assert set(structural) == set(parent_sentences)

print("\n✓ All 8,973 native passages have structural features")
print("✓ Frozen Max–Min segmentation aligned")
print("✓ No embeddings or segmentation recomputed")

Max–Min segmentation records: 8973
Sentence-corpus parent records: 8973


KeyError: 'sentence_ids'

In [ ]:
# Cell 4C — Inspect one frozen Max–Min chunk

seg = maxmin_seg["segmentation"]

print("Number of segmentation records:", len(seg))

print("\nFirst record keys:")
print(seg[0].keys())

print("\nFirst record:")
print(seg[0])

print("\nFirst chunk:")
print(seg[0]["chunks"][0])

print("\nFirst chunk keys:")
print(seg[0]["chunks"][0].keys())

Number of segmentation records: 8973

First record keys:
dict_keys(['parent_id', 'language', 'source_query_id', 'passage_idx', 'num_sentences', 'num_chunks', 'chunks'])

First record:
{'parent_id': 'hindi_1102432_p00', 'language': 'hindi', 'source_query_id': 1102432, 'passage_idx': 0, 'num_sentences': 4, 'num_chunks': 2, 'chunks': [{'chunk_idx': 0, 'sentence_indices': [0, 1], 'num_sentences': 2}, {'chunk_idx': 1, 'sentence_indices': [2, 3], 'num_sentences': 2}]}

First chunk:
{'chunk_idx': 0, 'sentence_indices': [0, 1], 'num_sentences': 2}

First chunk keys:
dict_keys(['chunk_idx', 'sentence_indices', 'num_sentences'])


In [ ]:
# Cell 4D — Build structural features for Max–Min vs Native analysis

# Map parent -> Max–Min segmentation record
parent_seg = {
    row["parent_id"]: row
    for row in maxmin_seg["segmentation"]
}

# Map parent -> sentence records
parent_sentences = {}

for row in sentence_corpus:
    parent_sentences.setdefault(row["parent_id"], []).append(row)

print("Segmentation parents:", len(parent_seg))
print("Sentence-corpus parents:", len(parent_sentences))


def structural_features(parent_id):
    seg = parent_seg[parent_id]

    n_sentences = seg["num_sentences"]
    n_chunks = seg["num_chunks"]

    chunks = seg["chunks"]

    # Exact number of sentences in every Max–Min chunk
    chunk_sizes = [c["num_sentences"] for c in chunks]

    singleton_pct = (
        sum(s == 1 for s in chunk_sizes) / n_chunks
        if n_chunks > 0 else np.nan
    )

    avg_sent_per_chunk = np.mean(chunk_sizes)

    # Adjacent-sentence cosine similarity statistics
    if n_sentences >= 2:
        sentences = parent_sentences[parent_id]

        vecs = np.vstack([
            sentence_embeddings[
                sentence_id_to_idx[s["sentence_id"]]
            ]
            for s in sentences
        ])

        adjacent_sims = np.sum(vecs[:-1] * vecs[1:], axis=1)

        adjacent_sim_std = float(np.std(adjacent_sims))
        adjacent_sim_mean = float(np.mean(adjacent_sims))
    else:
        adjacent_sim_std = 0.0
        adjacent_sim_mean = np.nan

    return {
        "n_sentences": n_sentences,
        "n_chunks": n_chunks,
        "avg_sentences_per_chunk": avg_sent_per_chunk,
        "singleton_pct": singleton_pct,
        "was_split": int(n_chunks > 1),
        "adjacent_sim_std": adjacent_sim_std,
        "adjacent_sim_mean": adjacent_sim_mean,
    }


# Compute features for all 8,973 passages
structural = {
    parent_id: structural_features(parent_id)
    for parent_id in parent_seg
}

print("\nStructural records:", len(structural))

# Sanity checks
assert len(structural) == 8973
assert set(structural) == set(parent_seg)
assert set(structural) == set(parent_sentences)

print("\nExample:")
example_parent = "hindi_1102432_p00"
print(example_parent)
print(structural[example_parent])

print("\n✓ All 8,973 passages aligned")
print("✓ Structural features computed from frozen artifacts")
print("✓ No segmentation or embeddings recomputed")

Segmentation parents: 8973
Sentence-corpus parents: 8973

Structural records: 8973

Example:
hindi_1102432_p00
{'n_sentences': 4, 'n_chunks': 2, 'avg_sentences_per_chunk': np.float64(2.0), 'singleton_pct': 0.0, 'was_split': 1, 'adjacent_sim_std': 0.091749407351017, 'adjacent_sim_mean': 0.47458615899086}

✓ All 8,973 passages aligned
✓ Structural features computed from frozen artifacts
✓ No segmentation or embeddings recomputed


In [ ]:
# Cell 5 — Attach passage structure to per-query ΔnDCG

def attach_structure(delta_df, split_name):
    rows = []

    # Frozen benchmark lookup
    benchmark_map = {
        (q["language"], q["query_id"]): q
        for q in benchmark
        if q["split"] == split_name
    }

    for _, row in delta_df.iterrows():

        qkey = (row["language"], row["query_id"])
        q = benchmark_map[qkey]

        # A query can have multiple candidate passages.
        # Use the structural features of the relevant passage(s).
        relevant_indices = [
            i for i, label in enumerate(q["is_selected"])
            if label == 1
        ]

        relevant_parents = []

        for idx in relevant_indices:
            key = (
                q["language"],
                q["query_id"],
                idx
            )

            # Exact frozen mapping
            parent_id = passage_key_to_parent[key]
            relevant_parents.append(parent_id)

        # Aggregate across relevant passages for this query.
        features = [
            structural[parent_id]
            for parent_id in relevant_parents
        ]

        row = row.copy()

        row["relevant_passages"] = len(features)

        for feature in [
            "n_sentences",
            "n_chunks",
            "avg_sentences_per_chunk",
            "singleton_pct",
            "was_split",
            "adjacent_sim_std",
            "adjacent_sim_mean",
        ]:
            values = [f[feature] for f in features]
            row[f"rel_{feature}"] = float(np.mean(values))

        rows.append(row)

    return pd.DataFrame(rows)


dev_struct = attach_structure(dev_delta, "dev")
test_struct = attach_structure(test_delta, "test")

print("DEV rows:", len(dev_struct))
print("TEST rows:", len(test_struct))

print("\nDEV example:")
print(
    dev_struct[
        [
            "language",
            "query_id",
            "delta_ndcg",
            "rel_n_sentences",
            "rel_n_chunks",
            "rel_singleton_pct",
            "rel_adjacent_sim_std",
        ]
    ].head().to_string(index=False)
)

print("\n✓ Structural features attached using exact parent mapping")

DEV rows: 225
TEST rows: 234

DEV example:
language  query_id  delta_ndcg  rel_n_sentences  rel_n_chunks  rel_singleton_pct  rel_adjacent_sim_std
 bengali     21860    0.000000              4.0           3.0           0.666667              0.167182
 bengali     53335    0.369070              3.0           1.0           0.000000              0.100553
 bengali     81586    0.000000              4.0           1.0           0.000000              0.042388
 bengali    113649    0.000000              1.0           1.0           1.000000              0.000000
 bengali    116898   -0.043824              2.0           1.0           0.000000              0.000000

✓ Structural features attached using exact parent mapping


In [ ]:
# Cell 6 — Correlation between Max–Min ΔnDCG and passage structure

from scipy.stats import spearmanr

features = [
    "rel_n_sentences",
    "rel_n_chunks",
    "rel_avg_sentences_per_chunk",
    "rel_singleton_pct",
    "rel_was_split",
    "rel_adjacent_sim_std",
    "rel_adjacent_sim_mean",
]

def correlation_table(df):
    results = []

    for feature in features:
        valid = df[[feature, "delta_ndcg"]].dropna()

        rho, p = spearmanr(
            valid[feature],
            valid["delta_ndcg"]
        )

        results.append({
            "feature": feature,
            "spearman_rho": rho,
            "p_value": p,
            "n": len(valid)
        })

    return pd.DataFrame(results).sort_values(
        "spearman_rho",
        key=lambda x: x.abs(),
        ascending=False
    )


dev_corr = correlation_table(dev_struct)
test_corr = correlation_table(test_struct)

print("=" * 75)
print("DEV — Spearman correlation with ΔnDCG (Max–Min − Native)")
print("=" * 75)
print(dev_corr.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

print("\n" + "=" * 75)
print("TEST — Spearman correlation with ΔnDCG (Max–Min − Native)")
print("=" * 75)
print(test_corr.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

print("\n✓ Correlation analysis complete")

DEV — Spearman correlation with ΔnDCG (Max–Min − Native)
                    feature  spearman_rho  p_value   n
rel_avg_sentences_per_chunk       0.05634  0.40031 225
              rel_was_split      -0.05611  0.40223 225
          rel_singleton_pct      -0.05415  0.41887 225
            rel_n_sentences       0.02958  0.65902 225
               rel_n_chunks      -0.01708  0.79885 225
      rel_adjacent_sim_mean      -0.00093  0.99025 174
       rel_adjacent_sim_std       0.00063  0.99254 225

TEST — Spearman correlation with ΔnDCG (Max–Min − Native)
                    feature  spearman_rho  p_value   n
            rel_n_sentences       0.08766  0.18142 234
rel_avg_sentences_per_chunk       0.08261  0.20798 234
       rel_adjacent_sim_std       0.05468  0.40509 234
               rel_n_chunks       0.05430  0.40840 234
          rel_singleton_pct      -0.04888  0.45676 234
      rel_adjacent_sim_mean       0.04718  0.51246 195
              rel_was_split      -0.01261  0.84787 234

✓ C

In [ ]:
# Cell 7 — Structural bins vs Max–Min ΔnDCG

def structural_bins(df):
    out = df.copy()

    # Fixed bins chosen only to describe the observed corpus structure.
    out["length_bin"] = pd.cut(
        out["rel_n_sentences"],
        bins=[0, 1, 2, 3, 5, np.inf],
        labels=["1", "2", "3", "4-5", "6+"],
        include_lowest=True
    )

    out["chunk_bin"] = pd.cut(
        out["rel_n_chunks"],
        bins=[0, 1, 2, 3, np.inf],
        labels=["1", "2", "3", "4+"],
        include_lowest=True
    )

    out["singleton_bin"] = pd.cut(
        out["rel_singleton_pct"],
        bins=[-0.001, 0.0, 0.5, 0.999, 1.001],
        labels=["0%", "0-50%", "50-<100%", "100%"],
        include_lowest=True
    )

    return out


def print_bin_summary(df, column):
    summary = (
        df.groupby(column, observed=True)["delta_ndcg"]
        .agg(
            queries="count",
            mean_delta="mean",
            median_delta="median",
            improved=lambda x: (x > 0).sum(),
            degraded=lambda x: (x < 0).sum(),
            unchanged=lambda x: (x == 0).sum()
        )
        .reset_index()
    )

    summary["improved_pct"] = (
        summary["improved"] / summary["queries"] * 100
    )

    summary["degraded_pct"] = (
        summary["degraded"] / summary["queries"] * 100
    )

    print(summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))


dev_bins = structural_bins(dev_struct)
test_bins = structural_bins(test_struct)

for name, df in [("DEV", dev_bins), ("TEST", test_bins)]:
    print("\n" + "=" * 80)
    print(f"{name} — PASSAGE SENTENCE COUNT")
    print("=" * 80)
    print_bin_summary(df, "length_bin")

    print("\n" + "=" * 80)
    print(f"{name} — MAX–MIN CHUNK COUNT")
    print("=" * 80)
    print_bin_summary(df, "chunk_bin")

    print("\n" + "=" * 80)
    print(f"{name} — SINGLETON CHUNK PROPORTION")
    print("=" * 80)
    print_bin_summary(df, "singleton_bin")

print("\n✓ Structural bin analysis complete")


DEV — PASSAGE SENTENCE COUNT
length_bin  queries  mean_delta  median_delta  improved  degraded  unchanged  improved_pct  degraded_pct
         1       39     -0.0253        0.0000         1         6         32        2.5641       15.3846
         2       35      0.0166        0.0000         6         8         21       17.1429       22.8571
         3       46     -0.0161        0.0000         9        11         26       19.5652       23.9130
       4-5       80     -0.0356        0.0000        14        25         41       17.5000       31.2500
        6+       25      0.0510        0.0000        11         7          7       44.0000       28.0000

DEV — MAX–MIN CHUNK COUNT
chunk_bin  queries  mean_delta  median_delta  improved  degraded  unchanged  improved_pct  degraded_pct
        1       94      0.0018        0.0000        12        17         65       12.7660       18.0851
        2       74     -0.0258        0.0000        11        22         41       14.8649       29.7297
 

In [ ]:
### phase 2

In [ ]:
# ============================================================
# EXPERIMENT 001 — CELL 1
# Reconnect + inspect existing Qwen results
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os

BASE = Path('/content/drive/MyDrive/maxmin_research')

print("BASE:", BASE)
print("Exists:", BASE.exists())

# Existing Qwen result files relevant to the audit
candidates = [
    BASE / "results" / "maxmin_qwen_official_results.json",
    BASE / "results" / "paired_bootstrap_10000_results.json",
    BASE / "results" / "native_baseline_corrected.json",
    BASE / "results" / "fixed2_per_query_results.json",
]

for p in candidates:
    print(f"\n{p.relative_to(BASE)}")
    print("  exists:", p.exists())
    if p.exists():
        print("  size:", round(p.stat().st_size / 1024, 2), "KB")

# Show result directory
results_dir = BASE / "results"

if results_dir.exists():
    print("\n=== RESULTS DIRECTORY ===")
    for p in sorted(results_dir.iterdir()):
        print(p.name)

Mounted at /content/drive
BASE: /content/drive/MyDrive/maxmin_research
Exists: True

results/maxmin_qwen_official_results.json
  exists: True
  size: 419.67 KB

results/paired_bootstrap_10000_results.json
  exists: True
  size: 4.18 KB

results/native_baseline_corrected.json
  exists: True
  size: 0.72 KB

results/fixed2_per_query_results.json
  exists: True
  size: 85.06 KB

=== RESULTS DIRECTORY ===
bge_m3_cluster_bootstrap_queryid_10000.json
bge_m3_paired_bootstrap_10000.json
bge_m3_random_boundary_seed42_results.json
bge_m3_random_vs_maxmin_bootstrap_10000.json
bge_m3_results.json
checkpoint_native_complete.json
fixed2_per_query_results.json
fixed2_results.json
maxmin_qwen_official_results.json
native_baseline_corrected.json
native_baseline_corrected.txt
native_per_query_results.json
paired_bootstrap_10000_results.json


In [ ]:
# ============================================================
# EXPERIMENT 001 — CELL 2
# Inspect Qwen per-query result structure
# ============================================================

path = BASE / "results" / "maxmin_qwen_official_results.json"

with open(path, "r", encoding="utf-8") as f:
    qwen_results = json.load(f)

print("Top-level type:", type(qwen_results).__name__)

if isinstance(qwen_results, dict):
    print("\nTop-level keys:")
    for k, v in qwen_results.items():
        if isinstance(v, (list, dict)):
            print(f"  {k}: {type(v).__name__}, len={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")

print("\n=== RESULT SECTIONS ===")

if isinstance(qwen_results, dict):
    for k, v in qwen_results.items():
        if isinstance(v, dict):
            print(f"\n[{k}]")
            print("Keys:")
            for k2, v2 in v.items():
                if isinstance(v2, (list, dict)):
                    print(f"  {k2}: {type(v2).__name__}, len={len(v2)}")
                else:
                    print(f"  {k2}: {type(v2).__name__} = {v2}")

            # Show first per-query item if present
            for k2, v2 in v.items():
                if isinstance(v2, list) and len(v2) > 0:
                    print(f"\nFirst item from '{k2}':")
                    print(v2[0])

Top-level type: dict

Top-level keys:
  method: str = max_min
  embedding_model: str = Qwen/Qwen3-Embedding-0.6B
  implementation_reference: str = official_author_code
  sigmoid_variant: str = sigmoid(|C| - 1)
  hard_thr: float = 0.6
  init_const: float = 1.5
  c: float = 0.9
  split_protocol: str = frozen_seed_42
  retrieval_scope: str = query_language_only
  retrieval: str = exact_cosine_dot_product
  parent_collapse: str = max_chunk_score
  metrics: dict, len=2
  per_query: dict, len=2

=== RESULT SECTIONS ===

[metrics]
Keys:
  dev: dict, len=5
  test: dict, len=5

[per_query]
Keys:
  dev: list, len=225
  test: list, len=234

First item from 'dev':
{'language': 'hindi', 'query_id': 1090350, 'split': 'dev', 'ndcg10': 1.0, 'mrr10': 1.0, 'recall1': 1.0, 'recall10': 1.0, 'num_relevant_parents': 1, 'top10_parent_ids': ['hindi_1090350_p02', 'hindi_1090350_p03', 'hindi_1090350_p08', 'hindi_1090350_p06', 'hindi_1090350_p05', 'hindi_1090350_p07', 'hindi_1090350_p09', 'hindi_1090350_p01', 'h

In [ ]:
# ============================================================
# EXPERIMENT 001 — CELL 3 (FIXED)
# Verify query_id clustering and split structure
# ============================================================

from collections import Counter

dev_rows = qwen_results["per_query"]["dev"]
test_rows = qwen_results["per_query"]["test"]
all_rows = dev_rows + test_rows

print("DEV rows  :", len(dev_rows))
print("TEST rows :", len(test_rows))
print("TOTAL     :", len(all_rows))

# Overall query_id structure
query_counts = Counter(r["query_id"] for r in all_rows)

print("\n=== OVERALL QUERY-ID STRUCTURE ===")
print("Unique query_ids:", len(query_counts))
print("Cluster sizes:", sorted(set(query_counts.values())))

duplicates = {
    qid: n for qid, n in query_counts.items()
    if n > 1
}

print("Duplicated query_ids:", len(duplicates))

# Per-language structure
lang_counts = Counter(r["language"] for r in all_rows)

print("\n=== LANGUAGE COUNTS ===")
print(dict(sorted(lang_counts.items())))

# Query ID × language uniqueness
pair_counts = Counter(
    (r["query_id"], r["language"])
    for r in all_rows
)

duplicate_pairs = {
    k: n for k, n in pair_counts.items()
    if n > 1
}

print("\n=== QUERY_ID × LANGUAGE ===")
print("Unique (query_id, language) pairs:", len(pair_counts))
print("Duplicated pairs:", len(duplicate_pairs))

if duplicate_pairs:
    print("Examples:", list(duplicate_pairs.items())[:10])

# DEV / TEST split
dev_ids = {r["query_id"] for r in dev_rows}
test_ids = {r["query_id"] for r in test_rows}

print("\n=== DEV / TEST SPLIT ===")
print("Unique DEV query_ids :", len(dev_ids))
print("Unique TEST query_ids:", len(test_ids))
print("DEV ∩ TEST query_ids :", len(dev_ids & test_ids))
print("DEV ∪ TEST query_ids :", len(dev_ids | test_ids))

# Example cluster
print("\n=== EXAMPLE QUERY_ID CLUSTER ===")
example_qid = sorted(query_counts)[0]

for r in all_rows:
    if r["query_id"] == example_qid:
        print(
            r["query_id"],
            r["language"],
            r["split"],
            r["ndcg10"]
        )

DEV rows  : 225
TEST rows : 234
TOTAL     : 459

=== OVERALL QUERY-ID STRUCTURE ===
Unique query_ids: 51
Cluster sizes: [9]
Duplicated query_ids: 51

=== LANGUAGE COUNTS ===
{'bengali': 51, 'gujarati': 51, 'hindi': 51, 'kannada': 51, 'malayalam': 51, 'marathi': 51, 'punjabi': 51, 'tamil': 51, 'telugu': 51}

=== QUERY_ID × LANGUAGE ===
Unique (query_id, language) pairs: 459
Duplicated pairs: 0

=== DEV / TEST SPLIT ===
Unique DEV query_ids : 51
Unique TEST query_ids: 51
DEV ∩ TEST query_ids : 51
DEV ∪ TEST query_ids : 51

=== EXAMPLE QUERY_ID CLUSTER ===
21860 hindi dev 0.5
21860 bengali dev 1.0
21860 tamil dev 0.3562071871080222
21860 telugu dev 0.6309297535714575
21860 gujarati dev 1.0
21860 punjabi dev 0.5
21860 malayalam test 0.38685280723454163
21860 kannada test 0.5
21860 marathi test 0.6309297535714575


In [ ]:
# ============================================================
# EXPERIMENT 001 — CELL 4
# Load Native + Fixed2 per-query results and verify alignment
# ============================================================

# ------------------------------------------------------------
# Load Native
# ------------------------------------------------------------

native_path = BASE / "results" / "native_per_query_results.json"

with open(native_path, "r", encoding="utf-8") as f:
    native_qwen = json.load(f)

print("NATIVE")
print("Type:", type(native_qwen).__name__)

if isinstance(native_qwen, dict):
    print("Keys:", list(native_qwen.keys()))

# ------------------------------------------------------------
# Load Fixed2
# ------------------------------------------------------------

fixed2_path = BASE / "results" / "fixed2_per_query_results.json"

with open(fixed2_path, "r", encoding="utf-8") as f:
    fixed2_qwen = json.load(f)

print("\nFIXED2")
print("Type:", type(fixed2_qwen).__name__)

if isinstance(fixed2_qwen, dict):
    print("Keys:", list(fixed2_qwen.keys()))

# ------------------------------------------------------------
# Inspect likely structures
# ------------------------------------------------------------

print("\n=== NATIVE STRUCTURE ===")

if isinstance(native_qwen, dict):
    for k, v in native_qwen.items():
        if isinstance(v, (list, dict)):
            print(f"  {k}: {type(v).__name__}, len={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")

print("\n=== FIXED2 STRUCTURE ===")

if isinstance(fixed2_qwen, dict):
    for k, v in fixed2_qwen.items():
        if isinstance(v, (list, dict)):
            print(f"  {k}: {type(v).__name__}, len={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")

NATIVE
Type: dict
Keys: ['dev', 'test']

FIXED2
Type: dict
Keys: ['dev', 'test']

=== NATIVE STRUCTURE ===
  dev: list, len=225
  test: list, len=234

=== FIXED2 STRUCTURE ===
  dev: list, len=225
  test: list, len=234


In [ ]:
# ============================================================
# EXPERIMENT 001 — CELL 5
# Load Max-Min + verify alignment across all 3 methods
# ============================================================

# Load Max-Min
maxmin_path = BASE / "results" / "maxmin_qwen_official_results.json"

with open(maxmin_path, "r", encoding="utf-8") as f:
    maxmin_qwen = json.load(f)

print("Loaded Max-Min.")
print("Top-level keys:", list(maxmin_qwen.keys()))

def get_mm_rows(split):
    return maxmin_qwen["per_query"][split]

def make_key_set(rows):
    return {(r["query_id"], r["language"]) for r in rows}

def inspect_split(split):
    mm = get_mm_rows(split)
    native = native_qwen[split]
    fixed2 = fixed2_qwen[split]

    mm_keys = make_key_set(mm)
    native_keys = make_key_set(native)
    fixed2_keys = make_key_set(fixed2)

    print(f"\n{'='*60}")
    print(f"{split.upper()}")
    print(f"{'='*60}")

    print("Rows:")
    print("  Max-Min:", len(mm))
    print("  Native :", len(native))
    print("  Fixed2 :", len(fixed2))

    print("\nUnique (query_id, language) keys:")
    print("  Max-Min:", len(mm_keys))
    print("  Native :", len(native_keys))
    print("  Fixed2 :", len(fixed2_keys))

    print("\nAlignment:")
    print("  MM == Native :", mm_keys == native_keys)
    print("  MM == Fixed2 :", mm_keys == fixed2_keys)
    print("  Native == Fixed2:", native_keys == fixed2_keys)

    print("\nMissing / extra keys:")
    print("  MM - Native :", len(mm_keys - native_keys))
    print("  Native - MM :", len(native_keys - mm_keys))
    print("  MM - Fixed2 :", len(mm_keys - fixed2_keys))
    print("  Fixed2 - MM :", len(fixed2_keys - mm_keys))

    print("\nDuplicate (query_id, language) keys:")
    for name, rows in [
        ("Max-Min", mm),
        ("Native", native),
        ("Fixed2", fixed2),
    ]:
        keys = [(r["query_id"], r["language"]) for r in rows]
        duplicates = len(keys) - len(set(keys))
        print(f"  {name}: {duplicates}")

inspect_split("dev")
inspect_split("test")

Loaded Max-Min.
Top-level keys: ['method', 'embedding_model', 'implementation_reference', 'sigmoid_variant', 'hard_thr', 'init_const', 'c', 'split_protocol', 'retrieval_scope', 'retrieval', 'parent_collapse', 'metrics', 'per_query']

DEV
Rows:
  Max-Min: 225
  Native : 225
  Fixed2 : 225

Unique (query_id, language) keys:
  Max-Min: 225
  Native : 225
  Fixed2 : 225

Alignment:
  MM == Native : True
  MM == Fixed2 : True
  Native == Fixed2: True

Missing / extra keys:
  MM - Native : 0
  Native - MM : 0
  MM - Fixed2 : 0
  Fixed2 - MM : 0

Duplicate (query_id, language) keys:
  Max-Min: 0
  Native: 0
  Fixed2: 0

TEST
Rows:
  Max-Min: 234
  Native : 234
  Fixed2 : 234

Unique (query_id, language) keys:
  Max-Min: 234
  Native : 234
  Fixed2 : 234

Alignment:
  MM == Native : True
  MM == Fixed2 : True
  Native == Fixed2: True

Missing / extra keys:
  MM - Native : 0
  Native - MM : 0
  MM - Fixed2 : 0
  Fixed2 - MM : 0

Duplicate (query_id, language) keys:
  Max-Min: 0
  Native: 0
  Fi

In [ ]:
# ============================================================
# EXPERIMENT 001 — CELL 6
# Query-ID Cluster Bootstrap
# ============================================================

import numpy as np
from collections import defaultdict

N_BOOT = 10_000
SEED = 20261003
ALPHA = 0.05

rng = np.random.default_rng(SEED)

METRICS = ["ndcg10", "mrr10", "recall1", "recall10"]


def cluster_bootstrap_pair(rows_a, rows_b, metric, n_boot=N_BOOT, seed=SEED):
    """
    Cluster bootstrap by query_id.

    Each query_id is one cluster containing all language observations
    belonging to that query within the given split.
    """

    # Index rows by query_id
    a_by_qid = defaultdict(list)
    b_by_qid = defaultdict(list)

    for r in rows_a:
        a_by_qid[r["query_id"]].append(float(r[metric]))

    for r in rows_b:
        b_by_qid[r["query_id"]].append(float(r[metric]))

    # Same cluster set expected
    query_ids = sorted(a_by_qid.keys())

    assert set(query_ids) == set(b_by_qid.keys())

    # Observed difference: mean over all observations
    a_all = np.array(
        [x for qid in query_ids for x in a_by_qid[qid]],
        dtype=float
    )
    b_all = np.array(
        [x for qid in query_ids for x in b_by_qid[qid]],
        dtype=float
    )

    observed = np.mean(a_all - b_all)

    # Independent RNG for reproducibility
    local_rng = np.random.default_rng(seed)

    boot = np.empty(n_boot, dtype=float)

    n_clusters = len(query_ids)

    for i in range(n_boot):
        sampled_indices = local_rng.integers(
            0, n_clusters, size=n_clusters
        )

        sampled_values_a = []
        sampled_values_b = []

        for idx in sampled_indices:
            qid = query_ids[idx]

            sampled_values_a.extend(a_by_qid[qid])
            sampled_values_b.extend(b_by_qid[qid])

        sampled_values_a = np.asarray(sampled_values_a)
        sampled_values_b = np.asarray(sampled_values_b)

        boot[i] = np.mean(sampled_values_a - sampled_values_b)

    lower = np.percentile(boot, 100 * ALPHA / 2)
    upper = np.percentile(boot, 100 * (1 - ALPHA / 2))

    return {
        "observed_diff": float(observed),
        "ci_lower": float(lower),
        "ci_upper": float(upper),
        "significant": bool(lower > 0 or upper < 0),
    }


def run_comparison(split, name_a, rows_a, name_b, rows_b):
    print(f"\n{'='*70}")
    print(f"{split.upper()}: {name_a} - {name_b}")
    print(f"{'='*70}")

    results = {}

    for metric in METRICS:
        result = cluster_bootstrap_pair(
            rows_a,
            rows_b,
            metric,
            n_boot=N_BOOT,
            seed=SEED
        )

        results[metric] = result

        print(
            f"{metric:8s} "
            f"diff={result['observed_diff']:+.6f}  "
            f"95% CI=[{result['ci_lower']:+.6f}, "
            f"{result['ci_upper']:+.6f}]  "
            f"significant={result['significant']}"
        )

    return results


# ------------------------------------------------------------
# Run separately for DEV and TEST
# ------------------------------------------------------------

cluster_bootstrap_001 = {
    "experiment": "001_qwen",
    "bootstrap_unit": "query_id",
    "n_clusters": 51,
    "n_bootstrap": N_BOOT,
    "seed": SEED,
    "confidence_level": 0.95,
    "splits": {}
}

for split in ["dev", "test"]:

    native_rows = native_qwen[split]
    fixed2_rows = fixed2_qwen[split]
    maxmin_rows = maxmin_qwen["per_query"][split]

    print(f"\n\n{'#'*70}")
    print(f"EXPERIMENT 001 — {split.upper()}")
    print(f"{'#'*70}")

    native_maxmin = run_comparison(
        split,
        "Native",
        native_rows,
        "Max-Min",
        maxmin_rows
    )

    fixed2_maxmin = run_comparison(
        split,
        "Fixed2",
        fixed2_rows,
        "Max-Min",
        maxmin_rows
    )

    native_fixed2 = run_comparison(
        split,
        "Native",
        native_rows,
        "Fixed2",
        fixed2_rows
    )

    cluster_bootstrap_001["splits"][split] = {
        "native_minus_maxmin": native_maxmin,
        "fixed2_minus_maxmin": fixed2_maxmin,
        "native_minus_fixed2": native_fixed2,
    }



######################################################################
EXPERIMENT 001 — DEV
######################################################################

DEV: Native - Max-Min
ndcg10   diff=+0.012103  95% CI=[-0.014650, +0.038935]  significant=False
mrr10    diff=+0.006725  95% CI=[-0.026622, +0.040526]  significant=False
recall1  diff=+0.013333  95% CI=[-0.035242, +0.066079]  significant=False
recall10 diff=+0.022222  95% CI=[-0.004274, +0.052301]  significant=False

DEV: Fixed2 - Max-Min
ndcg10   diff=+0.036086  95% CI=[-0.015183, +0.089970]  significant=False
mrr10    diff=+0.003344  95% CI=[-0.022358, +0.027067]  significant=False
recall1  diff=+0.024444  95% CI=[-0.011522, +0.057606]  significant=False
recall10 diff=+0.002222  95% CI=[-0.017699, +0.025317]  significant=False

DEV: Native - Fixed2
ndcg10   diff=-0.023983  95% CI=[-0.077093, +0.027334]  significant=False
mrr10    diff=+0.003381  95% CI=[-0.029170, +0.035020]  significant=False
recall1  diff=-0.011111  95